In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",              # Left to right
    "Gradient_Y",              # Bottom to top
    "Gradient_Diagonal_NE",    # Southwest to Northeast
    "Gradient_Diagonal_NW",    # Southeast to Northwest
    "Gradient_Radial_In",      # Center bright, edges dark
    "Gradient_Radial_Out",     # Edges bright, center dark
    
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",        # Vertical sinusoidal
    "Stripes_Horizontal",      # Horizontal sinusoidal
    "Stripes_Diagonal_45",     # 45-degree stripes
    "Stripes_Diagonal_135",    # 135-degree stripes
    "Waves_Concentric",        # Concentric circles (bullseye)
    "Waves_Spiral",            # Spiral pattern
    "Waves_Interference",      # Two-source interference
    "Waves_Ripple",            # Decaying ripples from center
    
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",             # Gaussian blob center
    "Blob_TopRight",           # Gaussian blob top-right
    "Blob_TopLeft",            # Gaussian blob top-left
    "Blob_BottomRight",        # Gaussian blob bottom-right
    "Blob_BottomLeft",         # Gaussian blob bottom-left
    "Spots_Grid_Dense",        # Dense polka dots
    "Spots_Grid_Sparse",       # Sparse polka dots
    "Spots_Random_Large",      # Large random spots
    "Spots_Triangular",        # Triangular arrangement
    "Spots_Hexagonal",         # Hexagonal arrangement
    
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",              # Inner ring
    "Ring_Outer",              # Outer ring
    "Ring_Double",             # Double concentric rings
    "Ring_Eccentric",          # Off-center ring
    "Ring_Elliptical",         # Elliptical ring
    "Ring_Partial",            # Partial arc/ring
    
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",       # Fine checkerboard
    "Checkerboard_Coarse",     # Coarse checkerboard
    "Quadrant_Alternating",    # Alternating quadrants
    "Sectors_4",               # 4 pie sectors
    "Sectors_8",               # 8 pie sectors
    "Triangle_Pattern",        # Triangular tessellation
    "Diamond_Pattern",         # Diamond/rhombus pattern
    "Honeycomb",               # Honeycomb hexagonal
    
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",         # Layered like cortex
    "Hotspot_Cluster",         # Multiple clustered hotspots
    "Edge_Enhancement",        # Enhanced at tissue edges
    "Core_Shell",              # Core-shell structure
    "Branching",               # Branching/dendritic pattern
    "Laminar_Curved",          # Curved laminar structure
    "Mosaic_Irregular",        # Irregular mosaic
    "Gradient_Sigmoid",        # Sigmoid transition
    "Bimodal_Distribution",    # Two distinct regions
    "Punctate_Dense",          # Dense punctate pattern
    "Periventricular",         # Around a central void
    "Asymmetric_Lobe",         # Asymmetric lobed structure
]

NOISE_FEATURES = 500  # Reduced but more realistic noise features

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    
    Realistic dimensions:
    - Max height: 4.9-5.4 mm (4900-5400 µm)
    - Max width: 4.4-5.0 mm (4400-5000 µm)  
    - Height/Width ratio: 1.0 to 1.2
    - Max width location: between 1/2 and 9/10 of the height from bottom
    - Bottom width: 70% of max width
    - Left side: straight vertical line (midline cut)
    - Bottom: straight horizontal line
    - Right and top: smooth D-curve
    
    Orientation:
    - Medial (midline) on LEFT - straight vertical cut
    - Lateral on RIGHT - curved
    - Dorsal at TOP - curved
    - Ventral at BOTTOM - straight, narrower
    """
    np.random.seed(sample_seed)
    
    # Random variation for realistic dimensions
    # Max height: 4900-5400 µm
    brain_height = np.random.uniform(4900, 5400)
    
    # Height/Width ratio: 1.0-1.2
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    
    # Clamp max width to 4400-5000 µm range
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    
    # Max width location: between 0.5 and 0.9 of height from bottom
    max_width_fraction = np.random.uniform(0.5, 0.9)
    
    # Small random variations
    rotation = np.random.uniform(-0.05, 0.05)  # up to ~3 degrees
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    
    # Position brain - center it in the field with some margin
    # Left edge (medial) positioned to center the brain horizontally
    left_edge = (width - brain_max_width) / 2 + offset_x
    # Bottom edge positioned to center vertically  
    bottom_edge = (height - brain_height) / 2 + offset_y
    
    # Key coordinates
    bottom_width = brain_max_width * 0.70  # Bottom is 70% of max width
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height
    
    def is_in_halfbrain(x, y):
        # Apply rotation around the center of the brain
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        
        # Check basic bounds first
        # Left boundary: straight vertical line
        if x_rot < left_edge:
            return False
        
        # Bottom boundary: straight horizontal line
        if y_rot < bottom_edge:
            return False
        
        # Top boundary
        if y_rot > top_edge:
            return False
        
        # Normalize y position within the brain (0 = bottom, 1 = top)
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        
        # Calculate the right boundary as a function of height
        # - At y_norm=0 (bottom): width = bottom_width (70% of max)
        # - At y_norm=max_width_fraction: width = brain_max_width (100%)
        # - At y_norm=1 (top): curves back to 0
        
        if y_norm <= max_width_fraction:
            # From bottom to max width: smooth curve outward
            t = y_norm / max_width_fraction  # 0 to 1 over this range
            # Smooth interpolation using sine
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            # From max width to top: curve inward (the dome)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)  # 0 to 1
            # Quarter ellipse curve back to 0
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        
        right_edge = left_edge + right_width
        
        if x_rot > right_edge:
            return False
        
        # Handle the dome region (top portion)
        if y_norm > max_width_fraction:
            # x position relative to the brain width at this height
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            
            # The dome follows approximately a quarter ellipse
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            
            # At the medial edge (x_rel near 0), allow more height
            # At lateral edge, height is constrained by the dome curve
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                # Elliptical constraint
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            
            if y_norm > max_y_norm:
                return False
        
        return True
    
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000, add_noise=False, noise_seed=None):
    """
    Generates intensity values (0-1) based on spatial coordinates.
    
    Parameters:
    -----------
    coords : numpy array
        Nx2 array of (x, y) coordinates
    pattern_type : str
        Name of the pattern to generate
    width, height : int
        Field dimensions
    add_noise : bool
        Whether to add small random noise (default False for exact matching)
    noise_seed : int or None
        Seed for noise generation (if add_noise=True)
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    
    # Normalized coordinates (0-1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    
    # Distance from center
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    
    # Angle from center
    angle = np.arctan2(y - cy, x - cx)
    
    # Use a deterministic seed for patterns that need randomness
    # This ensures the same pattern produces identical results regardless of when it's called
    pattern_seed = hash(pattern_type) % 2**31
    
    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm
        
    elif pattern_type == "Gradient_Y":
        val = y_norm
        
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2
        
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2
        
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm
        
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm
        
    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        val = (np.sin(x / 80) + 1) / 2
        
    elif pattern_type == "Stripes_Horizontal":
        val = (np.sin(y / 80) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_45":
        val = (np.sin((x + y) / 100) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_135":
        val = (np.sin((x - y) / 100) + 1) / 2
        
    elif pattern_type == "Waves_Concentric":
        val = (np.sin(dist_center / 60) + 1) / 2
        
    elif pattern_type == "Waves_Spiral":
        spiral = angle + dist_center / 150
        val = (np.sin(spiral * 3) + 1) / 2
        
    elif pattern_type == "Waves_Interference":
        # Two point sources
        d1 = np.sqrt((x - width*0.3)**2 + (y - height*0.5)**2)
        d2 = np.sqrt((x - width*0.7)**2 + (y - height*0.5)**2)
        val = (np.sin(d1/50) + np.sin(d2/50) + 2) / 4
        
    elif pattern_type == "Waves_Ripple":
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center / 40) + 1) / 2
        
    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        val = np.exp(-dist_center**2 / (2 * (width*0.2)**2))
        
    elif pattern_type == "Blob_TopRight":
        bx, by = x.max() * 0.75, y.max() * 0.75
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_TopLeft":
        bx, by = x.min() + (x.max()-x.min()) * 0.25, y.max() * 0.75
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_BottomRight":
        bx, by = x.max() * 0.75, y.min() + (y.max()-y.min()) * 0.25
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_BottomLeft":
        bx, by = x.min() + (x.max()-x.min()) * 0.25, y.min() + (y.max()-y.min()) * 0.25
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Spots_Grid_Dense":
        sx = np.sin(x / 100)
        sy = np.sin(y / 100)
        val = np.clip(sx * sy, 0, 1)
        
    elif pattern_type == "Spots_Grid_Sparse":
        sx = np.sin(x / 200)
        sy = np.sin(y / 200)
        val = np.clip(sx * sy, 0, 1)
        
    elif pattern_type == "Spots_Random_Large":
        # Multiple random Gaussian blobs - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max())
            by = rng.uniform(y.min(), y.max())
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * (width*0.1)**2))
        val = val / val.max()
        
    elif pattern_type == "Spots_Triangular":
        # Triangular grid of spots
        spacing = 400
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2)
                by = j * spacing * 0.866
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * 80**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2)
                by = j * spacing * 0.866
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * 60**2))
        val = np.clip(val, 0, 1)
        
    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * 0.2
        val = np.exp(-((dist_center - target_r)**2) / (2 * (width*0.04)**2))
        
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * 0.7
        val = np.exp(-((dist_center - target_r)**2) / (2 * (width*0.05)**2))
        
    elif pattern_type == "Ring_Double":
        r1 = max_dist * 0.25
        r2 = max_dist * 0.55
        val = np.exp(-((dist_center - r1)**2) / (2 * (width*0.03)**2)) + \
              np.exp(-((dist_center - r2)**2) / (2 * (width*0.03)**2))
        val = val / val.max()
        
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15
        off_cy = cy + height * 0.1
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * 0.35
        val = np.exp(-((dist_off - target_r)**2) / (2 * (width*0.05)**2))
        
    elif pattern_type == "Ring_Elliptical":
        a, b = max_dist * 0.5, max_dist * 0.3
        ellipse_dist = np.sqrt(((x - cx)/a)**2 + ((y - cy)/b)**2)
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * 0.1**2))
        
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * 0.4
        ring_val = np.exp(-((dist_center - target_r)**2) / (2 * (width*0.05)**2))
        # Only show part of the ring (angular mask)
        angle_mask = (angle > -np.pi/2) & (angle < np.pi/2)
        val = ring_val * angle_mask
        
    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200
        x_idx = (x // square_size).astype(int)
        y_idx = (y // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500
        x_idx = (x // square_size).astype(int)
        y_idx = (y // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx) & (y > cy)
        q3 = (x < cx) & (y < cy)
        val = (q1 | q3).astype(float)
        
    elif pattern_type == "Sectors_4":
        sector = ((angle + np.pi) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Sectors_8":
        sector = ((angle + np.pi) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Triangle_Pattern":
        # Triangular wave pattern
        period = 300
        tri_x = np.abs((x % period) - period/2) / (period/2)
        tri_y = np.abs((y % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
        
    elif pattern_type == "Diamond_Pattern":
        period = 400
        val = (np.abs((x % period) - period/2) + np.abs((y % period) - period/2)) / period
        
    elif pattern_type == "Honeycomb":
        # Hexagonal pattern using distance to nearest hex center
        spacing = 200
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2)
                hy = j * spacing * 0.866
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                # Create honeycomb walls (inverse of spots)
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4), 0, 1))
        val = 1 - val  # Invert to get honeycomb walls
        
    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        # Multiple concentric bands with varying intensities
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        layer_idx = np.clip((dist_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
        
    elif pattern_type == "Hotspot_Cluster":
        # Cluster of multiple hotspots - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1)
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1)
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * (width*0.05)**2))
        val = val / val.max()
        
    elif pattern_type == "Edge_Enhancement":
        # Higher at tissue edges (using distance from center as proxy)
        val = 1 - np.exp(-((dist_norm - 0.8)**2) / (2 * 0.15**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Core_Shell":
        # Core region + shell region
        core = np.exp(-dist_center**2 / (2 * (width*0.1)**2))
        shell = np.exp(-((dist_center - max_dist*0.5)**2) / (2 * (width*0.08)**2))
        val = 0.7 * core + 0.5 * shell
        val = val / val.max()
        
    elif pattern_type == "Branching":
        # Branching/dendritic pattern using polar coordinates
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        branch_width = 0.3
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
        
    elif pattern_type == "Laminar_Curved":
        # Curved layers following a sinusoidal boundary
        curved_y = y - 200 * np.sin(x / 400)
        layer_idx = (curved_y // 300).astype(int) % 2
        val = layer_idx.astype(float)
        
    elif pattern_type == "Mosaic_Irregular":
        # Irregular mosaic using Voronoi-like pattern - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        n_seeds = 20
        seed_x = rng.uniform(x.min(), x.max(), n_seeds)
        seed_y = rng.uniform(y.min(), y.max(), n_seeds)
        seed_val = rng.rand(n_seeds)
        # Find nearest seed for each point
        nearest_idx = np.zeros(n, dtype=int)
        min_dist = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist
            nearest_idx[closer] = i
            min_dist[closer] = dist[closer]
        val = seed_val[nearest_idx]
        
    elif pattern_type == "Gradient_Sigmoid":
        # Sharp sigmoid transition
        val = 1 / (1 + np.exp(-10 * (x_norm - 0.5)))
        
    elif pattern_type == "Bimodal_Distribution":
        # Two distinct regions with different base intensities
        region1 = (x < cx).astype(float) * 0.8
        region2 = (x >= cx).astype(float) * 0.3
        val = region1 + region2
        
    elif pattern_type == "Punctate_Dense":
        # Very dense small puncta - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())
            py = rng.uniform(y.min(), y.max())
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * 40**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Periventricular":
        # Pattern around a central void (like around ventricles)
        void_mask = dist_norm < 0.15
        ring_val = np.exp(-((dist_norm - 0.25)**2) / (2 * 0.08**2))
        val = ring_val * ~void_mask
        
    elif pattern_type == "Asymmetric_Lobe":
        # Asymmetric lobular structure
        lobe1_x, lobe1_y = cx - width*0.2, cy
        lobe2_x, lobe2_y = cx + width*0.15, cy + height*0.1
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * (width*0.15)**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * (width*0.1)**2))
        val = val / val.max()
        
    else:
        # Fallback: deterministic noise based on coordinates
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)
    
    # Optionally add small noise for realism (disabled by default for exact matching)
    if add_noise:
        if noise_seed is not None:
            rng = np.random.RandomState(noise_seed)
        else:
            rng = np.random.RandomState(pattern_seed + 999)
        val = val + rng.normal(0, 0.03, size=n)
    
    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000):
    """
    Generate noise that is more similar to real patterns - 
    combinations, distortions, and partial matches of GT patterns.
    
    Uses deterministic seeding for reproducibility.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    
    # Use a deterministic seed based on noise_idx
    rng = np.random.RandomState(noise_idx * 7 + 42)
    
    noise_type = noise_idx % 10
    
    if noise_type == 0:
        # Mixed gradient with random direction
        theta = rng.uniform(0, 2*np.pi)
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)
        
    elif noise_type == 1:
        # Partial blob (cropped/shifted version of GT blob)
        bx = rng.uniform(x.min() - 500, x.max() + 500)
        by = rng.uniform(y.min() - 500, y.max() + 500)
        sigma = rng.uniform(200, 600)
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif noise_type == 2:
        # Distorted stripes (different frequency than GT)
        freq = rng.uniform(50, 200)
        angle = rng.uniform(0, np.pi)
        proj = x * np.cos(angle) + y * np.sin(angle)
        val = (np.sin(proj / freq) + 1) / 2
        
    elif noise_type == 3:
        # Ring at wrong radius
        cx, cy = x.mean(), y.mean()
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        target_r = rng.uniform(100, 800)
        sigma = rng.uniform(50, 150)
        val = np.exp(-((dist - target_r)**2) / (2 * sigma**2))
        
    elif noise_type == 4:
        # Combination of two random GT-like patterns
        p1_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p2_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p1 = generate_pattern_values(coords, gt_patterns_list[p1_idx], width, height)
        p2 = generate_pattern_values(coords, gt_patterns_list[p2_idx], width, height)
        w = rng.uniform(0.3, 0.7)
        val = w * p1 + (1 - w) * p2
        
    elif noise_type == 5:
        # Inverted version of a GT-like pattern
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height)
        val = 1 - base_pattern
        
    elif noise_type == 6:
        # Thresholded/binarized pattern
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height)
        threshold = rng.uniform(0.3, 0.7)
        val = (base_pattern > threshold).astype(float)
        
    elif noise_type == 7:
        # Spatially varying noise (smooth random field)
        # Create low-frequency spatial noise
        freq_x = rng.uniform(200, 600)
        freq_y = rng.uniform(200, 600)
        phase_x = rng.uniform(0, 2*np.pi)
        phase_y = rng.uniform(0, 2*np.pi)
        val = (np.sin(x / freq_x + phase_x) * np.sin(y / freq_y + phase_y) + 1) / 2
        
    elif noise_type == 8:
        # Scattered spots with different spacing
        val = np.zeros(n)
        n_spots = rng.randint(10, 30)
        for _ in range(n_spots):
            sx = rng.uniform(x.min(), x.max())
            sy = rng.uniform(y.min(), y.max())
            sigma = rng.uniform(30, 80)
            dist = np.sqrt((x - sx)**2 + (y - sy)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)
        
    else:
        # Pure random but smoothed (correlated noise)
        val = rng.rand(n)
        # Add some spatial structure by mixing with coordinates
        val = 0.5 * val + 0.25 * (x - x.min()) / (x.max() - x.min() + 1e-8) + \
              0.25 * rng.rand(n)
    
    # Add extra random noise using the same rng
    val = val + rng.normal(0, 0.08, size=n)
    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    """
    Generates coordinates for a Visium-like hexagonal grid.
    
    Visium specifications:
    - Spot diameter: 55 µm
    - Center-to-center spacing: 100 µm
    - Hexagonal arrangement
    
    Parameters:
    -----------
    width, height : float
        Dimensions of the area in micrometers
    spacing : float
        Center-to-center distance between spots (default 100 µm for Visium)
    
    Returns:
    --------
    numpy array of (x, y) coordinates
    """
    # For hexagonal grid:
    # - Horizontal spacing between spots in the same row = spacing
    # - Vertical spacing between rows = spacing * sqrt(3)/2
    # - Alternate rows are offset by spacing/2
    
    dy = spacing * np.sqrt(3) / 2  # Vertical distance between rows
    
    coords = []
    row = 0
    y = 0
    
    while y < height:
        # Offset for alternating rows
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        
        while x < width:
            coords.append([x, y])
            x += spacing
        
        y += dy
        row += 1
    
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

# Field size - needs to accommodate brain dimensions (up to 5.4mm height, 5mm width)
# Using 6000x6000 µm field to give some margin
FIELD_WIDTH = 6000
FIELD_HEIGHT = 6000

print("=" * 60)
print("Generating Ground Truth Dummy Data with 50 PATTERNS")
print("Half-brain shapes, Visium-like RNA grid, no rotation")
print("=" * 60)
print(f"Groups: {GROUPS}")
print(f"GT Patterns: {len(GT_PATTERNS)}")
print(f"Noise Features: {NOISE_FEATURES}")
print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} µm")
print(f"Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2")
print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}µm spacing")
print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}µm spacing")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

ground_truth_pairs = []

for group_idx, group in enumerate(GROUPS):
    for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
        sample_id = f"{group}_{sample_idx}"
        seed = hash(sample_id) % 2**32
        print(f"  Processing {sample_id}...")
        
        # 1. Define Half-Brain Tissue Shape
        tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)
        
        # 2. Generate MSI Data (Cartesian grid)
        raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
        mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
        msi_coords = raw_msi_coords[mask_msi]
        
        n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
        msi_data = np.zeros((len(msi_coords), n_total_features))
        var_names_msi = []
        
        # Generate GT pattern features
        for i, pat in enumerate(GT_PATTERNS):
            msi_data[:, i] = generate_pattern_values(msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT)
            var_names_msi.append(f"MZ_{pat}")
        
        # Generate realistic noise features
        for i in range(NOISE_FEATURES):
            msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                msi_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT
            )
            var_names_msi.append(f"MZ_Noise_{i}")
            
        obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
        obs_msi['x_um'] = msi_coords[:, 0]
        obs_msi['y_um'] = msi_coords[:, 1]
        
        adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
        adata_msi.var_names = var_names_msi
        adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))
        
        
        # 3. Generate RNA Data (Visium-like hexagonal grid)
        raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
        mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
        rna_coords = raw_rna_coords[mask_rna]
        
        rna_data = np.zeros((len(rna_coords), n_total_features))
        var_names_rna = []
        
        # Generate GT pattern features (NO ROTATION - same coordinates as mask)
        for i, pat in enumerate(GT_PATTERNS):
            rna_data[:, i] = generate_pattern_values(rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT)
            var_names_rna.append(f"Gene_{pat}")
            
            if group_idx == 0 and sample_idx == 1:
                ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
        
        # Generate realistic noise features
        for i in range(NOISE_FEATURES):
            rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                rna_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT
            )
            var_names_rna.append(f"Gene_Noise_{i}")
        
        # Store coordinates directly (NO rotation/transformation)
        obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
        obs_rna['x_um'] = rna_coords[:, 0]
        obs_rna['y_um'] = rna_coords[:, 1]
        
        adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
        adata_rna.var_names = var_names_rna
        adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

print("\n" + "=" * 60)
print("DONE! Data generated.")
print("=" * 60)
print("\n50 GROUND TRUTH MATCHES:")
print("-" * 60)
for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
    print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
print("-" * 60)
print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, no rotation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Field size: 6000x6000 µm
Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2
RNA grid: Visium-like hexagonal, 100µm spacing
MSI grid: Cartesian, 60µm spacing
Output: dummy_data_50
  Processing YC_1...
  Processing YC_2...
  Processing YC_3...
  Processing YC_4...
  Processing YAD_1...
  Processing YAD_2...
  Processing YAD_3...
  Processing YAD_4...
  Processing AC_1...
  Processing AC_2...
  Processing AC_3...
  Processing AC_4...
  Processing AAD_1...
  Processing AAD_2...
  Processing AAD_3...
  Processing AAD_4...

DONE! Data generated.

50 GROUND TRUTH MATCHES:
------------------------------------------------------------
   1. Gene_Gradient_X                 <==>  MZ_Gradient_X
   2. Gene_Gradient_Y                 <==>  MZ_Gradient_Y
   3. Gene_Gradient_Diagonal_NE       <==>  MZ_Gradient_Diagonal_NE
  

In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",              # Left to right
    "Gradient_Y",              # Bottom to top
    "Gradient_Diagonal_NE",    # Southwest to Northeast
    "Gradient_Diagonal_NW",    # Southeast to Northwest
    "Gradient_Radial_In",      # Center bright, edges dark
    "Gradient_Radial_Out",     # Edges bright, center dark
    
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",        # Vertical sinusoidal
    "Stripes_Horizontal",      # Horizontal sinusoidal
    "Stripes_Diagonal_45",     # 45-degree stripes
    "Stripes_Diagonal_135",    # 135-degree stripes
    "Waves_Concentric",        # Concentric circles (bullseye)
    "Waves_Spiral",            # Spiral pattern
    "Waves_Interference",      # Two-source interference
    "Waves_Ripple",            # Decaying ripples from center
    
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",             # Gaussian blob center
    "Blob_TopRight",           # Gaussian blob top-right
    "Blob_TopLeft",            # Gaussian blob top-left
    "Blob_BottomRight",        # Gaussian blob bottom-right
    "Blob_BottomLeft",         # Gaussian blob bottom-left
    "Spots_Grid_Dense",        # Dense polka dots
    "Spots_Grid_Sparse",       # Sparse polka dots
    "Spots_Random_Large",      # Large random spots
    "Spots_Triangular",        # Triangular arrangement
    "Spots_Hexagonal",         # Hexagonal arrangement
    
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",              # Inner ring
    "Ring_Outer",              # Outer ring
    "Ring_Double",             # Double concentric rings
    "Ring_Eccentric",          # Off-center ring
    "Ring_Elliptical",         # Elliptical ring
    "Ring_Partial",            # Partial arc/ring
    
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",       # Fine checkerboard
    "Checkerboard_Coarse",     # Coarse checkerboard
    "Quadrant_Alternating",    # Alternating quadrants
    "Sectors_4",               # 4 pie sectors
    "Sectors_8",               # 8 pie sectors
    "Triangle_Pattern",        # Triangular tessellation
    "Diamond_Pattern",         # Diamond/rhombus pattern
    "Honeycomb",               # Honeycomb hexagonal
    
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",         # Layered like cortex
    "Hotspot_Cluster",         # Multiple clustered hotspots
    "Edge_Enhancement",        # Enhanced at tissue edges
    "Core_Shell",              # Core-shell structure
    "Branching",               # Branching/dendritic pattern
    "Laminar_Curved",          # Curved laminar structure
    "Mosaic_Irregular",        # Irregular mosaic
    "Gradient_Sigmoid",        # Sigmoid transition
    "Bimodal_Distribution",    # Two distinct regions
    "Punctate_Dense",          # Dense punctate pattern
    "Periventricular",         # Around a central void
    "Asymmetric_Lobe",         # Asymmetric lobed structure
]

NOISE_FEATURES = 500  # Reduced but more realistic noise features

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    """
    np.random.seed(sample_seed)
    
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height
    
    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        
        if x_rot < left_edge:
            return False
        if y_rot < bottom_edge:
            return False
        if y_rot > top_edge:
            return False
        
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        
        right_edge = left_edge + right_width
        
        if x_rot > right_edge:
            return False
        
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            
            if y_norm > max_y_norm:
                return False
        
        return True
    
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True):
    """
    Generates intensity values (0-1) based on spatial coordinates.

    Per-sample variation is introduced via `sample_seed`, which perturbs:
      - Spatial offsets (blob/ring centers shifted slightly)
      - Phase offsets (stripes/waves shifted)
      - Frequency jitter (stripe period varies slightly)
      - Amplitude scaling (modest intensity rescaling)
      - Additive biological noise scaled by sample_seed

    The perturbations are intentionally small so that matched patterns across
    samples remain clearly correlated, while each animal looks distinct.

    Parameters
    ----------
    coords : np.ndarray  (N, 2)
    pattern_type : str
    width, height : int
        Field dimensions.
    sample_seed : int or None
        Per-sample integer seed.  When None the function behaves exactly as
        the original (all samples identical).
    add_noise : bool
        Whether to add small biological noise (default True).
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    
    # Normalized coordinates (0-1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    
    # Distance from center
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    
    # Angle from center
    angle = np.arctan2(y - cy, x - cx)
    
    # -------------------------------------------------------------------------
    # Per-pattern deterministic seed (unchanged from original)
    # -------------------------------------------------------------------------
    pattern_seed = hash(pattern_type) % 2**31

    # -------------------------------------------------------------------------
    # Per-sample perturbation parameters
    # Each parameter is a small, reproducible deviation driven by sample_seed.
    # -------------------------------------------------------------------------
    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)

        # Spatial offset: shift blob/ring centres by up to ±5% of field size
        dx_offset = rng_s.uniform(-0.05, 0.05) * width
        dy_offset = rng_s.uniform(-0.05, 0.05) * height

        # Phase offset for wave patterns (±0.5 radians / ±half period)
        phase_offset = rng_s.uniform(-0.5, 0.5)

        # Frequency jitter for stripe patterns (±15%)
        freq_jitter = rng_s.uniform(0.85, 1.15)

        # Amplitude scale: modest variation in overall intensity (0.85–1.0 kept
        # to [0,1] after clipping)
        amp_scale = rng_s.uniform(0.80, 1.05)

        # Biological noise level (extra additive noise, 0–8%)
        bio_noise_std = rng_s.uniform(0.0, 0.08)

        # Shifted centre coordinates used by blob/ring patterns
        cx_s = cx + dx_offset
        cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0
        freq_jitter = 1.0
        amp_scale = 1.0
        bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center
        angle_s = angle

    # =========================================================================
    # PATTERN DEFINITIONS
    # =========================================================================

    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Y":
        val = y_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm + phase_offset * 0.1
        
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm + phase_offset * 0.1
        
    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
        
    elif pattern_type == "Waves_Interference":
        # Two point sources, centres shifted per sample
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
        
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        val = np.exp(-dist_center_s**2 / (2 * (width*0.2)**2))
        
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.max() * 0.75 + dy_offset
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.max() * 0.75 + dy_offset
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        val = np.clip(sx * sy, 0, 1)
        
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        val = np.clip(sx * sy, 0, 1)
        
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset
            by = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * (width*0.1)**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * 80**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * 60**2))
        val = np.clip(val, 0, 1)
        
    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2 + phase_offset * 0.05)
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * (width*0.04)**2))
        
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7 + phase_offset * 0.05)
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * (width*0.05)**2))
        
    elif pattern_type == "Ring_Double":
        r1 = max_dist * (0.25 + phase_offset * 0.03)
        r2 = max_dist * (0.55 + phase_offset * 0.03)
        val = np.exp(-((dist_center_s - r1)**2) / (2 * (width*0.03)**2)) + \
              np.exp(-((dist_center_s - r2)**2) / (2 * (width*0.03)**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15 + dx_offset
        off_cy = cy + height * 0.1 + dy_offset
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * (0.35 + phase_offset * 0.04)
        val = np.exp(-((dist_off - target_r)**2) / (2 * (width*0.05)**2))
        
    elif pattern_type == "Ring_Elliptical":
        a = max_dist * (0.5 + phase_offset * 0.05)
        b = max_dist * (0.3 + phase_offset * 0.03)
        ellipse_dist = np.sqrt(((x - cx_s)/a)**2 + ((y - cy_s)/b)**2)
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * 0.1**2))
        
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * (0.4 + phase_offset * 0.04)
        ring_val = np.exp(-((dist_center_s - target_r)**2) / (2 * (width*0.05)**2))
        # Angular mask shifts slightly per sample
        a_start = -np.pi/2 + phase_offset * 0.3
        a_end   =  np.pi/2 + phase_offset * 0.3
        angle_mask = (angle_s > a_start) & (angle_s < a_end)
        val = ring_val * angle_mask
        
    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx_s) & (y > cy_s)
        q3 = (x < cx_s) & (y < cy_s)
        val = (q1 | q3).astype(float)
        
    elif pattern_type == "Sectors_4":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Sectors_8":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Triangle_Pattern":
        period = 300 * freq_jitter
        tri_x = np.abs(((x + dx_offset) % period) - period/2) / (period/2)
        tri_y = np.abs(((y + dy_offset) % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
        
    elif pattern_type == "Diamond_Pattern":
        period = 400 * freq_jitter
        val = (np.abs(((x + dx_offset) % period) - period/2) +
               np.abs(((y + dy_offset) % period) - period/2)) / period
        
    elif pattern_type == "Honeycomb":
        spacing = 200 * freq_jitter
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                hy = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4), 0, 1))
        val = 1 - val
        
    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        # Radius boundaries shift slightly per sample via phase_offset
        shifted_dist = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        shifted_norm = shifted_dist / (shifted_dist.max() + 1e-8)
        layer_idx = np.clip((shifted_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
        
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1) + dx_offset
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1) + dy_offset
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * (width*0.05)**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Edge_Enhancement":
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        target = 0.8 + phase_offset * 0.05
        val = 1 - np.exp(-((shifted_norm - target)**2) / (2 * 0.15**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Core_Shell":
        core = np.exp(-dist_center_s**2 / (2 * (width*0.1)**2))
        shell = np.exp(-((dist_center_s - max_dist*(0.5 + phase_offset*0.05))**2) /
                       (2 * (width*0.08)**2))
        val = 0.7 * core + 0.5 * shell
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Branching":
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        branch_width = 0.3
        # Rotate the whole pattern per sample
        rotated_angle = angle_s + phase_offset * 0.5
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(rotated_angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
        
    elif pattern_type == "Laminar_Curved":
        # Amplitude and phase of sinusoidal curvature vary per sample
        curve_amp  = 200 * (1 + phase_offset * 0.3)
        curve_freq = 400 * freq_jitter
        curved_y = (y + dy_offset) - curve_amp * np.sin((x + dx_offset) / curve_freq)
        layer_idx = (curved_y // 300).astype(int) % 2
        val = layer_idx.astype(float)
        
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed)
        n_seeds = 20
        seed_x = rng.uniform(x.min(), x.max(), n_seeds) + dx_offset
        seed_y = rng.uniform(y.min(), y.max(), n_seeds) + dy_offset
        seed_val = rng.rand(n_seeds)
        nearest_idx = np.zeros(n, dtype=int)
        min_dist_arr = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist_arr
            nearest_idx[closer] = i
            min_dist_arr[closer] = dist[closer]
        val = seed_val[nearest_idx]
        
    elif pattern_type == "Gradient_Sigmoid":
        # Centre of sigmoid shifts per sample
        shift = 0.5 + phase_offset * 0.1
        val = 1 / (1 + np.exp(-10 * (x_norm - shift)))
        
    elif pattern_type == "Bimodal_Distribution":
        region1 = (x < cx_s).astype(float) * 0.8
        region2 = (x >= cx_s).astype(float) * 0.3
        val = region1 + region2
        
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max()) + dx_offset
            py = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * 40**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Periventricular":
        void_r    = 0.15 + phase_offset * 0.02
        ring_r    = 0.25 + phase_offset * 0.02
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        void_mask = shifted_norm < void_r
        ring_val  = np.exp(-((shifted_norm - ring_r)**2) / (2 * 0.08**2))
        val = ring_val * ~void_mask
        
    elif pattern_type == "Asymmetric_Lobe":
        lobe1_x = cx - width*0.2 + dx_offset
        lobe1_y = cy         + dy_offset
        lobe2_x = cx + width*0.15 + dx_offset
        lobe2_y = cy + height*0.1 + dy_offset
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * (width*0.15)**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * (width*0.1)**2))
        val = val / (val.max() + 1e-8)
        
    else:
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)
    
    # -------------------------------------------------------------------------
    # Apply per-sample amplitude scaling
    # -------------------------------------------------------------------------
    val = val * amp_scale

    # -------------------------------------------------------------------------
    # Add biological noise (per-sample, reproducible)
    # -------------------------------------------------------------------------
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000,
                             sample_seed=None):
    """
    Generate noise that is more similar to real patterns - 
    combinations, distortions, and partial matches of GT patterns.
    
    Uses deterministic seeding for reproducibility, with optional per-sample
    variation through sample_seed.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    
    # Combine noise_idx seed with sample_seed so each sample gets different noise
    base_seed = noise_idx * 7 + 42
    if sample_seed is not None:
        combined_seed = (base_seed ^ (sample_seed * 31)) % 2**31
    else:
        combined_seed = base_seed
    rng = np.random.RandomState(combined_seed)
    
    noise_type = noise_idx % 10
    
    if noise_type == 0:
        theta = rng.uniform(0, 2*np.pi)
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)
        
    elif noise_type == 1:
        bx = rng.uniform(x.min() - 500, x.max() + 500)
        by = rng.uniform(y.min() - 500, y.max() + 500)
        sigma = rng.uniform(200, 600)
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif noise_type == 2:
        freq = rng.uniform(50, 200)
        angle_dir = rng.uniform(0, np.pi)
        proj = x * np.cos(angle_dir) + y * np.sin(angle_dir)
        val = (np.sin(proj / freq) + 1) / 2
        
    elif noise_type == 3:
        cx_n, cy_n = x.mean(), y.mean()
        dist = np.sqrt((x - cx_n)**2 + (y - cy_n)**2)
        target_r = rng.uniform(100, 800)
        sigma = rng.uniform(50, 150)
        val = np.exp(-((dist - target_r)**2) / (2 * sigma**2))
        
    elif noise_type == 4:
        p1_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p2_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p1 = generate_pattern_values(coords, gt_patterns_list[p1_idx], width, height,
                                     sample_seed=sample_seed)
        p2 = generate_pattern_values(coords, gt_patterns_list[p2_idx], width, height,
                                     sample_seed=sample_seed)
        w = rng.uniform(0.3, 0.7)
        val = w * p1 + (1 - w) * p2
        
    elif noise_type == 5:
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height,
                                               sample_seed=sample_seed)
        val = 1 - base_pattern
        
    elif noise_type == 6:
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height,
                                               sample_seed=sample_seed)
        threshold = rng.uniform(0.3, 0.7)
        val = (base_pattern > threshold).astype(float)
        
    elif noise_type == 7:
        freq_x = rng.uniform(200, 600)
        freq_y = rng.uniform(200, 600)
        phase_x = rng.uniform(0, 2*np.pi)
        phase_y = rng.uniform(0, 2*np.pi)
        val = (np.sin(x / freq_x + phase_x) * np.sin(y / freq_y + phase_y) + 1) / 2
        
    elif noise_type == 8:
        val = np.zeros(n)
        n_spots = rng.randint(10, 30)
        for _ in range(n_spots):
            sx = rng.uniform(x.min(), x.max())
            sy = rng.uniform(y.min(), y.max())
            sigma = rng.uniform(30, 80)
            dist = np.sqrt((x - sx)**2 + (y - sy)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)
        
    else:
        val = rng.rand(n)
        val = 0.5 * val + 0.25 * (x - x.min()) / (x.max() - x.min() + 1e-8) + \
              0.25 * rng.rand(n)
    
    val = val + rng.normal(0, 0.08, size=n)
    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    """Generates coordinates for a Visium-like hexagonal grid."""
    dy = spacing * np.sqrt(3) / 2
    
    coords = []
    row = 0
    y = 0
    
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        
        while x < width:
            coords.append([x, y])
            x += spacing
        
        y += dy
        row += 1
    
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000

print("=" * 60)
print("Generating Ground Truth Dummy Data with 50 PATTERNS")
print("Half-brain shapes, Visium-like RNA grid, per-sample variation")
print("=" * 60)
print(f"Groups: {GROUPS}")
print(f"GT Patterns: {len(GT_PATTERNS)}")
print(f"Noise Features: {NOISE_FEATURES}")
print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} µm")
print(f"Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2")
print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}µm spacing")
print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}µm spacing")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

ground_truth_pairs = []

for group_idx, group in enumerate(GROUPS):
    for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
        sample_id = f"{group}_{sample_idx}"

        # ------------------------------------------------------------------
        # Derive a unique integer seed per sample.
        # This seed drives both tissue shape AND per-sample pattern variation.
        # ------------------------------------------------------------------
        seed = hash(sample_id) % 2**32

        print(f"  Processing {sample_id} (seed={seed})...")
        
        # 1. Define Half-Brain Tissue Shape
        tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)
        
        # 2. Generate MSI Data (Cartesian grid)
        raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
        mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
        msi_coords = raw_msi_coords[mask_msi]
        
        n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
        msi_data = np.zeros((len(msi_coords), n_total_features))
        var_names_msi = []
        
        # Generate GT pattern features — pass sample_seed for per-animal variation
        for i, pat in enumerate(GT_PATTERNS):
            msi_data[:, i] = generate_pattern_values(
                msi_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_msi.append(f"MZ_{pat}")
        
        # Generate realistic noise features — also per-sample
        for i in range(NOISE_FEATURES):
            msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                msi_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_msi.append(f"MZ_Noise_{i}")
            
        obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
        obs_msi['x_um'] = msi_coords[:, 0]
        obs_msi['y_um'] = msi_coords[:, 1]
        
        adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
        adata_msi.var_names = var_names_msi
        adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))
        
        # 3. Generate RNA Data (Visium-like hexagonal grid)
        raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
        mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
        rna_coords = raw_rna_coords[mask_rna]
        
        rna_data = np.zeros((len(rna_coords), n_total_features))
        var_names_rna = []
        
        # Generate GT pattern features — same sample_seed so RNA and MSI match
        for i, pat in enumerate(GT_PATTERNS):
            rna_data[:, i] = generate_pattern_values(
                rna_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_rna.append(f"Gene_{pat}")
            
            if group_idx == 0 and sample_idx == 1:
                ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
        
        # Generate realistic noise features — per-sample
        for i in range(NOISE_FEATURES):
            rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                rna_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_rna.append(f"Gene_Noise_{i}")
        
        obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
        obs_rna['x_um'] = rna_coords[:, 0]
        obs_rna['y_um'] = rna_coords[:, 1]
        
        adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
        adata_rna.var_names = var_names_rna
        adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

print("\n" + "=" * 60)
print("DONE! Data generated.")
print("=" * 60)
print("\n50 GROUND TRUTH MATCHES:")
print("-" * 60)
for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
    print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
print("-" * 60)
print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, per-sample variation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Field size: 6000x6000 µm
Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2
RNA grid: Visium-like hexagonal, 100µm spacing
MSI grid: Cartesian, 60µm spacing
Output: dummy_data_50
  Processing YC_1 (seed=652569551)...
  Processing YC_2 (seed=3269930845)...
  Processing YC_3 (seed=2585244821)...
  Processing YC_4 (seed=405400305)...
  Processing YAD_1 (seed=3502737025)...
  Processing YAD_2 (seed=9874879)...
  Processing YAD_3 (seed=2646740571)...
  Processing YAD_4 (seed=1699975265)...
  Processing AC_1 (seed=3414647840)...
  Processing AC_2 (seed=3057797209)...
  Processing AC_3 (seed=4109376570)...
  Processing AC_4 (seed=1315553681)...
  Processing AAD_1 (seed=2227101042)...
  Processing AAD_2 (seed=3901879616)...
  Processing AAD_3 (seed=241351907)...
  Processing AAD_4 (seed=3322996982)...

DONE

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",              # Left to right
    "Gradient_Y",              # Bottom to top
    "Gradient_Diagonal_NE",    # Southwest to Northeast
    "Gradient_Diagonal_NW",    # Southeast to Northwest
    "Gradient_Radial_In",      # Center bright, edges dark
    "Gradient_Radial_Out",     # Edges bright, center dark
    
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",        # Vertical sinusoidal
    "Stripes_Horizontal",      # Horizontal sinusoidal
    "Stripes_Diagonal_45",     # 45-degree stripes
    "Stripes_Diagonal_135",    # 135-degree stripes
    "Waves_Concentric",        # Concentric circles (bullseye)
    "Waves_Spiral",            # Spiral pattern
    "Waves_Interference",      # Two-source interference
    "Waves_Ripple",            # Decaying ripples from center
    
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",             # Gaussian blob center
    "Blob_TopRight",           # Gaussian blob top-right
    "Blob_TopLeft",            # Gaussian blob top-left
    "Blob_BottomRight",        # Gaussian blob bottom-right
    "Blob_BottomLeft",         # Gaussian blob bottom-left
    "Spots_Grid_Dense",        # Dense polka dots
    "Spots_Grid_Sparse",       # Sparse polka dots
    "Spots_Random_Large",      # Large random spots
    "Spots_Triangular",        # Triangular arrangement
    "Spots_Hexagonal",         # Hexagonal arrangement
    
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",              # Inner ring
    "Ring_Outer",              # Outer ring
    "Ring_Double",             # Double concentric rings
    "Ring_Eccentric",          # Off-center ring
    "Ring_Elliptical",         # Elliptical ring
    "Ring_Partial",            # Partial arc/ring
    
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",       # Fine checkerboard
    "Checkerboard_Coarse",     # Coarse checkerboard
    "Quadrant_Alternating",    # Alternating quadrants
    "Sectors_4",               # 4 pie sectors
    "Sectors_8",               # 8 pie sectors
    "Triangle_Pattern",        # Triangular tessellation
    "Diamond_Pattern",         # Diamond/rhombus pattern
    "Honeycomb",               # Honeycomb hexagonal
    
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",         # Layered like cortex
    "Hotspot_Cluster",         # Multiple clustered hotspots
    "Edge_Enhancement",        # Enhanced at tissue edges
    "Core_Shell",              # Core-shell structure
    "Branching",               # Branching/dendritic pattern
    "Laminar_Curved",          # Curved laminar structure
    "Mosaic_Irregular",        # Irregular mosaic
    "Gradient_Sigmoid",        # Sigmoid transition
    "Bimodal_Distribution",    # Two distinct regions
    "Punctate_Dense",          # Dense punctate pattern
    "Periventricular",         # Around a central void
    "Asymmetric_Lobe",         # Asymmetric lobed structure
]

NOISE_FEATURES = 500  # Reduced but more realistic noise features

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    """
    np.random.seed(sample_seed)
    
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height
    
    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        
        if x_rot < left_edge:
            return False
        if y_rot < bottom_edge:
            return False
        if y_rot > top_edge:
            return False
        
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        
        right_edge = left_edge + right_width
        
        if x_rot > right_edge:
            return False
        
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            
            if y_norm > max_y_norm:
                return False
        
        return True
    
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True):
    """
    Generates intensity values (0-1) based on spatial coordinates.

    Per-sample variation is introduced via `sample_seed`, which perturbs:
      - Spatial offsets (blob/ring centers shifted slightly)
      - Phase offsets (stripes/waves shifted)
      - Frequency jitter (stripe period varies slightly)
      - Amplitude scaling (modest intensity rescaling)
      - Additive biological noise scaled by sample_seed

    The perturbations are intentionally small so that matched patterns across
    samples remain clearly correlated, while each animal looks distinct.

    Parameters
    ----------
    coords : np.ndarray  (N, 2)
    pattern_type : str
    width, height : int
        Field dimensions.
    sample_seed : int or None
        Per-sample integer seed.  When None the function behaves exactly as
        the original (all samples identical).
    add_noise : bool
        Whether to add small biological noise (default True).
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    
    # Normalized coordinates (0-1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    
    # Distance from center
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    
    # Angle from center
    angle = np.arctan2(y - cy, x - cx)
    
    # -------------------------------------------------------------------------
    # Per-pattern deterministic seed (unchanged from original)
    # -------------------------------------------------------------------------
    pattern_seed = hash(pattern_type) % 2**31

    # -------------------------------------------------------------------------
    # Per-sample perturbation parameters
    # Each parameter is a small, reproducible deviation driven by sample_seed.
    # -------------------------------------------------------------------------
    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)

        # Spatial offset: shift blob/ring/spot centres by up to ±15% of field
        # (noticeably different locations across animals)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height

        # Phase offset for wave patterns (±1 full period shift)
        phase_offset = rng_s.uniform(-1.0, 1.0)

        # Frequency jitter for stripe/spot patterns (±25%)
        freq_jitter = rng_s.uniform(0.75, 1.25)

        # Size scale: blobs can be 70–140% of their nominal sigma/radius.
        # This makes features visibly larger or smaller between animals.
        size_scale = rng_s.uniform(0.70, 1.40)

        # Amplitude scale: modest variation in overall intensity
        amp_scale = rng_s.uniform(0.75, 1.05)

        # Biological noise level (extra additive noise, 0–10%)
        bio_noise_std = rng_s.uniform(0.0, 0.10)

        # Shifted centre coordinates used by blob/ring patterns
        cx_s = cx + dx_offset
        cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0
        freq_jitter = 1.0
        size_scale = 1.0
        amp_scale = 1.0
        bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center
        angle_s = angle

    # =========================================================================
    # PATTERN DEFINITIONS
    # =========================================================================

    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Y":
        val = y_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm + phase_offset * 0.1
        
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm + phase_offset * 0.1
        
    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
        
    elif pattern_type == "Waves_Interference":
        # Two point sources, centres shifted per sample
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
        
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Spots_Grid_Dense":
        # size_scale controls sharpness of each spot (wider = softer cutoff)
        base_freq = 100 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        # Raise to 1/size_scale: <1 broadens spots, >1 sharpens them
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.1 * size_scale
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset
            by = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter
        spot_sigma = 80 * size_scale
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter
        spot_sigma = 60 * size_scale
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        # size_scale shifts the ring radius (larger = ring sits further out)
        target_r = max_dist * (0.2 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.04 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Double":
        r1 = max_dist * (0.25 * size_scale + phase_offset * 0.03)
        r2 = max_dist * (0.55 * size_scale + phase_offset * 0.03)
        ring_width = width * 0.03 * size_scale
        val = np.exp(-((dist_center_s - r1)**2) / (2 * ring_width**2)) + \
              np.exp(-((dist_center_s - r2)**2) / (2 * ring_width**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15 + dx_offset
        off_cy = cy + height * 0.1 + dy_offset
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * (0.35 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_off - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Elliptical":
        # size_scale stretches/shrinks the ellipse axes
        a = max_dist * (0.5 * size_scale + phase_offset * 0.05)
        b = max_dist * (0.3 * size_scale + phase_offset * 0.03)
        ellipse_dist = np.sqrt(((x - cx_s)/a)**2 + ((y - cy_s)/b)**2)
        ring_width = 0.1 * size_scale
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * (0.4 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        ring_val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        # Angular mask shifts slightly per sample; arc width also varies
        arc_half = (np.pi/2) * size_scale
        a_start = -arc_half + phase_offset * 0.3
        a_end   =  arc_half + phase_offset * 0.3
        angle_mask = (angle_s > a_start) & (angle_s < a_end)
        val = ring_val * angle_mask
        
    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx_s) & (y > cy_s)
        q3 = (x < cx_s) & (y < cy_s)
        val = (q1 | q3).astype(float)
        
    elif pattern_type == "Sectors_4":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Sectors_8":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Triangle_Pattern":
        period = 300 * freq_jitter
        tri_x = np.abs(((x + dx_offset) % period) - period/2) / (period/2)
        tri_y = np.abs(((y + dy_offset) % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
        
    elif pattern_type == "Diamond_Pattern":
        period = 400 * freq_jitter
        val = (np.abs(((x + dx_offset) % period) - period/2) +
               np.abs(((y + dy_offset) % period) - period/2)) / period
        
    elif pattern_type == "Honeycomb":
        spacing = 200 * freq_jitter
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                hy = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                # size_scale changes how thick the hex walls appear
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4 * size_scale), 0, 1))
        val = 1 - val
        
    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        # size_scale compresses/expands the layer spacing
        shifted_dist = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        shifted_norm = shifted_dist / (shifted_dist.max() + 1e-8)
        # Divide by size_scale: small = layers are wider (more spread out)
        scaled_norm = np.clip(shifted_norm / size_scale, 0, 1)
        layer_idx = np.clip((scaled_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
        
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.05 * size_scale
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1) + dx_offset
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1) + dy_offset
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Edge_Enhancement":
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        # size_scale moves the enhanced edge inward or outward
        target = 0.8 * size_scale + phase_offset * 0.05
        target = np.clip(target, 0.3, 1.2)
        band_width = 0.15 * size_scale
        val = 1 - np.exp(-((shifted_norm - target)**2) / (2 * band_width**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Core_Shell":
        core_sigma = width * 0.1 * size_scale
        shell_r    = max_dist * (0.5 + phase_offset * 0.05) * size_scale
        shell_sigma = width * 0.08 * size_scale
        core  = np.exp(-dist_center_s**2 / (2 * core_sigma**2))
        shell = np.exp(-((dist_center_s - shell_r)**2) / (2 * shell_sigma**2))
        val = 0.7 * core + 0.5 * shell
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Branching":
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        # size_scale widens/narrows each branch
        branch_width = 0.3 * size_scale
        rotated_angle = angle_s + phase_offset * 0.5
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(rotated_angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
        
    elif pattern_type == "Laminar_Curved":
        # size_scale changes the thickness of each laminar band
        curve_amp  = 200 * (1 + phase_offset * 0.3)
        curve_freq = 400 * freq_jitter
        band_thick = 300 * size_scale
        curved_y = (y + dy_offset) - curve_amp * np.sin((x + dx_offset) / curve_freq)
        layer_idx = (curved_y // band_thick).astype(int) % 2
        val = layer_idx.astype(float)
        
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed)
        # size_scale adjusts the number of Voronoi tiles (fewer = larger patches)
        n_seeds = max(5, int(20 / size_scale))
        seed_x = rng.uniform(x.min(), x.max(), n_seeds) + dx_offset
        seed_y = rng.uniform(y.min(), y.max(), n_seeds) + dy_offset
        seed_val = rng.rand(n_seeds)
        nearest_idx = np.zeros(n, dtype=int)
        min_dist_arr = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist_arr
            nearest_idx[closer] = i
            min_dist_arr[closer] = dist[closer]
        val = seed_val[nearest_idx]
        
    elif pattern_type == "Gradient_Sigmoid":
        # size_scale controls steepness of the transition (sharper vs smoother)
        shift = 0.5 + phase_offset * 0.1
        steepness = 10.0 / size_scale   # smaller size_scale = steeper step
        val = 1 / (1 + np.exp(-steepness * (x_norm - shift)))
        
    elif pattern_type == "Bimodal_Distribution":
        region1 = (x < cx_s).astype(float) * 0.8
        region2 = (x >= cx_s).astype(float) * 0.3
        val = region1 + region2
        
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed)
        punctum_sigma = 40 * size_scale
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max()) + dx_offset
            py = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * punctum_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Periventricular":
        # size_scale enlarges/shrinks both the void and the periventricular ring
        void_r = 0.15 * size_scale + phase_offset * 0.02
        ring_r = 0.25 * size_scale + phase_offset * 0.02
        ring_w = 0.08 * size_scale
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        void_mask = shifted_norm < void_r
        ring_val  = np.exp(-((shifted_norm - ring_r)**2) / (2 * ring_w**2))
        val = ring_val * ~void_mask
        
    elif pattern_type == "Asymmetric_Lobe":
        lobe1_x = cx - width*0.2 + dx_offset
        lobe1_y = cy              + dy_offset
        lobe2_x = cx + width*0.15 + dx_offset
        lobe2_y = cy + height*0.1 + dy_offset
        sigma1 = width  * 0.15 * size_scale
        sigma2 = width  * 0.10 * size_scale
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * sigma1**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * sigma2**2))
        val = val / (val.max() + 1e-8)
        
    else:
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)
    
    # -------------------------------------------------------------------------
    # Apply per-sample amplitude scaling
    # -------------------------------------------------------------------------
    val = val * amp_scale

    # -------------------------------------------------------------------------
    # Add biological noise (per-sample, reproducible)
    # -------------------------------------------------------------------------
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000,
                             sample_seed=None):
    """
    Generate noise that is more similar to real patterns - 
    combinations, distortions, and partial matches of GT patterns.
    
    Uses deterministic seeding for reproducibility, with optional per-sample
    variation through sample_seed.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    
    # Combine noise_idx seed with sample_seed so each sample gets different noise
    base_seed = noise_idx * 7 + 42
    if sample_seed is not None:
        combined_seed = (base_seed ^ (sample_seed * 31)) % 2**31
    else:
        combined_seed = base_seed
    rng = np.random.RandomState(combined_seed)
    
    noise_type = noise_idx % 10
    
    if noise_type == 0:
        theta = rng.uniform(0, 2*np.pi)
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)
        
    elif noise_type == 1:
        bx = rng.uniform(x.min() - 500, x.max() + 500)
        by = rng.uniform(y.min() - 500, y.max() + 500)
        sigma = rng.uniform(200, 600)
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif noise_type == 2:
        freq = rng.uniform(50, 200)
        angle_dir = rng.uniform(0, np.pi)
        proj = x * np.cos(angle_dir) + y * np.sin(angle_dir)
        val = (np.sin(proj / freq) + 1) / 2
        
    elif noise_type == 3:
        cx_n, cy_n = x.mean(), y.mean()
        dist = np.sqrt((x - cx_n)**2 + (y - cy_n)**2)
        target_r = rng.uniform(100, 800)
        sigma = rng.uniform(50, 150)
        val = np.exp(-((dist - target_r)**2) / (2 * sigma**2))
        
    elif noise_type == 4:
        p1_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p2_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p1 = generate_pattern_values(coords, gt_patterns_list[p1_idx], width, height,
                                     sample_seed=sample_seed)
        p2 = generate_pattern_values(coords, gt_patterns_list[p2_idx], width, height,
                                     sample_seed=sample_seed)
        w = rng.uniform(0.3, 0.7)
        val = w * p1 + (1 - w) * p2
        
    elif noise_type == 5:
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height,
                                               sample_seed=sample_seed)
        val = 1 - base_pattern
        
    elif noise_type == 6:
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height,
                                               sample_seed=sample_seed)
        threshold = rng.uniform(0.3, 0.7)
        val = (base_pattern > threshold).astype(float)
        
    elif noise_type == 7:
        freq_x = rng.uniform(200, 600)
        freq_y = rng.uniform(200, 600)
        phase_x = rng.uniform(0, 2*np.pi)
        phase_y = rng.uniform(0, 2*np.pi)
        val = (np.sin(x / freq_x + phase_x) * np.sin(y / freq_y + phase_y) + 1) / 2
        
    elif noise_type == 8:
        val = np.zeros(n)
        n_spots = rng.randint(10, 30)
        for _ in range(n_spots):
            sx = rng.uniform(x.min(), x.max())
            sy = rng.uniform(y.min(), y.max())
            sigma = rng.uniform(30, 80)
            dist = np.sqrt((x - sx)**2 + (y - sy)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)
        
    else:
        val = rng.rand(n)
        val = 0.5 * val + 0.25 * (x - x.min()) / (x.max() - x.min() + 1e-8) + \
              0.25 * rng.rand(n)
    
    val = val + rng.normal(0, 0.08, size=n)
    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    """Generates coordinates for a Visium-like hexagonal grid."""
    dy = spacing * np.sqrt(3) / 2
    
    coords = []
    row = 0
    y = 0
    
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        
        while x < width:
            coords.append([x, y])
            x += spacing
        
        y += dy
        row += 1
    
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000

print("=" * 60)
print("Generating Ground Truth Dummy Data with 50 PATTERNS")
print("Half-brain shapes, Visium-like RNA grid, per-sample variation")
print("=" * 60)
print(f"Groups: {GROUPS}")
print(f"GT Patterns: {len(GT_PATTERNS)}")
print(f"Noise Features: {NOISE_FEATURES}")
print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} µm")
print(f"Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2")
print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}µm spacing")
print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}µm spacing")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

ground_truth_pairs = []

for group_idx, group in enumerate(GROUPS):
    for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
        sample_id = f"{group}_{sample_idx}"

        # ------------------------------------------------------------------
        # Derive a unique integer seed per sample.
        # This seed drives both tissue shape AND per-sample pattern variation.
        # ------------------------------------------------------------------
        seed = hash(sample_id) % 2**32

        print(f"  Processing {sample_id} (seed={seed})...")
        
        # 1. Define Half-Brain Tissue Shape
        tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)
        
        # 2. Generate MSI Data (Cartesian grid)
        raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
        mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
        msi_coords = raw_msi_coords[mask_msi]
        
        n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
        msi_data = np.zeros((len(msi_coords), n_total_features))
        var_names_msi = []
        
        # Generate GT pattern features — pass sample_seed for per-animal variation
        for i, pat in enumerate(GT_PATTERNS):
            msi_data[:, i] = generate_pattern_values(
                msi_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_msi.append(f"MZ_{pat}")
        
        # Generate realistic noise features — also per-sample
        for i in range(NOISE_FEATURES):
            msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                msi_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_msi.append(f"MZ_Noise_{i}")
            
        obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
        obs_msi['x_um'] = msi_coords[:, 0]
        obs_msi['y_um'] = msi_coords[:, 1]
        
        adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
        adata_msi.var_names = var_names_msi
        adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))
        
        # 3. Generate RNA Data (Visium-like hexagonal grid)
        raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
        mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
        rna_coords = raw_rna_coords[mask_rna]
        
        rna_data = np.zeros((len(rna_coords), n_total_features))
        var_names_rna = []
        
        # Generate GT pattern features — same sample_seed so RNA and MSI match
        for i, pat in enumerate(GT_PATTERNS):
            rna_data[:, i] = generate_pattern_values(
                rna_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_rna.append(f"Gene_{pat}")
            
            if group_idx == 0 and sample_idx == 1:
                ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
        
        # Generate realistic noise features — per-sample
        for i in range(NOISE_FEATURES):
            rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                rna_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed
            )
            var_names_rna.append(f"Gene_Noise_{i}")
        
        obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
        obs_rna['x_um'] = rna_coords[:, 0]
        obs_rna['y_um'] = rna_coords[:, 1]
        
        adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
        adata_rna.var_names = var_names_rna
        adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

print("\n" + "=" * 60)
print("DONE! Data generated.")
print("=" * 60)
print("\n50 GROUND TRUTH MATCHES:")
print("-" * 60)
for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
    print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
print("-" * 60)
print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")

In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# Per-sample pattern size variation range.
# Each animal draws a size_scale ~ Uniform(low, high) per pattern, controlling
# blob sigmas, ring radii, spot sizes, band widths, etc.
# (1.0, 1.0) = no size variation; (0.5, 1.5) = aggressive variation.
PATTERN_SIZE_SCALE_RANGE = (0.85, 1.15)

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",              # Left to right
    "Gradient_Y",              # Bottom to top
    "Gradient_Diagonal_NE",    # Southwest to Northeast
    "Gradient_Diagonal_NW",    # Southeast to Northwest
    "Gradient_Radial_In",      # Center bright, edges dark
    "Gradient_Radial_Out",     # Edges bright, center dark
    
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",        # Vertical sinusoidal
    "Stripes_Horizontal",      # Horizontal sinusoidal
    "Stripes_Diagonal_45",     # 45-degree stripes
    "Stripes_Diagonal_135",    # 135-degree stripes
    "Waves_Concentric",        # Concentric circles (bullseye)
    "Waves_Spiral",            # Spiral pattern
    "Waves_Interference",      # Two-source interference
    "Waves_Ripple",            # Decaying ripples from center
    
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",             # Gaussian blob center
    "Blob_TopRight",           # Gaussian blob top-right
    "Blob_TopLeft",            # Gaussian blob top-left
    "Blob_BottomRight",        # Gaussian blob bottom-right
    "Blob_BottomLeft",         # Gaussian blob bottom-left
    "Spots_Grid_Dense",        # Dense polka dots
    "Spots_Grid_Sparse",       # Sparse polka dots
    "Spots_Random_Large",      # Large random spots
    "Spots_Triangular",        # Triangular arrangement
    "Spots_Hexagonal",         # Hexagonal arrangement
    
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",              # Inner ring
    "Ring_Outer",              # Outer ring
    "Ring_Double",             # Double concentric rings
    "Ring_Eccentric",          # Off-center ring
    "Ring_Elliptical",         # Elliptical ring
    "Ring_Partial",            # Partial arc/ring
    
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",       # Fine checkerboard
    "Checkerboard_Coarse",     # Coarse checkerboard
    "Quadrant_Alternating",    # Alternating quadrants
    "Sectors_4",               # 4 pie sectors
    "Sectors_8",               # 8 pie sectors
    "Triangle_Pattern",        # Triangular tessellation
    "Diamond_Pattern",         # Diamond/rhombus pattern
    "Honeycomb",               # Honeycomb hexagonal
    
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",         # Layered like cortex
    "Hotspot_Cluster",         # Multiple clustered hotspots
    "Edge_Enhancement",        # Enhanced at tissue edges
    "Core_Shell",              # Core-shell structure
    "Branching",               # Branching/dendritic pattern
    "Laminar_Curved",          # Curved laminar structure
    "Mosaic_Irregular",        # Irregular mosaic
    "Gradient_Sigmoid",        # Sigmoid transition
    "Bimodal_Distribution",    # Two distinct regions
    "Punctate_Dense",          # Dense punctate pattern
    "Periventricular",         # Around a central void
    "Asymmetric_Lobe",         # Asymmetric lobed structure
]

NOISE_FEATURES = 500  # Reduced but more realistic noise features

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    """
    np.random.seed(sample_seed)
    
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height
    
    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        
        if x_rot < left_edge:
            return False
        if y_rot < bottom_edge:
            return False
        if y_rot > top_edge:
            return False
        
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        
        right_edge = left_edge + right_width
        
        if x_rot > right_edge:
            return False
        
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            
            if y_norm > max_y_norm:
                return False
        
        return True
    
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    """
    Generates intensity values (0-1) based on spatial coordinates.

    Per-sample variation is introduced via `sample_seed`, which perturbs:
      - Spatial offsets (blob/ring centres shifted by up to ±15% of field)
      - Size scaling (blob sigmas, ring radii, band widths — range set by
        size_scale_range, passed from PATTERN_SIZE_SCALE_RANGE in config)
      - Phase offsets (stripes/waves shifted by ±1 period)
      - Frequency jitter (stripe/spot period varies ±25%)
      - Amplitude scaling (modest intensity rescaling)
      - Additive biological noise (0–10% σ)

    The perturbations keep matched patterns correlated across animals while
    making each sample visually distinct.

    Parameters
    ----------
    coords : np.ndarray  (N, 2)
    pattern_type : str
    width, height : int
        Field dimensions.
    sample_seed : int or None
        Per-sample integer seed.  When None the function behaves exactly as
        the original (all samples identical).
    add_noise : bool
        Whether to add small biological noise (default True).
    size_scale_range : tuple (low, high)
        Uniform range from which the per-sample size_scale is drawn.
        Controls how much larger or smaller features are between animals.
        (1.0, 1.0) disables size variation; (0.5, 1.5) is aggressive.
        Passed in from PATTERN_SIZE_SCALE_RANGE in the config.
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    
    # Normalized coordinates (0-1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    
    # Distance from center
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    
    # Angle from center
    angle = np.arctan2(y - cy, x - cx)
    
    # -------------------------------------------------------------------------
    # Per-pattern deterministic seed (unchanged from original)
    # -------------------------------------------------------------------------
    pattern_seed = hash(pattern_type) % 2**31

    # -------------------------------------------------------------------------
    # Per-sample perturbation parameters
    # Each parameter is a small, reproducible deviation driven by sample_seed.
    # -------------------------------------------------------------------------
    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)

        # Spatial offset: shift blob/ring/spot centres by up to ±15% of field
        # (noticeably different locations across animals)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height

        # Phase offset for wave patterns (±1 full period shift)
        phase_offset = rng_s.uniform(-1.0, 1.0)

        # Frequency jitter for stripe/spot patterns (±25%)
        freq_jitter = rng_s.uniform(0.75, 1.25)

        # Size scale: drawn from size_scale_range (set via PATTERN_SIZE_SCALE_RANGE
        # in config). Controls how much features grow/shrink between animals.
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])

        # Amplitude scale: modest variation in overall intensity
        amp_scale = rng_s.uniform(0.75, 1.05)

        # Biological noise level (extra additive noise, 0–10%)
        bio_noise_std = rng_s.uniform(0.0, 0.10)

        # Shifted centre coordinates used by blob/ring patterns
        cx_s = cx + dx_offset
        cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0
        freq_jitter = 1.0
        size_scale = 1.0
        amp_scale = 1.0
        bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center
        angle_s = angle

    # =========================================================================
    # PATTERN DEFINITIONS
    # =========================================================================

    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Y":
        val = y_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm + phase_offset * 0.1
        
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm + phase_offset * 0.1
        
    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
        
    elif pattern_type == "Waves_Interference":
        # Two point sources, centres shifted per sample
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
        
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Spots_Grid_Dense":
        # size_scale controls sharpness of each spot (wider = softer cutoff)
        base_freq = 100 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        # Raise to 1/size_scale: <1 broadens spots, >1 sharpens them
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.1 * size_scale
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset
            by = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter
        spot_sigma = 80 * size_scale
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter
        spot_sigma = 60 * size_scale
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        # size_scale shifts the ring radius (larger = ring sits further out)
        target_r = max_dist * (0.2 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.04 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Double":
        r1 = max_dist * (0.25 * size_scale + phase_offset * 0.03)
        r2 = max_dist * (0.55 * size_scale + phase_offset * 0.03)
        ring_width = width * 0.03 * size_scale
        val = np.exp(-((dist_center_s - r1)**2) / (2 * ring_width**2)) + \
              np.exp(-((dist_center_s - r2)**2) / (2 * ring_width**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15 + dx_offset
        off_cy = cy + height * 0.1 + dy_offset
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * (0.35 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_off - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Elliptical":
        # size_scale stretches/shrinks the ellipse axes
        a = max_dist * (0.5 * size_scale + phase_offset * 0.05)
        b = max_dist * (0.3 * size_scale + phase_offset * 0.03)
        ellipse_dist = np.sqrt(((x - cx_s)/a)**2 + ((y - cy_s)/b)**2)
        ring_width = 0.1 * size_scale
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * (0.4 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        ring_val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        # Angular mask shifts slightly per sample; arc width also varies
        arc_half = (np.pi/2) * size_scale
        a_start = -arc_half + phase_offset * 0.3
        a_end   =  arc_half + phase_offset * 0.3
        angle_mask = (angle_s > a_start) & (angle_s < a_end)
        val = ring_val * angle_mask
        
    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx_s) & (y > cy_s)
        q3 = (x < cx_s) & (y < cy_s)
        val = (q1 | q3).astype(float)
        
    elif pattern_type == "Sectors_4":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Sectors_8":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Triangle_Pattern":
        period = 300 * freq_jitter
        tri_x = np.abs(((x + dx_offset) % period) - period/2) / (period/2)
        tri_y = np.abs(((y + dy_offset) % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
        
    elif pattern_type == "Diamond_Pattern":
        period = 400 * freq_jitter
        val = (np.abs(((x + dx_offset) % period) - period/2) +
               np.abs(((y + dy_offset) % period) - period/2)) / period
        
    elif pattern_type == "Honeycomb":
        spacing = 200 * freq_jitter
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                hy = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                # size_scale changes how thick the hex walls appear
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4 * size_scale), 0, 1))
        val = 1 - val
        
    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        # size_scale compresses/expands the layer spacing
        shifted_dist = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        shifted_norm = shifted_dist / (shifted_dist.max() + 1e-8)
        # Divide by size_scale: small = layers are wider (more spread out)
        scaled_norm = np.clip(shifted_norm / size_scale, 0, 1)
        layer_idx = np.clip((scaled_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
        
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.05 * size_scale
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1) + dx_offset
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1) + dy_offset
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Edge_Enhancement":
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        # size_scale moves the enhanced edge inward or outward
        target = 0.8 * size_scale + phase_offset * 0.05
        target = np.clip(target, 0.3, 1.2)
        band_width = 0.15 * size_scale
        val = 1 - np.exp(-((shifted_norm - target)**2) / (2 * band_width**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Core_Shell":
        core_sigma = width * 0.1 * size_scale
        shell_r    = max_dist * (0.5 + phase_offset * 0.05) * size_scale
        shell_sigma = width * 0.08 * size_scale
        core  = np.exp(-dist_center_s**2 / (2 * core_sigma**2))
        shell = np.exp(-((dist_center_s - shell_r)**2) / (2 * shell_sigma**2))
        val = 0.7 * core + 0.5 * shell
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Branching":
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        # size_scale widens/narrows each branch
        branch_width = 0.3 * size_scale
        rotated_angle = angle_s + phase_offset * 0.5
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(rotated_angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
        
    elif pattern_type == "Laminar_Curved":
        # size_scale changes the thickness of each laminar band
        curve_amp  = 200 * (1 + phase_offset * 0.3)
        curve_freq = 400 * freq_jitter
        band_thick = 300 * size_scale
        curved_y = (y + dy_offset) - curve_amp * np.sin((x + dx_offset) / curve_freq)
        layer_idx = (curved_y // band_thick).astype(int) % 2
        val = layer_idx.astype(float)
        
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed)
        # size_scale adjusts the number of Voronoi tiles (fewer = larger patches)
        n_seeds = max(5, int(20 / size_scale))
        seed_x = rng.uniform(x.min(), x.max(), n_seeds) + dx_offset
        seed_y = rng.uniform(y.min(), y.max(), n_seeds) + dy_offset
        seed_val = rng.rand(n_seeds)
        nearest_idx = np.zeros(n, dtype=int)
        min_dist_arr = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist_arr
            nearest_idx[closer] = i
            min_dist_arr[closer] = dist[closer]
        val = seed_val[nearest_idx]
        
    elif pattern_type == "Gradient_Sigmoid":
        # size_scale controls steepness of the transition (sharper vs smoother)
        shift = 0.5 + phase_offset * 0.1
        steepness = 10.0 / size_scale   # smaller size_scale = steeper step
        val = 1 / (1 + np.exp(-steepness * (x_norm - shift)))
        
    elif pattern_type == "Bimodal_Distribution":
        region1 = (x < cx_s).astype(float) * 0.8
        region2 = (x >= cx_s).astype(float) * 0.3
        val = region1 + region2
        
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed)
        punctum_sigma = 40 * size_scale
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max()) + dx_offset
            py = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * punctum_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Periventricular":
        # size_scale enlarges/shrinks both the void and the periventricular ring
        void_r = 0.15 * size_scale + phase_offset * 0.02
        ring_r = 0.25 * size_scale + phase_offset * 0.02
        ring_w = 0.08 * size_scale
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        void_mask = shifted_norm < void_r
        ring_val  = np.exp(-((shifted_norm - ring_r)**2) / (2 * ring_w**2))
        val = ring_val * ~void_mask
        
    elif pattern_type == "Asymmetric_Lobe":
        lobe1_x = cx - width*0.2 + dx_offset
        lobe1_y = cy              + dy_offset
        lobe2_x = cx + width*0.15 + dx_offset
        lobe2_y = cy + height*0.1 + dy_offset
        sigma1 = width  * 0.15 * size_scale
        sigma2 = width  * 0.10 * size_scale
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * sigma1**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * sigma2**2))
        val = val / (val.max() + 1e-8)
        
    else:
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)
    
    # -------------------------------------------------------------------------
    # Apply per-sample amplitude scaling
    # -------------------------------------------------------------------------
    val = val * amp_scale

    # -------------------------------------------------------------------------
    # Add biological noise (per-sample, reproducible)
    # -------------------------------------------------------------------------
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000,
                             sample_seed=None, size_scale_range=(0.70, 1.40)):
    """
    Generate noise that is more similar to real patterns - 
    combinations, distortions, and partial matches of GT patterns.
    
    Uses deterministic seeding for reproducibility, with optional per-sample
    variation through sample_seed.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    
    # Combine noise_idx seed with sample_seed so each sample gets different noise
    base_seed = noise_idx * 7 + 42
    if sample_seed is not None:
        combined_seed = (base_seed ^ (sample_seed * 31)) % 2**31
    else:
        combined_seed = base_seed
    rng = np.random.RandomState(combined_seed)
    
    noise_type = noise_idx % 10
    
    if noise_type == 0:
        theta = rng.uniform(0, 2*np.pi)
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)
        
    elif noise_type == 1:
        bx = rng.uniform(x.min() - 500, x.max() + 500)
        by = rng.uniform(y.min() - 500, y.max() + 500)
        sigma = rng.uniform(200, 600)
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif noise_type == 2:
        freq = rng.uniform(50, 200)
        angle_dir = rng.uniform(0, np.pi)
        proj = x * np.cos(angle_dir) + y * np.sin(angle_dir)
        val = (np.sin(proj / freq) + 1) / 2
        
    elif noise_type == 3:
        cx_n, cy_n = x.mean(), y.mean()
        dist = np.sqrt((x - cx_n)**2 + (y - cy_n)**2)
        target_r = rng.uniform(100, 800)
        sigma = rng.uniform(50, 150)
        val = np.exp(-((dist - target_r)**2) / (2 * sigma**2))
        
    elif noise_type == 4:
        p1_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p2_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p1 = generate_pattern_values(coords, gt_patterns_list[p1_idx], width, height,
                                     sample_seed=sample_seed, size_scale_range=size_scale_range)
        p2 = generate_pattern_values(coords, gt_patterns_list[p2_idx], width, height,
                                     sample_seed=sample_seed, size_scale_range=size_scale_range)
        w = rng.uniform(0.3, 0.7)
        val = w * p1 + (1 - w) * p2
        
    elif noise_type == 5:
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height,
                                               sample_seed=sample_seed, size_scale_range=size_scale_range)
        val = 1 - base_pattern
        
    elif noise_type == 6:
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height,
                                               sample_seed=sample_seed, size_scale_range=size_scale_range)
        threshold = rng.uniform(0.3, 0.7)
        val = (base_pattern > threshold).astype(float)
        
    elif noise_type == 7:
        freq_x = rng.uniform(200, 600)
        freq_y = rng.uniform(200, 600)
        phase_x = rng.uniform(0, 2*np.pi)
        phase_y = rng.uniform(0, 2*np.pi)
        val = (np.sin(x / freq_x + phase_x) * np.sin(y / freq_y + phase_y) + 1) / 2
        
    elif noise_type == 8:
        val = np.zeros(n)
        n_spots = rng.randint(10, 30)
        for _ in range(n_spots):
            sx = rng.uniform(x.min(), x.max())
            sy = rng.uniform(y.min(), y.max())
            sigma = rng.uniform(30, 80)
            dist = np.sqrt((x - sx)**2 + (y - sy)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)
        
    else:
        val = rng.rand(n)
        val = 0.5 * val + 0.25 * (x - x.min()) / (x.max() - x.min() + 1e-8) + \
              0.25 * rng.rand(n)
    
    val = val + rng.normal(0, 0.08, size=n)
    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    """Generates coordinates for a Visium-like hexagonal grid."""
    dy = spacing * np.sqrt(3) / 2
    
    coords = []
    row = 0
    y = 0
    
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        
        while x < width:
            coords.append([x, y])
            x += spacing
        
        y += dy
        row += 1
    
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000

print("=" * 60)
print("Generating Ground Truth Dummy Data with 50 PATTERNS")
print("Half-brain shapes, Visium-like RNA grid, per-sample variation")
print("=" * 60)
print(f"Groups: {GROUPS}")
print(f"GT Patterns: {len(GT_PATTERNS)}")
print(f"Noise Features: {NOISE_FEATURES}")
print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} µm")
print(f"Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2")
print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}µm spacing")
print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}µm spacing")
print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}–{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

ground_truth_pairs = []

for group_idx, group in enumerate(GROUPS):
    for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
        sample_id = f"{group}_{sample_idx}"

        # ------------------------------------------------------------------
        # Derive a unique integer seed per sample.
        # This seed drives both tissue shape AND per-sample pattern variation.
        # ------------------------------------------------------------------
        seed = hash(sample_id) % 2**32

        print(f"  Processing {sample_id} (seed={seed})...")
        
        # 1. Define Half-Brain Tissue Shape
        tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)
        
        # 2. Generate MSI Data (Cartesian grid)
        raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
        mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
        msi_coords = raw_msi_coords[mask_msi]
        
        n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
        msi_data = np.zeros((len(msi_coords), n_total_features))
        var_names_msi = []
        
        # Generate GT pattern features — pass sample_seed for per-animal variation
        for i, pat in enumerate(GT_PATTERNS):
            msi_data[:, i] = generate_pattern_values(
                msi_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_msi.append(f"MZ_{pat}")
        
        # Generate realistic noise features — also per-sample
        for i in range(NOISE_FEATURES):
            msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                msi_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_msi.append(f"MZ_Noise_{i}")
            
        obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
        obs_msi['x_um'] = msi_coords[:, 0]
        obs_msi['y_um'] = msi_coords[:, 1]
        
        adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
        adata_msi.var_names = var_names_msi
        adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))
        
        # 3. Generate RNA Data (Visium-like hexagonal grid)
        raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
        mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
        rna_coords = raw_rna_coords[mask_rna]
        
        rna_data = np.zeros((len(rna_coords), n_total_features))
        var_names_rna = []
        
        # Generate GT pattern features — same sample_seed so RNA and MSI match
        for i, pat in enumerate(GT_PATTERNS):
            rna_data[:, i] = generate_pattern_values(
                rna_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_rna.append(f"Gene_{pat}")
            
            if group_idx == 0 and sample_idx == 1:
                ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
        
        # Generate realistic noise features — per-sample
        for i in range(NOISE_FEATURES):
            rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                rna_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_rna.append(f"Gene_Noise_{i}")
        
        obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
        obs_rna['x_um'] = rna_coords[:, 0]
        obs_rna['y_um'] = rna_coords[:, 1]
        
        adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
        adata_rna.var_names = var_names_rna
        adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

print("\n" + "=" * 60)
print("DONE! Data generated.")
print("=" * 60)
print("\n50 GROUND TRUTH MATCHES:")
print("-" * 60)
for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
    print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
print("-" * 60)
print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, per-sample variation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Field size: 6000x6000 µm
Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2
RNA grid: Visium-like hexagonal, 100µm spacing
MSI grid: Cartesian, 60µm spacing
Pattern size scale range: 0.85–1.15
Output: dummy_data_50
  Processing YC_1 (seed=1761886413)...
  Processing YC_2 (seed=2487579961)...
  Processing YC_3 (seed=2961126115)...
  Processing YC_4 (seed=2169539358)...
  Processing YAD_1 (seed=1716195892)...
  Processing YAD_2 (seed=3557677097)...
  Processing YAD_3 (seed=2169944517)...
  Processing YAD_4 (seed=2843319265)...
  Processing AC_1 (seed=2547557736)...
  Processing AC_2 (seed=3798791526)...
  Processing AC_3 (seed=1970269661)...
  Processing AC_4 (seed=2477310936)...
  Processing AAD_1 (seed=3234924883)...
  Processing AAD_2 (seed=2408467896)...
  Processing AAD_3 (seed=3651347787)...
  P

In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# Per-sample pattern size variation range.
# Each animal draws a size_scale ~ Uniform(low, high) per pattern, controlling
# blob sigmas, ring radii, spot sizes, band widths, etc.
# (1.0, 1.0) = no size variation; (0.5, 1.5) = aggressive variation.
PATTERN_SIZE_SCALE_RANGE = (0.85, 1.15)

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",              # Left to right
    "Gradient_Y",              # Bottom to top
    "Gradient_Diagonal_NE",    # Southwest to Northeast
    "Gradient_Diagonal_NW",    # Southeast to Northwest
    "Gradient_Radial_In",      # Center bright, edges dark
    "Gradient_Radial_Out",     # Edges bright, center dark
    
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",        # Vertical sinusoidal
    "Stripes_Horizontal",      # Horizontal sinusoidal
    "Stripes_Diagonal_45",     # 45-degree stripes
    "Stripes_Diagonal_135",    # 135-degree stripes
    "Waves_Concentric",        # Concentric circles (bullseye)
    "Waves_Spiral",            # Spiral pattern
    "Waves_Interference",      # Two-source interference
    "Waves_Ripple",            # Decaying ripples from center
    
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",             # Gaussian blob center
    "Blob_TopRight",           # Gaussian blob top-right
    "Blob_TopLeft",            # Gaussian blob top-left
    "Blob_BottomRight",        # Gaussian blob bottom-right
    "Blob_BottomLeft",         # Gaussian blob bottom-left
    "Spots_Grid_Dense",        # Dense polka dots
    "Spots_Grid_Sparse",       # Sparse polka dots
    "Spots_Random_Large",      # Large random spots
    "Spots_Triangular",        # Triangular arrangement
    "Spots_Hexagonal",         # Hexagonal arrangement
    
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",              # Inner ring
    "Ring_Outer",              # Outer ring
    "Ring_Double",             # Double concentric rings
    "Ring_Eccentric",          # Off-center ring
    "Ring_Elliptical",         # Elliptical ring
    "Ring_Partial",            # Partial arc/ring
    
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",       # Fine checkerboard
    "Checkerboard_Coarse",     # Coarse checkerboard
    "Quadrant_Alternating",    # Alternating quadrants
    "Sectors_4",               # 4 pie sectors
    "Sectors_8",               # 8 pie sectors
    "Triangle_Pattern",        # Triangular tessellation
    "Diamond_Pattern",         # Diamond/rhombus pattern
    "Honeycomb",               # Honeycomb hexagonal
    
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",         # Layered like cortex
    "Hotspot_Cluster",         # Multiple clustered hotspots
    "Edge_Enhancement",        # Enhanced at tissue edges
    "Core_Shell",              # Core-shell structure
    "Branching",               # Branching/dendritic pattern
    "Laminar_Curved",          # Curved laminar structure
    "Mosaic_Irregular",        # Irregular mosaic
    "Gradient_Sigmoid",        # Sigmoid transition
    "Bimodal_Distribution",    # Two distinct regions
    "Punctate_Dense",          # Dense punctate pattern
    "Periventricular",         # Around a central void
    "Asymmetric_Lobe",         # Asymmetric lobed structure
]

NOISE_FEATURES = 500  # Reduced but more realistic noise features

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    """
    np.random.seed(sample_seed)
    
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height
    
    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        
        if x_rot < left_edge:
            return False
        if y_rot < bottom_edge:
            return False
        if y_rot > top_edge:
            return False
        
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        
        right_edge = left_edge + right_width
        
        if x_rot > right_edge:
            return False
        
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            
            if y_norm > max_y_norm:
                return False
        
        return True
    
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    """
    Generates intensity values (0-1) based on spatial coordinates.

    Per-sample variation is introduced via `sample_seed`, which perturbs:
      - Spatial offsets (blob/ring centres shifted by up to ±15% of field)
      - Size scaling (blob sigmas, ring radii, band widths — range set by
        size_scale_range, passed from PATTERN_SIZE_SCALE_RANGE in config)
      - Phase offsets (stripes/waves shifted by ±1 period)
      - Frequency jitter (stripe/spot period varies ±25%)
      - Amplitude scaling (modest intensity rescaling)
      - Additive biological noise (0–10% σ)

    The perturbations keep matched patterns correlated across animals while
    making each sample visually distinct.

    Parameters
    ----------
    coords : np.ndarray  (N, 2)
    pattern_type : str
    width, height : int
        Field dimensions.
    sample_seed : int or None
        Per-sample integer seed.  When None the function behaves exactly as
        the original (all samples identical).
    add_noise : bool
        Whether to add small biological noise (default True).
    size_scale_range : tuple (low, high)
        Uniform range from which the per-sample size_scale is drawn.
        Controls how much larger or smaller features are between animals.
        (1.0, 1.0) disables size variation; (0.5, 1.5) is aggressive.
        Passed in from PATTERN_SIZE_SCALE_RANGE in the config.
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    
    # Normalized coordinates (0-1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    
    # Distance from center
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    
    # Angle from center
    angle = np.arctan2(y - cy, x - cx)
    
    # -------------------------------------------------------------------------
    # Per-pattern deterministic seed (unchanged from original)
    # -------------------------------------------------------------------------
    pattern_seed = hash(pattern_type) % 2**31

    # -------------------------------------------------------------------------
    # Per-sample perturbation parameters
    # Each parameter is a small, reproducible deviation driven by sample_seed.
    # -------------------------------------------------------------------------
    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)

        # Spatial offset: shift blob/ring/spot centres by up to ±15% of field
        # (noticeably different locations across animals)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height

        # Phase offset for wave patterns (±1 full period shift)
        phase_offset = rng_s.uniform(-1.0, 1.0)

        # Frequency jitter for stripe/spot patterns (±25%)
        freq_jitter = rng_s.uniform(0.75, 1.25)

        # Size scale: drawn from size_scale_range (set via PATTERN_SIZE_SCALE_RANGE
        # in config). Controls how much features grow/shrink between animals.
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])

        # Amplitude scale: modest variation in overall intensity
        amp_scale = rng_s.uniform(0.75, 1.05)

        # Biological noise level (extra additive noise, 0–10%)
        bio_noise_std = rng_s.uniform(0.0, 0.10)

        # Shifted centre coordinates used by blob/ring patterns
        cx_s = cx + dx_offset
        cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0
        freq_jitter = 1.0
        size_scale = 1.0
        amp_scale = 1.0
        bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center
        angle_s = angle

    # =========================================================================
    # PATTERN DEFINITIONS
    # =========================================================================

    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Y":
        val = y_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm + phase_offset * 0.1
        
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm + phase_offset * 0.1
        
    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
        
    elif pattern_type == "Waves_Interference":
        # Two point sources, centres shifted per sample
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
        
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Spots_Grid_Dense":
        # size_scale controls sharpness of each spot (wider = softer cutoff)
        base_freq = 100 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        # Raise to 1/size_scale: <1 broadens spots, >1 sharpens them
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.1 * size_scale
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset
            by = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter
        spot_sigma = 80 * size_scale
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter
        spot_sigma = 60 * size_scale
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        # size_scale shifts the ring radius (larger = ring sits further out)
        target_r = max_dist * (0.2 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.04 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Double":
        r1 = max_dist * (0.25 * size_scale + phase_offset * 0.03)
        r2 = max_dist * (0.55 * size_scale + phase_offset * 0.03)
        ring_width = width * 0.03 * size_scale
        val = np.exp(-((dist_center_s - r1)**2) / (2 * ring_width**2)) + \
              np.exp(-((dist_center_s - r2)**2) / (2 * ring_width**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15 + dx_offset
        off_cy = cy + height * 0.1 + dy_offset
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * (0.35 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_off - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Elliptical":
        # size_scale stretches/shrinks the ellipse axes
        a = max_dist * (0.5 * size_scale + phase_offset * 0.05)
        b = max_dist * (0.3 * size_scale + phase_offset * 0.03)
        ellipse_dist = np.sqrt(((x - cx_s)/a)**2 + ((y - cy_s)/b)**2)
        ring_width = 0.1 * size_scale
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * (0.4 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        ring_val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        # Angular mask shifts slightly per sample; arc width also varies
        arc_half = (np.pi/2) * size_scale
        a_start = -arc_half + phase_offset * 0.3
        a_end   =  arc_half + phase_offset * 0.3
        angle_mask = (angle_s > a_start) & (angle_s < a_end)
        val = ring_val * angle_mask
        
    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx_s) & (y > cy_s)
        q3 = (x < cx_s) & (y < cy_s)
        val = (q1 | q3).astype(float)
        
    elif pattern_type == "Sectors_4":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Sectors_8":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Triangle_Pattern":
        period = 300 * freq_jitter
        tri_x = np.abs(((x + dx_offset) % period) - period/2) / (period/2)
        tri_y = np.abs(((y + dy_offset) % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
        
    elif pattern_type == "Diamond_Pattern":
        period = 400 * freq_jitter
        val = (np.abs(((x + dx_offset) % period) - period/2) +
               np.abs(((y + dy_offset) % period) - period/2)) / period
        
    elif pattern_type == "Honeycomb":
        spacing = 200 * freq_jitter
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                hy = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                # size_scale changes how thick the hex walls appear
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4 * size_scale), 0, 1))
        val = 1 - val
        
    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        # size_scale compresses/expands the layer spacing
        shifted_dist = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        shifted_norm = shifted_dist / (shifted_dist.max() + 1e-8)
        # Divide by size_scale: small = layers are wider (more spread out)
        scaled_norm = np.clip(shifted_norm / size_scale, 0, 1)
        layer_idx = np.clip((scaled_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
        
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.05 * size_scale
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1) + dx_offset
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1) + dy_offset
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Edge_Enhancement":
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        # size_scale moves the enhanced edge inward or outward
        target = 0.8 * size_scale + phase_offset * 0.05
        target = np.clip(target, 0.3, 1.2)
        band_width = 0.15 * size_scale
        val = 1 - np.exp(-((shifted_norm - target)**2) / (2 * band_width**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Core_Shell":
        core_sigma = width * 0.1 * size_scale
        shell_r    = max_dist * (0.5 + phase_offset * 0.05) * size_scale
        shell_sigma = width * 0.08 * size_scale
        core  = np.exp(-dist_center_s**2 / (2 * core_sigma**2))
        shell = np.exp(-((dist_center_s - shell_r)**2) / (2 * shell_sigma**2))
        val = 0.7 * core + 0.5 * shell
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Branching":
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        # size_scale widens/narrows each branch
        branch_width = 0.3 * size_scale
        rotated_angle = angle_s + phase_offset * 0.5
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(rotated_angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
        
    elif pattern_type == "Laminar_Curved":
        # size_scale changes the thickness of each laminar band
        curve_amp  = 200 * (1 + phase_offset * 0.3)
        curve_freq = 400 * freq_jitter
        band_thick = 300 * size_scale
        curved_y = (y + dy_offset) - curve_amp * np.sin((x + dx_offset) / curve_freq)
        layer_idx = (curved_y // band_thick).astype(int) % 2
        val = layer_idx.astype(float)
        
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed)
        # size_scale adjusts the number of Voronoi tiles (fewer = larger patches)
        n_seeds = max(5, int(20 / size_scale))
        seed_x = rng.uniform(x.min(), x.max(), n_seeds) + dx_offset
        seed_y = rng.uniform(y.min(), y.max(), n_seeds) + dy_offset
        seed_val = rng.rand(n_seeds)
        nearest_idx = np.zeros(n, dtype=int)
        min_dist_arr = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist_arr
            nearest_idx[closer] = i
            min_dist_arr[closer] = dist[closer]
        val = seed_val[nearest_idx]
        
    elif pattern_type == "Gradient_Sigmoid":
        # size_scale controls steepness of the transition (sharper vs smoother)
        shift = 0.5 + phase_offset * 0.1
        steepness = 10.0 / size_scale   # smaller size_scale = steeper step
        val = 1 / (1 + np.exp(-steepness * (x_norm - shift)))
        
    elif pattern_type == "Bimodal_Distribution":
        region1 = (x < cx_s).astype(float) * 0.8
        region2 = (x >= cx_s).astype(float) * 0.3
        val = region1 + region2
        
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed)
        punctum_sigma = 40 * size_scale
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max()) + dx_offset
            py = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * punctum_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Periventricular":
        # size_scale enlarges/shrinks both the void and the periventricular ring
        void_r = 0.15 * size_scale + phase_offset * 0.02
        ring_r = 0.25 * size_scale + phase_offset * 0.02
        ring_w = 0.08 * size_scale
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        void_mask = shifted_norm < void_r
        ring_val  = np.exp(-((shifted_norm - ring_r)**2) / (2 * ring_w**2))
        val = ring_val * ~void_mask
        
    elif pattern_type == "Asymmetric_Lobe":
        lobe1_x = cx - width*0.2 + dx_offset
        lobe1_y = cy              + dy_offset
        lobe2_x = cx + width*0.15 + dx_offset
        lobe2_y = cy + height*0.1 + dy_offset
        sigma1 = width  * 0.15 * size_scale
        sigma2 = width  * 0.10 * size_scale
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * sigma1**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * sigma2**2))
        val = val / (val.max() + 1e-8)
        
    else:
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)
    
    # -------------------------------------------------------------------------
    # Apply per-sample amplitude scaling
    # -------------------------------------------------------------------------
    val = val * amp_scale

    # -------------------------------------------------------------------------
    # Add biological noise (per-sample, reproducible)
    # -------------------------------------------------------------------------
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000,
                             sample_seed=None, size_scale_range=(0.70, 1.40)):
    """
    Generate noise features that are correlated across animals for the same
    noise_idx, with modest per-sample perturbations — exactly mirroring
    generate_pattern_values.

    Structure seed  (noise_idx only)    → same base shape across all animals
    Perturbation RNG (noise_idx + seed) → small shifts/scales differ per animal

    Key design rules so correlation stays high:
      - dx/dy offsets are small (±5% of field) to avoid relocating features
      - For periodic patterns, freq_jitter is mild (±10%) and no phase flipping
      - For GT-delegating types (4,5,6), GT patterns already carry per-sample
        variation, so no extra spatial offset is added
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    cx_n, cy_n = x.mean(), y.mean()

    # ------------------------------------------------------------------
    # Structure RNG — noise_idx only, identical across all animals.
    # ------------------------------------------------------------------
    struct_seed = noise_idx * 7 + 42
    rng_struct  = np.random.RandomState(struct_seed)

    noise_type = noise_idx % 10

    # Pre-draw all structural parameters (order fixed — drawn unconditionally)
    base_theta    = rng_struct.uniform(0, 2*np.pi)
    base_bx       = rng_struct.uniform(x.min()-200, x.max()+200)
    base_by       = rng_struct.uniform(y.min()-200, y.max()+200)
    base_sigma_b  = rng_struct.uniform(300, 700)
    base_freq     = rng_struct.uniform(80, 300)
    base_angle_d  = rng_struct.uniform(0, np.pi)
    base_target_r = rng_struct.uniform(0.15, 0.55)   # fraction of max_dist
    base_sigma_r  = rng_struct.uniform(0.04, 0.12)   # fraction of max_dist
    base_p1_idx   = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_p2_idx   = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_mix_w    = rng_struct.uniform(0.3, 0.7)
    base_inv_idx  = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_thr_idx  = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_threshold= rng_struct.uniform(0.35, 0.65)
    base_freq_x   = rng_struct.uniform(300, 800)
    base_freq_y   = rng_struct.uniform(300, 800)
    base_phase_x  = rng_struct.uniform(0, 2*np.pi)
    base_phase_y  = rng_struct.uniform(0, 2*np.pi)
    base_n_spots  = rng_struct.randint(8, 25)
    base_spot_xs  = rng_struct.uniform(x.min(), x.max(), 30)
    base_spot_ys  = rng_struct.uniform(y.min(), y.max(), 30)
    base_spot_sigs= rng_struct.uniform(100, 300, 30)
    base_noise_v  = rng_struct.rand(n)

    # ------------------------------------------------------------------
    # Per-sample perturbation RNG — small offsets to keep correlation high
    # ------------------------------------------------------------------
    if sample_seed is not None:
        perturb_seed = (struct_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)

        # Small spatial offset (±5%) — features stay largely overlapping
        dx_offset    = rng_p.uniform(-0.05, 0.05) * width
        dy_offset    = rng_p.uniform(-0.05, 0.05) * height
        # Mild frequency jitter (±10%) — won't decorrelate periodic patterns
        freq_jitter  = rng_p.uniform(0.90, 1.10)
        # Size scale from config range
        size_scale   = rng_p.uniform(size_scale_range[0], size_scale_range[1])
        # Rotation jitter for gradient (≤~8°)
        rot_jitter   = rng_p.uniform(-0.15, 0.15)
        amp_scale    = rng_p.uniform(0.80, 1.05)
        bio_noise_std= rng_p.uniform(0.0, 0.08)
    else:
        dx_offset = dy_offset = 0.0
        freq_jitter  = 1.0
        size_scale   = 1.0
        rot_jitter   = 0.0
        amp_scale    = 1.0
        bio_noise_std= 0.0

    # ------------------------------------------------------------------
    # Pattern construction
    # ------------------------------------------------------------------
    if noise_type == 0:
        # Rotated gradient — slight rotation per sample
        theta = base_theta + rot_jitter
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val   = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)

    elif noise_type == 1:
        # Gaussian blob — small centre shift and size variation per sample
        bx    = base_bx + dx_offset
        by    = base_by + dy_offset
        sigma = base_sigma_b * size_scale
        dist  = np.sqrt((x - bx)**2 + (y - by)**2)
        val   = np.exp(-dist**2 / (2 * sigma**2))

    elif noise_type == 2:
        # Directional stripe — only direction rotates slightly per sample.
        # Use a wide-stripe frequency (×4) so small angle jitter doesn't
        # decorrelate the pattern across animals.
        angle_dir = base_angle_d + rot_jitter * 0.3
        proj      = x * np.cos(angle_dir) + y * np.sin(angle_dir)
        val       = (np.sin(proj / (base_freq * 4.0)) + 1) / 2

    elif noise_type == 3:
        # Ring — relative radius keeps ring visible; small centre shift per sample
        cx_r     = cx_n + dx_offset
        cy_r     = cy_n + dy_offset
        dist     = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        max_r    = dist.max() + 1e-8
        target_r = max_r * base_target_r * np.clip(size_scale, 0.9, 1.1)
        sigma    = max_r * base_sigma_r  * np.clip(size_scale, 0.9, 1.1)
        val      = np.exp(-((dist - target_r)**2) / (2 * sigma**2))

    elif noise_type == 4:
        # Mixture of two GT patterns — use structural (seed=None) base so
        # the pattern shape is identical across animals; per-sample variation
        # comes only from the small bio_noise added at the end.
        p1  = generate_pattern_values(coords, gt_patterns_list[base_p1_idx],
                                      width, height, sample_seed=None)
        p2  = generate_pattern_values(coords, gt_patterns_list[base_p2_idx],
                                      width, height, sample_seed=None)
        val = base_mix_w * p1 + (1 - base_mix_w) * p2

    elif noise_type == 5:
        # Inverted GT pattern — structural base for consistent shape across animals
        base_pattern = generate_pattern_values(coords, gt_patterns_list[base_inv_idx],
                                               width, height, sample_seed=None)
        val = 1 - base_pattern

    elif noise_type == 6:
        # Soft-thresholded GT pattern — sigmoid instead of hard step so
        # per-sample amplitude variation doesn't create all-0/all-1 outputs.
        base_pattern = generate_pattern_values(coords, gt_patterns_list[base_thr_idx],
                                               width, height, sample_seed=None)
        # Sigmoid centred on base_threshold; steepness 12 gives a clear edge
        val = 1.0 / (1.0 + np.exp(-12.0 * (base_pattern - base_threshold)))

    elif noise_type == 7:
        # 2-D sinusoidal field — only frequency varies per sample (no phase shift)
        val = (np.sin(x / (base_freq_x / freq_jitter) + base_phase_x) *
               np.sin(y / (base_freq_y / freq_jitter) + base_phase_y) + 1) / 2

    elif noise_type == 8:
        # Scattered spots — positions are structural (no per-sample offset so
        # spots stay aligned across animals); only sigma varies per sample.
        val = np.zeros(n)
        for k in range(base_n_spots):
            sx    = base_spot_xs[k]
            sy    = base_spot_ys[k]
            sigma = base_spot_sigs[k] * size_scale
            dist  = np.sqrt((x - sx)**2 + (y - sy)**2)
            val  += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)

    else:
        # Smooth correlated field: mostly structural values, slight x-gradient shift
        val = (0.6 * base_noise_v +
               0.2 * (x - x.min()) / (x.max() - x.min() + 1e-8) +
               0.2 * (y - y.min()) / (y.max() - y.min() + 1e-8))

    # Amplitude scaling
    val = val * amp_scale

    # Per-sample biological noise
    if sample_seed is not None and bio_noise_std > 0:
        rng_bio = np.random.RandomState((struct_seed ^ sample_seed) + 2)
        val = val + rng_bio.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)

def get_visium_hex_grid(width, height, spacing=100):
    """Generates coordinates for a Visium-like hexagonal grid."""
    dy = spacing * np.sqrt(3) / 2
    
    coords = []
    row = 0
    y = 0
    
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        
        while x < width:
            coords.append([x, y])
            x += spacing
        
        y += dy
        row += 1
    
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000

print("=" * 60)
print("Generating Ground Truth Dummy Data with 50 PATTERNS")
print("Half-brain shapes, Visium-like RNA grid, per-sample variation")
print("=" * 60)
print(f"Groups: {GROUPS}")
print(f"GT Patterns: {len(GT_PATTERNS)}")
print(f"Noise Features: {NOISE_FEATURES}")
print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} µm")
print(f"Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2")
print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}µm spacing")
print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}µm spacing")
print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}–{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

ground_truth_pairs = []

for group_idx, group in enumerate(GROUPS):
    for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
        sample_id = f"{group}_{sample_idx}"

        # ------------------------------------------------------------------
        # Derive a unique integer seed per sample.
        # This seed drives both tissue shape AND per-sample pattern variation.
        # ------------------------------------------------------------------
        seed = hash(sample_id) % 2**32

        print(f"  Processing {sample_id} (seed={seed})...")
        
        # 1. Define Half-Brain Tissue Shape
        tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)
        
        # 2. Generate MSI Data (Cartesian grid)
        raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
        mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
        msi_coords = raw_msi_coords[mask_msi]
        
        n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
        msi_data = np.zeros((len(msi_coords), n_total_features))
        var_names_msi = []
        
        # Generate GT pattern features — pass sample_seed for per-animal variation
        for i, pat in enumerate(GT_PATTERNS):
            msi_data[:, i] = generate_pattern_values(
                msi_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_msi.append(f"MZ_{pat}")
        
        # Generate realistic noise features — also per-sample
        for i in range(NOISE_FEATURES):
            msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                msi_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_msi.append(f"MZ_Noise_{i}")
            
        obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
        obs_msi['x_um'] = msi_coords[:, 0]
        obs_msi['y_um'] = msi_coords[:, 1]
        
        adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
        adata_msi.var_names = var_names_msi
        adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))
        
        # 3. Generate RNA Data (Visium-like hexagonal grid)
        raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
        mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
        rna_coords = raw_rna_coords[mask_rna]
        
        rna_data = np.zeros((len(rna_coords), n_total_features))
        var_names_rna = []
        
        # Generate GT pattern features — same sample_seed so RNA and MSI match
        for i, pat in enumerate(GT_PATTERNS):
            rna_data[:, i] = generate_pattern_values(
                rna_coords, pat,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_rna.append(f"Gene_{pat}")
            
            if group_idx == 0 and sample_idx == 1:
                ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
        
        # Generate realistic noise features — per-sample
        for i in range(NOISE_FEATURES):
            rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                rna_coords, i, GT_PATTERNS,
                width=FIELD_WIDTH, height=FIELD_HEIGHT,
                sample_seed=seed,
                size_scale_range=PATTERN_SIZE_SCALE_RANGE
            )
            var_names_rna.append(f"Gene_Noise_{i}")
        
        obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
        obs_rna['x_um'] = rna_coords[:, 0]
        obs_rna['y_um'] = rna_coords[:, 1]
        
        adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
        adata_rna.var_names = var_names_rna
        adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

print("\n" + "=" * 60)
print("DONE! Data generated.")
print("=" * 60)
print("\n50 GROUND TRUTH MATCHES:")
print("-" * 60)
for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
    print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
print("-" * 60)
print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, per-sample variation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Field size: 6000x6000 µm
Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2
RNA grid: Visium-like hexagonal, 100µm spacing
MSI grid: Cartesian, 60µm spacing
Pattern size scale range: 0.85–1.15
Output: dummy_data_50
  Processing YC_1 (seed=321968443)...
  Processing YC_2 (seed=347226907)...
  Processing YC_3 (seed=3182348929)...
  Processing YC_4 (seed=4114315907)...
  Processing YAD_1 (seed=7651968)...
  Processing YAD_2 (seed=2917800819)...
  Processing YAD_3 (seed=191746656)...
  Processing YAD_4 (seed=891620259)...
  Processing AC_1 (seed=1504943285)...
  Processing AC_2 (seed=968954664)...
  Processing AC_3 (seed=3690293104)...
  Processing AC_4 (seed=566065078)...
  Processing AAD_1 (seed=728933825)...
  Processing AAD_2 (seed=3595572711)...
  Processing AAD_3 (seed=2883678813)...
  Processing 

In [3]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# Per-sample pattern size variation range.
# Each animal draws a size_scale ~ Uniform(low, high) per pattern, controlling
# blob sigmas, ring radii, spot sizes, band widths, etc.
# (1.0, 1.0) = no size variation; (0.5, 1.5) = aggressive variation.
PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

# =============================================================================
# NOISE DISSIMILARITY CONTROL
# =============================================================================
# Controls how different noise features are from ground truth patterns.
# Range: 1 (minimal difference — noise closely resembles GT patterns)
#     to 10 (maximal difference — noise is heavily distorted/unrecognizable).
# Default 5 provides moderate separation.
#
# Mechanistically, this parameter affects:
#   - Spatial offsets applied to GT-derived noise (types that remix/invert GT)
#   - Frequency/phase distortion magnitude for periodic noise types
#   - Number of components mixed together (more mixing = less like any single GT)
#   - Gaussian blur/sharpening applied as post-processing
#   - Additional structured perturbations layered on top
#
# At level 1: noise features are only mildly perturbed from GT-like structures.
# At level 5: clear structural differences but still spatially structured.
# At level 10: heavy distortion; noise features bear little resemblance to GT.
NOISE_DISSIMILARITY = 5  # <-- USER-ADJUSTABLE: integer 1–10

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",              # Left to right
    "Gradient_Y",              # Bottom to top
    "Gradient_Diagonal_NE",    # Southwest to Northeast
    "Gradient_Diagonal_NW",    # Southeast to Northwest
    "Gradient_Radial_In",      # Center bright, edges dark
    "Gradient_Radial_Out",     # Edges bright, center dark
    
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",        # Vertical sinusoidal
    "Stripes_Horizontal",      # Horizontal sinusoidal
    "Stripes_Diagonal_45",     # 45-degree stripes
    "Stripes_Diagonal_135",    # 135-degree stripes
    "Waves_Concentric",        # Concentric circles (bullseye)
    "Waves_Spiral",            # Spiral pattern
    "Waves_Interference",      # Two-source interference
    "Waves_Ripple",            # Decaying ripples from center
    
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",             # Gaussian blob center
    "Blob_TopRight",           # Gaussian blob top-right
    "Blob_TopLeft",            # Gaussian blob top-left
    "Blob_BottomRight",        # Gaussian blob bottom-right
    "Blob_BottomLeft",         # Gaussian blob bottom-left
    "Spots_Grid_Dense",        # Dense polka dots
    "Spots_Grid_Sparse",       # Sparse polka dots
    "Spots_Random_Large",      # Large random spots
    "Spots_Triangular",        # Triangular arrangement
    "Spots_Hexagonal",         # Hexagonal arrangement
    
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",              # Inner ring
    "Ring_Outer",              # Outer ring
    "Ring_Double",             # Double concentric rings
    "Ring_Eccentric",          # Off-center ring
    "Ring_Elliptical",         # Elliptical ring
    "Ring_Partial",            # Partial arc/ring
    
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",       # Fine checkerboard
    "Checkerboard_Coarse",     # Coarse checkerboard
    "Quadrant_Alternating",    # Alternating quadrants
    "Sectors_4",               # 4 pie sectors
    "Sectors_8",               # 8 pie sectors
    "Triangle_Pattern",        # Triangular tessellation
    "Diamond_Pattern",         # Diamond/rhombus pattern
    "Honeycomb",               # Honeycomb hexagonal
    
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",         # Layered like cortex
    "Hotspot_Cluster",         # Multiple clustered hotspots
    "Edge_Enhancement",        # Enhanced at tissue edges
    "Core_Shell",              # Core-shell structure
    "Branching",               # Branching/dendritic pattern
    "Laminar_Curved",          # Curved laminar structure
    "Mosaic_Irregular",        # Irregular mosaic
    "Gradient_Sigmoid",        # Sigmoid transition
    "Bimodal_Distribution",    # Two distinct regions
    "Punctate_Dense",          # Dense punctate pattern
    "Periventricular",         # Around a central void
    "Asymmetric_Lobe",         # Asymmetric lobed structure
]

NOISE_FEATURES = 500  # Reduced but more realistic noise features

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

# Number of noise types now scales with the number of GT pattern categories
# 10 original + 10 new = 20 total, covering all pattern families
N_NOISE_TYPES = 20


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    """
    np.random.seed(sample_seed)
    
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height
    
    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        
        if x_rot < left_edge:
            return False
        if y_rot < bottom_edge:
            return False
        if y_rot > top_edge:
            return False
        
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        
        right_edge = left_edge + right_width
        
        if x_rot > right_edge:
            return False
        
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            
            if y_norm > max_y_norm:
                return False
        
        return True
    
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    """
    Generates intensity values (0-1) based on spatial coordinates.

    Per-sample variation is introduced via `sample_seed`, which perturbs:
      - Spatial offsets (blob/ring centres shifted by up to ±15% of field)
      - Size scaling (blob sigmas, ring radii, band widths — range set by
        size_scale_range, passed from PATTERN_SIZE_SCALE_RANGE in config)
      - Phase offsets (stripes/waves shifted by ±1 period)
      - Frequency jitter (stripe/spot period varies ±25%)
      - Amplitude scaling (modest intensity rescaling)
      - Additive biological noise (0–10% σ)

    The perturbations keep matched patterns correlated across animals while
    making each sample visually distinct.

    Parameters
    ----------
    coords : np.ndarray  (N, 2)
    pattern_type : str
    width, height : int
        Field dimensions.
    sample_seed : int or None
        Per-sample integer seed.  When None the function behaves exactly as
        the original (all samples identical).
    add_noise : bool
        Whether to add small biological noise (default True).
    size_scale_range : tuple (low, high)
        Uniform range from which the per-sample size_scale is drawn.
        Controls how much larger or smaller features are between animals.
        (1.0, 1.0) disables size variation; (0.5, 1.5) is aggressive.
        Passed in from PATTERN_SIZE_SCALE_RANGE in the config.
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    
    # Normalized coordinates (0-1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    
    # Distance from center
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    
    # Angle from center
    angle = np.arctan2(y - cy, x - cx)
    
    # -------------------------------------------------------------------------
    # Per-pattern deterministic seed (unchanged from original)
    # -------------------------------------------------------------------------
    pattern_seed = hash(pattern_type) % 2**31

    # -------------------------------------------------------------------------
    # Per-sample perturbation parameters
    # Each parameter is a small, reproducible deviation driven by sample_seed.
    # -------------------------------------------------------------------------
    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)

        # Spatial offset: shift blob/ring/spot centres by up to ±15% of field
        # (noticeably different locations across animals)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height

        # Phase offset for wave patterns (±1 full period shift)
        phase_offset = rng_s.uniform(-1.0, 1.0)

        # Frequency jitter for stripe/spot patterns (±25%)
        freq_jitter = rng_s.uniform(0.75, 1.25)

        # Size scale: drawn from size_scale_range (set via PATTERN_SIZE_SCALE_RANGE
        # in config). Controls how much features grow/shrink between animals.
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])

        # Amplitude scale: modest variation in overall intensity
        amp_scale = rng_s.uniform(0.75, 1.05)

        # Biological noise level (extra additive noise, 0–10%)
        bio_noise_std = rng_s.uniform(0.0, 0.10)

        # Shifted centre coordinates used by blob/ring patterns
        cx_s = cx + dx_offset
        cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0
        freq_jitter = 1.0
        size_scale = 1.0
        amp_scale = 1.0
        bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center
        angle_s = angle

    # =========================================================================
    # PATTERN DEFINITIONS
    # =========================================================================

    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Y":
        val = y_norm + phase_offset * 0.2
        
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
        
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm + phase_offset * 0.1
        
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm + phase_offset * 0.1
        
    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
        
    elif pattern_type == "Waves_Interference":
        # Two point sources, centres shifted per sample
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
        
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
        
    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif pattern_type == "Spots_Grid_Dense":
        # size_scale controls sharpness of each spot (wider = softer cutoff)
        base_freq = 100 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        # Raise to 1/size_scale: <1 broadens spots, >1 sharpens them
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
        
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.1 * size_scale
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset
            by = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter
        spot_sigma = 80 * size_scale
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter
        spot_sigma = 60 * size_scale
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
        
    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        # size_scale shifts the ring radius (larger = ring sits further out)
        target_r = max_dist * (0.2 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.04 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Double":
        r1 = max_dist * (0.25 * size_scale + phase_offset * 0.03)
        r2 = max_dist * (0.55 * size_scale + phase_offset * 0.03)
        ring_width = width * 0.03 * size_scale
        val = np.exp(-((dist_center_s - r1)**2) / (2 * ring_width**2)) + \
              np.exp(-((dist_center_s - r2)**2) / (2 * ring_width**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15 + dx_offset
        off_cy = cy + height * 0.1 + dy_offset
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * (0.35 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_off - target_r)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Elliptical":
        # size_scale stretches/shrinks the ellipse axes
        a = max_dist * (0.5 * size_scale + phase_offset * 0.05)
        b = max_dist * (0.3 * size_scale + phase_offset * 0.03)
        ellipse_dist = np.sqrt(((x - cx_s)/a)**2 + ((y - cy_s)/b)**2)
        ring_width = 0.1 * size_scale
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * ring_width**2))
        
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * (0.4 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        ring_val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        # Angular mask shifts slightly per sample; arc width also varies
        arc_half = (np.pi/2) * size_scale
        a_start = -arc_half + phase_offset * 0.3
        a_end   =  arc_half + phase_offset * 0.3
        angle_mask = (angle_s > a_start) & (angle_s < a_end)
        val = ring_val * angle_mask
        
    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx_s) & (y > cy_s)
        q3 = (x < cx_s) & (y < cy_s)
        val = (q1 | q3).astype(float)
        
    elif pattern_type == "Sectors_4":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Sectors_8":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Triangle_Pattern":
        period = 300 * freq_jitter
        tri_x = np.abs(((x + dx_offset) % period) - period/2) / (period/2)
        tri_y = np.abs(((y + dy_offset) % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
        
    elif pattern_type == "Diamond_Pattern":
        period = 400 * freq_jitter
        val = (np.abs(((x + dx_offset) % period) - period/2) +
               np.abs(((y + dy_offset) % period) - period/2)) / period
        
    elif pattern_type == "Honeycomb":
        spacing = 200 * freq_jitter
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                hy = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                # size_scale changes how thick the hex walls appear
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4 * size_scale), 0, 1))
        val = 1 - val
        
    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        # size_scale compresses/expands the layer spacing
        shifted_dist = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        shifted_norm = shifted_dist / (shifted_dist.max() + 1e-8)
        # Divide by size_scale: small = layers are wider (more spread out)
        scaled_norm = np.clip(shifted_norm / size_scale, 0, 1)
        layer_idx = np.clip((scaled_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
        
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.05 * size_scale
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1) + dx_offset
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1) + dy_offset
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Edge_Enhancement":
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        # size_scale moves the enhanced edge inward or outward
        target = 0.8 * size_scale + phase_offset * 0.05
        target = np.clip(target, 0.3, 1.2)
        band_width = 0.15 * size_scale
        val = 1 - np.exp(-((shifted_norm - target)**2) / (2 * band_width**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Core_Shell":
        core_sigma = width * 0.1 * size_scale
        shell_r    = max_dist * (0.5 + phase_offset * 0.05) * size_scale
        shell_sigma = width * 0.08 * size_scale
        core  = np.exp(-dist_center_s**2 / (2 * core_sigma**2))
        shell = np.exp(-((dist_center_s - shell_r)**2) / (2 * shell_sigma**2))
        val = 0.7 * core + 0.5 * shell
        val = val / (val.max() + 1e-8)
        
    elif pattern_type == "Branching":
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        # size_scale widens/narrows each branch
        branch_width = 0.3 * size_scale
        rotated_angle = angle_s + phase_offset * 0.5
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(rotated_angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
        
    elif pattern_type == "Laminar_Curved":
        # size_scale changes the thickness of each laminar band
        curve_amp  = 200 * (1 + phase_offset * 0.3)
        curve_freq = 400 * freq_jitter
        band_thick = 300 * size_scale
        curved_y = (y + dy_offset) - curve_amp * np.sin((x + dx_offset) / curve_freq)
        layer_idx = (curved_y // band_thick).astype(int) % 2
        val = layer_idx.astype(float)
        
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed)
        # size_scale adjusts the number of Voronoi tiles (fewer = larger patches)
        n_seeds = max(5, int(20 / size_scale))
        seed_x = rng.uniform(x.min(), x.max(), n_seeds) + dx_offset
        seed_y = rng.uniform(y.min(), y.max(), n_seeds) + dy_offset
        seed_val = rng.rand(n_seeds)
        nearest_idx = np.zeros(n, dtype=int)
        min_dist_arr = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist_arr
            nearest_idx[closer] = i
            min_dist_arr[closer] = dist[closer]
        val = seed_val[nearest_idx]
        
    elif pattern_type == "Gradient_Sigmoid":
        # size_scale controls steepness of the transition (sharper vs smoother)
        shift = 0.5 + phase_offset * 0.1
        steepness = 10.0 / size_scale   # smaller size_scale = steeper step
        val = 1 / (1 + np.exp(-steepness * (x_norm - shift)))
        
    elif pattern_type == "Bimodal_Distribution":
        region1 = (x < cx_s).astype(float) * 0.8
        region2 = (x >= cx_s).astype(float) * 0.3
        val = region1 + region2
        
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed)
        punctum_sigma = 40 * size_scale
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max()) + dx_offset
            py = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * punctum_sigma**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Periventricular":
        # size_scale enlarges/shrinks both the void and the periventricular ring
        void_r = 0.15 * size_scale + phase_offset * 0.02
        ring_r = 0.25 * size_scale + phase_offset * 0.02
        ring_w = 0.08 * size_scale
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        void_mask = shifted_norm < void_r
        ring_val  = np.exp(-((shifted_norm - ring_r)**2) / (2 * ring_w**2))
        val = ring_val * ~void_mask
        
    elif pattern_type == "Asymmetric_Lobe":
        lobe1_x = cx - width*0.2 + dx_offset
        lobe1_y = cy              + dy_offset
        lobe2_x = cx + width*0.15 + dx_offset
        lobe2_y = cy + height*0.1 + dy_offset
        sigma1 = width  * 0.15 * size_scale
        sigma2 = width  * 0.10 * size_scale
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * sigma1**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * sigma2**2))
        val = val / (val.max() + 1e-8)
        
    else:
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)
    
    # -------------------------------------------------------------------------
    # Apply per-sample amplitude scaling
    # -------------------------------------------------------------------------
    val = val * amp_scale

    # -------------------------------------------------------------------------
    # Add biological noise (per-sample, reproducible)
    # -------------------------------------------------------------------------
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def _apply_dissimilarity_distortion(val, coords, noise_idx, sample_seed,
                                    dissimilarity, width, height):
    """
    Post-processing step that distorts a noise feature proportionally to the
    NOISE_DISSIMILARITY level (1–10).  Applied uniformly to all noise types.

    Distortion mechanisms (each scaled by `d`, a 0–1 normalized dissimilarity):
      1. Multiplicative spatial modulation — overlays a low-frequency sinusoidal
         field that reshapes the intensity landscape.
      2. Additive structured noise — a second sinusoidal field at different
         frequency adds structure unrelated to GT patterns.
      3. Power-law warping — raises values to a data-dependent exponent,
         compressing or expanding the dynamic range.
      4. Gaussian additive noise — random per-pixel noise that degrades
         fine spatial correlations.

    At dissimilarity=1: almost no distortion (d≈0).
    At dissimilarity=10: all four mechanisms at full strength (d=1).
    """
    d = (dissimilarity - 1) / 9.0  # Normalize 1–10 → 0–1

    if d < 0.01:
        return val

    n = len(val)
    x, y = coords[:, 0], coords[:, 1]

    # Deterministic RNG for reproducibility
    rng_d = np.random.RandomState((noise_idx * 31 + 997) % 2**31)
    if sample_seed is not None:
        rng_d = np.random.RandomState((noise_idx * 31 + 997 + sample_seed) % 2**31)

    # 1. Multiplicative spatial modulation
    mod_freq_x = rng_d.uniform(200, 600)
    mod_freq_y = rng_d.uniform(200, 600)
    mod_phase  = rng_d.uniform(0, 2 * np.pi)
    modulation = (np.sin(x / mod_freq_x + mod_phase) *
                  np.cos(y / mod_freq_y + mod_phase * 0.7) + 1) / 2
    # Blend: at d=0 → val unchanged; at d=1 → val * modulation
    val = val * (1 - d * 0.4) + val * modulation * (d * 0.4)

    # 2. Additive structured noise (different spatial frequency)
    add_freq = rng_d.uniform(100, 400)
    add_phase = rng_d.uniform(0, 2 * np.pi)
    structured = (np.sin((x + y) / add_freq + add_phase) + 1) / 2
    val = val + structured * d * 0.25

    # 3. Power-law warping
    exponent = 1.0 + d * 1.5 * (rng_d.choice([-1, 1]))  # range [−0.5, 2.5]
    exponent = np.clip(exponent, 0.3, 2.5)
    val_clipped = np.clip(val, 0, 1)
    val = val_clipped ** exponent

    # 4. Gaussian additive noise
    noise_std = d * 0.15
    val = val + rng_d.normal(0, noise_std, size=n)

    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000,
                             sample_seed=None, size_scale_range=(0.70, 1.40),
                             dissimilarity=5):
    """
    Generate noise features that are correlated across animals for the same
    noise_idx, with modest per-sample perturbations — exactly mirroring
    generate_pattern_values.

    Structure seed  (noise_idx only)    → same base shape across all animals
    Perturbation RNG (noise_idx + seed) → small shifts/scales differ per animal

    Now uses N_NOISE_TYPES (20) types covering all GT pattern families:
      0: Rotated gradient              (covers: Gradients)
      1: Gaussian blob                 (covers: Blobs)
      2: Directional stripe            (covers: Stripes)
      3: Ring                          (covers: Rings)
      4: Mixture of two GT patterns    (covers: Complex combinations)
      5: Inverted GT pattern           (covers: Inversions)
      6: Soft-thresholded GT pattern   (covers: Threshold-based)
      7: 2-D sinusoidal field          (covers: Waves periodic)
      8: Scattered spots               (covers: Spots)
      9: Smooth correlated field       (covers: Smooth noise)
     10: Checkerboard with offset      (covers: Checkerboard/Geometric)
     11: Sector/pie pattern            (covers: Sectors)
     12: Concentric wave (damped)      (covers: Waves_Concentric/Ripple)
     13: Spiral pattern                (covers: Waves_Spiral)
     14: Elliptical ring               (covers: Ring_Elliptical)
     15: Cortical layers               (covers: Cortical_Layers)
     16: Branching/radial              (covers: Branching)
     17: Voronoi mosaic                (covers: Mosaic_Irregular)
     18: Sigmoid transition            (covers: Gradient_Sigmoid/Bimodal)
     19: Multi-source interference     (covers: Waves_Interference)

    The `dissimilarity` parameter (1–10) controls post-processing distortion
    applied to all noise types.  Higher values = more different from GT patterns.

    Key design rules so correlation stays high:
      - dx/dy offsets are small (±5% of field) to avoid relocating features
      - For periodic patterns, freq_jitter is mild (±10%) and no phase flipping
      - For GT-delegating types (4,5,6), GT patterns already carry per-sample
        variation, so no extra spatial offset is added
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    cx_n, cy_n = x.mean(), y.mean()

    # ------------------------------------------------------------------
    # Structure RNG — noise_idx only, identical across all animals.
    # ------------------------------------------------------------------
    struct_seed = noise_idx * 7 + 42
    rng_struct  = np.random.RandomState(struct_seed)

    noise_type = noise_idx % N_NOISE_TYPES

    # Pre-draw all structural parameters (order fixed — drawn unconditionally)
    base_theta    = rng_struct.uniform(0, 2*np.pi)
    base_bx       = rng_struct.uniform(x.min()-200, x.max()+200)
    base_by       = rng_struct.uniform(y.min()-200, y.max()+200)
    base_sigma_b  = rng_struct.uniform(300, 700)
    base_freq     = rng_struct.uniform(80, 300)
    base_angle_d  = rng_struct.uniform(0, np.pi)
    base_target_r = rng_struct.uniform(0.15, 0.55)   # fraction of max_dist
    base_sigma_r  = rng_struct.uniform(0.04, 0.12)   # fraction of max_dist
    base_p1_idx   = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_p2_idx   = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_mix_w    = rng_struct.uniform(0.3, 0.7)
    base_inv_idx  = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_thr_idx  = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_threshold= rng_struct.uniform(0.35, 0.65)
    base_freq_x   = rng_struct.uniform(300, 800)
    base_freq_y   = rng_struct.uniform(300, 800)
    base_phase_x  = rng_struct.uniform(0, 2*np.pi)
    base_phase_y  = rng_struct.uniform(0, 2*np.pi)
    base_n_spots  = rng_struct.randint(8, 25)
    base_spot_xs  = rng_struct.uniform(x.min(), x.max(), 30)
    base_spot_ys  = rng_struct.uniform(y.min(), y.max(), 30)
    base_spot_sigs= rng_struct.uniform(100, 300, 30)
    base_noise_v  = rng_struct.rand(n)
    # Additional structural params for new noise types (10–19)
    base_checker_size = rng_struct.uniform(150, 500)
    base_n_sectors    = rng_struct.choice([3, 5, 6, 7, 9, 10, 11])
    base_conc_freq    = rng_struct.uniform(30, 80)
    base_spiral_rate  = rng_struct.uniform(100, 250)
    base_ellipse_a    = rng_struct.uniform(0.3, 0.6)
    base_ellipse_b    = rng_struct.uniform(0.15, 0.40)
    base_n_layers     = rng_struct.randint(3, 8)
    base_layer_vals   = rng_struct.rand(8)
    base_n_branches   = rng_struct.randint(3, 9)
    base_branch_width = rng_struct.uniform(0.15, 0.45)
    base_n_voronoi    = rng_struct.randint(10, 35)
    base_voronoi_xs   = rng_struct.uniform(x.min(), x.max(), 35)
    base_voronoi_ys   = rng_struct.uniform(y.min(), y.max(), 35)
    base_voronoi_vals = rng_struct.rand(35)
    base_sigmoid_shift= rng_struct.uniform(0.3, 0.7)
    base_sigmoid_steep= rng_struct.uniform(6, 18)
    base_intf_src1_x  = rng_struct.uniform(0.15, 0.40) * width
    base_intf_src1_y  = rng_struct.uniform(0.3, 0.7) * height
    base_intf_src2_x  = rng_struct.uniform(0.60, 0.85) * width
    base_intf_src2_y  = rng_struct.uniform(0.3, 0.7) * height
    base_intf_freq    = rng_struct.uniform(30, 80)

    # ------------------------------------------------------------------
    # Per-sample perturbation RNG — small offsets to keep correlation high
    # ------------------------------------------------------------------
    if sample_seed is not None:
        perturb_seed = (struct_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)

        # Small spatial offset (±5%) — features stay largely overlapping
        dx_offset    = rng_p.uniform(-0.05, 0.05) * width
        dy_offset    = rng_p.uniform(-0.05, 0.05) * height
        # Mild frequency jitter (±10%) — won't decorrelate periodic patterns
        freq_jitter  = rng_p.uniform(0.90, 1.10)
        # Size scale from config range
        size_scale   = rng_p.uniform(size_scale_range[0], size_scale_range[1])
        # Rotation jitter for gradient (≤~8°)
        rot_jitter   = rng_p.uniform(-0.15, 0.15)
        amp_scale    = rng_p.uniform(0.80, 1.05)
        bio_noise_std= rng_p.uniform(0.0, 0.08)
    else:
        dx_offset = dy_offset = 0.0
        freq_jitter  = 1.0
        size_scale   = 1.0
        rot_jitter   = 0.0
        amp_scale    = 1.0
        bio_noise_std= 0.0

    # ------------------------------------------------------------------
    # Pattern construction
    # ------------------------------------------------------------------
    if noise_type == 0:
        # Rotated gradient — slight rotation per sample
        theta = base_theta + rot_jitter
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val   = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)

    elif noise_type == 1:
        # Gaussian blob — small centre shift and size variation per sample
        bx    = base_bx + dx_offset
        by    = base_by + dy_offset
        sigma = base_sigma_b * size_scale
        dist  = np.sqrt((x - bx)**2 + (y - by)**2)
        val   = np.exp(-dist**2 / (2 * sigma**2))

    elif noise_type == 2:
        # Directional stripe — only direction rotates slightly per sample.
        # Use a wide-stripe frequency (×4) so small angle jitter doesn't
        # decorrelate the pattern across animals.
        angle_dir = base_angle_d + rot_jitter * 0.3
        proj      = x * np.cos(angle_dir) + y * np.sin(angle_dir)
        val       = (np.sin(proj / (base_freq * 4.0)) + 1) / 2

    elif noise_type == 3:
        # Ring — relative radius keeps ring visible; small centre shift per sample
        cx_r     = cx_n + dx_offset
        cy_r     = cy_n + dy_offset
        dist     = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        max_r    = dist.max() + 1e-8
        target_r = max_r * base_target_r * np.clip(size_scale, 0.9, 1.1)
        sigma    = max_r * base_sigma_r  * np.clip(size_scale, 0.9, 1.1)
        val      = np.exp(-((dist - target_r)**2) / (2 * sigma**2))

    elif noise_type == 4:
        # Mixture of two GT patterns — use structural (seed=None) base so
        # the pattern shape is identical across animals; per-sample variation
        # comes only from the small bio_noise added at the end.
        p1  = generate_pattern_values(coords, gt_patterns_list[base_p1_idx],
                                      width, height, sample_seed=None)
        p2  = generate_pattern_values(coords, gt_patterns_list[base_p2_idx],
                                      width, height, sample_seed=None)
        val = base_mix_w * p1 + (1 - base_mix_w) * p2

    elif noise_type == 5:
        # Inverted GT pattern — structural base for consistent shape across animals
        base_pattern = generate_pattern_values(coords, gt_patterns_list[base_inv_idx],
                                               width, height, sample_seed=None)
        val = 1 - base_pattern

    elif noise_type == 6:
        # Soft-thresholded GT pattern — sigmoid instead of hard step so
        # per-sample amplitude variation doesn't create all-0/all-1 outputs.
        base_pattern = generate_pattern_values(coords, gt_patterns_list[base_thr_idx],
                                               width, height, sample_seed=None)
        # Sigmoid centred on base_threshold; steepness 12 gives a clear edge
        val = 1.0 / (1.0 + np.exp(-12.0 * (base_pattern - base_threshold)))

    elif noise_type == 7:
        # 2-D sinusoidal field — only frequency varies per sample (no phase shift)
        val = (np.sin(x / (base_freq_x / freq_jitter) + base_phase_x) *
               np.sin(y / (base_freq_y / freq_jitter) + base_phase_y) + 1) / 2

    elif noise_type == 8:
        # Scattered spots — positions are structural (no per-sample offset so
        # spots stay aligned across animals); only sigma varies per sample.
        val = np.zeros(n)
        for k in range(base_n_spots):
            sx    = base_spot_xs[k]
            sy    = base_spot_ys[k]
            sigma = base_spot_sigs[k] * size_scale
            dist  = np.sqrt((x - sx)**2 + (y - sy)**2)
            val  += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)

    elif noise_type == 9:
        # Smooth correlated field: mostly structural values, slight x-gradient shift
        val = (0.6 * base_noise_v +
               0.2 * (x - x.min()) / (x.max() - x.min() + 1e-8) +
               0.2 * (y - y.min()) / (y.max() - y.min() + 1e-8))

    # === NEW NOISE TYPES (10–19) covering missing pattern families ===

    elif noise_type == 10:
        # Checkerboard with random cell size — covers Checkerboard/Geometric
        sq = base_checker_size * freq_jitter
        x_idx = ((x + dx_offset) // sq).astype(int)
        y_idx = ((y + dy_offset) // sq).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)

    elif noise_type == 11:
        # Sector/pie pattern with non-GT sector count — covers Sectors
        # Uses an odd sector count (3,5,6,7,9,10,11) distinct from GT's 4 and 8
        cx_r = cx_n + dx_offset
        cy_r = cy_n + dy_offset
        ang = np.arctan2(y - cy_r, x - cx_r)
        sector_width = 2 * np.pi / base_n_sectors
        sector_idx = ((ang + np.pi + rot_jitter) / sector_width).astype(int) % base_n_sectors
        val = (sector_idx % 2).astype(float)

    elif noise_type == 12:
        # Damped concentric waves — covers Waves_Concentric / Waves_Ripple
        cx_r = cx_n + dx_offset
        cy_r = cy_n + dy_offset
        dist = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        max_r = dist.max() + 1e-8
        dist_n = dist / max_r
        freq = base_conc_freq * freq_jitter
        val = np.exp(-dist_n * 1.5) * (np.sin(dist / freq) + 1) / 2

    elif noise_type == 13:
        # Spiral pattern — covers Waves_Spiral
        cx_r = cx_n + dx_offset
        cy_r = cy_n + dy_offset
        dist = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        ang = np.arctan2(y - cy_r, x - cx_r)
        spiral = ang + dist / (base_spiral_rate * freq_jitter) + rot_jitter
        val = (np.sin(spiral * 3) + 1) / 2

    elif noise_type == 14:
        # Elliptical ring — covers Ring_Elliptical
        cx_r = cx_n + dx_offset
        cy_r = cy_n + dy_offset
        dist = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        max_r = dist.max() + 1e-8
        a = max_r * base_ellipse_a * size_scale
        b = max_r * base_ellipse_b * size_scale
        ellipse_dist = np.sqrt(((x - cx_r) / (a + 1e-8))**2 + ((y - cy_r) / (b + 1e-8))**2)
        ring_w = 0.12 * size_scale
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * ring_w**2))

    elif noise_type == 15:
        # Cortical layers — covers Cortical_Layers
        cx_r = cx_n + dx_offset
        cy_r = cy_n + dy_offset
        dist = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        dist_n = dist / (dist.max() + 1e-8)
        scaled = np.clip(dist_n / size_scale, 0, 1)
        layer_idx = np.clip((scaled * base_n_layers).astype(int), 0, base_n_layers - 1)
        val = base_layer_vals[layer_idx]

    elif noise_type == 16:
        # Branching/radial pattern — covers Branching
        cx_r = cx_n + dx_offset
        cy_r = cy_n + dy_offset
        dist = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        max_r = dist.max() + 1e-8
        dist_n = dist / max_r
        ang = np.arctan2(y - cy_r, x - cx_r) + rot_jitter
        b_angle = 2 * np.pi / base_n_branches
        bw = base_branch_width * size_scale
        min_ang_dist = np.inf * np.ones(n)
        for i in range(base_n_branches):
            t_angle = i * b_angle
            ang_diff = np.abs(np.mod(ang - t_angle + np.pi, 2 * np.pi) - np.pi)
            min_ang_dist = np.minimum(min_ang_dist, ang_diff)
        val = np.exp(-min_ang_dist**2 / (2 * bw**2)) * (1 - np.exp(-dist_n * 3))

    elif noise_type == 17:
        # Voronoi mosaic — covers Mosaic_Irregular
        nv = base_n_voronoi
        vx = base_voronoi_xs[:nv] + dx_offset
        vy = base_voronoi_ys[:nv] + dy_offset
        vv = base_voronoi_vals[:nv]
        nearest_idx = np.zeros(n, dtype=int)
        min_dist_arr = np.inf * np.ones(n)
        for i in range(nv):
            dist = np.sqrt((x - vx[i])**2 + (y - vy[i])**2)
            closer = dist < min_dist_arr
            nearest_idx[closer] = i
            min_dist_arr[closer] = dist[closer]
        val = vv[nearest_idx]

    elif noise_type == 18:
        # Sigmoid transition along random axis — covers Gradient_Sigmoid / Bimodal
        theta = base_theta + rot_jitter
        proj = x * np.cos(theta) + y * np.sin(theta)
        proj_norm = (proj - proj.min()) / (proj.max() - proj.min() + 1e-8)
        val = 1 / (1 + np.exp(-base_sigmoid_steep * (proj_norm - base_sigmoid_shift)))

    elif noise_type == 19:
        # Multi-source interference — covers Waves_Interference
        s1x = base_intf_src1_x + dx_offset
        s1y = base_intf_src1_y + dy_offset
        s2x = base_intf_src2_x + dx_offset
        s2y = base_intf_src2_y + dy_offset
        d1 = np.sqrt((x - s1x)**2 + (y - s1y)**2)
        d2 = np.sqrt((x - s2x)**2 + (y - s2y)**2)
        freq = base_intf_freq * freq_jitter
        val = (np.sin(d1 / freq) + np.sin(d2 / freq) + 2) / 4

    else:
        # Fallback (should not reach here with N_NOISE_TYPES=20)
        val = base_noise_v

    # Amplitude scaling
    val = val * amp_scale

    # Per-sample biological noise
    if sample_seed is not None and bio_noise_std > 0:
        rng_bio = np.random.RandomState((struct_seed ^ sample_seed) + 2)
        val = val + rng_bio.normal(0, bio_noise_std, size=n)

    val = np.clip(val, 0, 1)

    # ------------------------------------------------------------------
    # Apply dissimilarity-based distortion (post-processing)
    # ------------------------------------------------------------------
    val = _apply_dissimilarity_distortion(val, coords, noise_idx, sample_seed,
                                          dissimilarity, width, height)

    return val

def get_visium_hex_grid(width, height, spacing=100):
    """Generates coordinates for a Visium-like hexagonal grid."""
    dy = spacing * np.sqrt(3) / 2
    
    coords = []
    row = 0
    y = 0
    
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        
        while x < width:
            coords.append([x, y])
            x += spacing
        
        y += dy
        row += 1
    
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# USER INPUT: NOISE DISSIMILARITY LEVEL
# =============================================================================
def get_dissimilarity_from_user():
    """
    Prompt the user for a noise dissimilarity level (1–10).
    Falls back to the configured NOISE_DISSIMILARITY default if input is
    invalid or empty.
    """
    default = NOISE_DISSIMILARITY
    try:
        raw = input(f"Enter noise dissimilarity level (1–10, default={default}): ").strip()
        if raw == "":
            return default
        val = int(raw)
        if val < 1 or val > 10:
            print(f"  Value out of range. Using default={default}.")
            return default
        return val
    except (ValueError, EOFError):
        print(f"  Invalid input. Using default={default}.")
        return default


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000

if __name__ == "__main__":
    dissimilarity = get_dissimilarity_from_user()

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Half-brain shapes, Visium-like RNA grid, per-sample variation")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {len(GT_PATTERNS)}")
    print(f"Noise Features: {NOISE_FEATURES}")
    print(f"Noise Types: {N_NOISE_TYPES} (covering all pattern families)")
    print(f"Noise Dissimilarity Level: {dissimilarity}/10")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} µm")
    print(f"Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}µm spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}µm spacing")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}–{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"

            # ------------------------------------------------------------------
            # Derive a unique integer seed per sample.
            # This seed drives both tissue shape AND per-sample pattern variation.
            # ------------------------------------------------------------------
            seed = hash(sample_id) % 2**32

            print(f"  Processing {sample_id} (seed={seed})...")
            
            # 1. Define Half-Brain Tissue Shape
            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)
            
            # 2. Generate MSI Data (Cartesian grid)
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]
            
            n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []
            
            # Generate GT pattern features — pass sample_seed for per-animal variation
            for i, pat in enumerate(GT_PATTERNS):
                msi_data[:, i] = generate_pattern_values(
                    msi_coords, pat,
                    width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed,
                    size_scale_range=PATTERN_SIZE_SCALE_RANGE
                )
                var_names_msi.append(f"MZ_{pat}")
            
            # Generate realistic noise features — also per-sample
            for i in range(NOISE_FEATURES):
                msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                    msi_coords, i, GT_PATTERNS,
                    width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed,
                    size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                    dissimilarity=dissimilarity
                )
                var_names_msi.append(f"MZ_Noise_{i}")
                
            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]
            
            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))
            
            # 3. Generate RNA Data (Visium-like hexagonal grid)
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]
            
            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []
            
            # Generate GT pattern features — same sample_seed so RNA and MSI match
            for i, pat in enumerate(GT_PATTERNS):
                rna_data[:, i] = generate_pattern_values(
                    rna_coords, pat,
                    width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed,
                    size_scale_range=PATTERN_SIZE_SCALE_RANGE
                )
                var_names_rna.append(f"Gene_{pat}")
                
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
            
            # Generate realistic noise features — per-sample
            for i in range(NOISE_FEATURES):
                rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                    rna_coords, i, GT_PATTERNS,
                    width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed,
                    size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                    dissimilarity=dissimilarity
                )
                var_names_rna.append(f"Gene_Noise_{i}")
            
            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]
            
            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print("\n50 GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")
    print(f"Noise dissimilarity level: {dissimilarity}/10")
    print(f"Noise types: {N_NOISE_TYPES} (scaled to cover all {len(GT_PATTERNS)} pattern families)")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, per-sample variation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Noise Types: 20 (covering all pattern families)
Noise Dissimilarity Level: 5/10
Field size: 6000x6000 µm
Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2
RNA grid: Visium-like hexagonal, 100µm spacing
MSI grid: Cartesian, 60µm spacing
Pattern size scale range: 0.90–1.10
Output: dummy_data_50
  Processing YC_1 (seed=321968443)...
  Processing YC_2 (seed=347226907)...
  Processing YC_3 (seed=3182348929)...
  Processing YC_4 (seed=4114315907)...
  Processing YAD_1 (seed=7651968)...
  Processing YAD_2 (seed=2917800819)...
  Processing YAD_3 (seed=191746656)...
  Processing YAD_4 (seed=891620259)...
  Processing AC_1 (seed=1504943285)...
  Processing AC_2 (seed=968954664)...
  Processing AC_3 (seed=3690293104)...
  Processing AC_4 (seed=566065078)...
  Processing AAD_1 (seed=728933825)...
  Processing 

In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# Per-sample pattern size variation range.
PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

# =============================================================================
# NOISE DISSIMILARITY CONTROL
# =============================================================================
# Controls how different noise features are from ground truth patterns.
# Range: 1 (minimal difference) to 10 (maximal difference).
# Default 5 provides moderate separation.
#
# All distortions use smooth continuous spatial functions so that RNA and MSI
# grids sampling the same underlying field stay perfectly correlated.
NOISE_DISSIMILARITY = 2  # <-- USER-ADJUSTABLE: integer 1-10

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",
    "Gradient_Y",
    "Gradient_Diagonal_NE",
    "Gradient_Diagonal_NW",
    "Gradient_Radial_In",
    "Gradient_Radial_Out",
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",
    "Stripes_Horizontal",
    "Stripes_Diagonal_45",
    "Stripes_Diagonal_135",
    "Waves_Concentric",
    "Waves_Spiral",
    "Waves_Interference",
    "Waves_Ripple",
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",
    "Blob_TopRight",
    "Blob_TopLeft",
    "Blob_BottomRight",
    "Blob_BottomLeft",
    "Spots_Grid_Dense",
    "Spots_Grid_Sparse",
    "Spots_Random_Large",
    "Spots_Triangular",
    "Spots_Hexagonal",
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",
    "Ring_Outer",
    "Ring_Double",
    "Ring_Eccentric",
    "Ring_Elliptical",
    "Ring_Partial",
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",
    "Checkerboard_Coarse",
    "Quadrant_Alternating",
    "Sectors_4",
    "Sectors_8",
    "Triangle_Pattern",
    "Diamond_Pattern",
    "Honeycomb",
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",
    "Hotspot_Cluster",
    "Edge_Enhancement",
    "Core_Shell",
    "Branching",
    "Laminar_Curved",
    "Mosaic_Irregular",
    "Gradient_Sigmoid",
    "Bimodal_Distribution",
    "Punctate_Dense",
    "Periventricular",
    "Asymmetric_Lobe",
]

NOISE_FEATURES = 500

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

# Number of noise types: 10 original + 10 new = 20, covering all pattern families
N_NOISE_TYPES = 50


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    """
    np.random.seed(sample_seed)

    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)

    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)

    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y

    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height

    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5

        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy

        if x_rot < left_edge:
            return False
        if y_rot < bottom_edge:
            return False
        if y_rot > top_edge:
            return False

        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)

        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))

        right_edge = left_edge + right_width

        if x_rot > right_edge:
            return False

        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)

            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)

            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))

            if y_norm > max_y_norm:
                return False

        return True

    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    """
    Generates intensity values (0-1) based on spatial coordinates.
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)

    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()

    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist

    angle = np.arctan2(y - cy, x - cx)

    pattern_seed = hash(pattern_type) % 2**31

    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height
        phase_offset = rng_s.uniform(-1.0, 1.0)
        freq_jitter = rng_s.uniform(0.75, 1.25)
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])
        amp_scale = rng_s.uniform(0.75, 1.05)
        bio_noise_std = rng_s.uniform(0.0, 0.10)
        cx_s = cx + dx_offset
        cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0
        freq_jitter = 1.0
        size_scale = 1.0
        amp_scale = 1.0
        bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center
        angle_s = angle

    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Y":
        val = y_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm + phase_offset * 0.1
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm + phase_offset * 0.1

    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
    elif pattern_type == "Waves_Interference":
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2

    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        sx = np.sin(x / base_freq + phase_offset)
        sy = np.sin(y / base_freq + phase_offset)
        raw = np.clip(sx * sy, 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.1 * size_scale
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset
            by = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter
        spot_sigma = 80 * size_scale
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter
        spot_sigma = 60 * size_scale
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                by = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * spot_sigma**2))
        val = np.clip(val, 0, 1)

    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.04 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7 * size_scale + phase_offset * 0.05)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
    elif pattern_type == "Ring_Double":
        r1 = max_dist * (0.25 * size_scale + phase_offset * 0.03)
        r2 = max_dist * (0.55 * size_scale + phase_offset * 0.03)
        ring_width = width * 0.03 * size_scale
        val = np.exp(-((dist_center_s - r1)**2) / (2 * ring_width**2)) + \
              np.exp(-((dist_center_s - r2)**2) / (2 * ring_width**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15 + dx_offset
        off_cy = cy + height * 0.1 + dy_offset
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * (0.35 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        val = np.exp(-((dist_off - target_r)**2) / (2 * ring_width**2))
    elif pattern_type == "Ring_Elliptical":
        a = max_dist * (0.5 * size_scale + phase_offset * 0.05)
        b = max_dist * (0.3 * size_scale + phase_offset * 0.03)
        ellipse_dist = np.sqrt(((x - cx_s)/a)**2 + ((y - cy_s)/b)**2)
        ring_width = 0.1 * size_scale
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * ring_width**2))
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * (0.4 * size_scale + phase_offset * 0.04)
        ring_width = width * 0.05 * size_scale
        ring_val = np.exp(-((dist_center_s - target_r)**2) / (2 * ring_width**2))
        arc_half = (np.pi/2) * size_scale
        a_start = -arc_half + phase_offset * 0.3
        a_end   =  arc_half + phase_offset * 0.3
        angle_mask = (angle_s > a_start) & (angle_s < a_end)
        val = ring_val * angle_mask

    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500 * freq_jitter
        x_idx = ((x + dx_offset) // square_size).astype(int)
        y_idx = ((y + dy_offset) // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx_s) & (y > cy_s)
        q3 = (x < cx_s) & (y < cy_s)
        val = (q1 | q3).astype(float)
    elif pattern_type == "Sectors_4":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
    elif pattern_type == "Sectors_8":
        sector = ((angle_s + np.pi + phase_offset) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
    elif pattern_type == "Triangle_Pattern":
        period = 300 * freq_jitter
        tri_x = np.abs(((x + dx_offset) % period) - period/2) / (period/2)
        tri_y = np.abs(((y + dy_offset) % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
    elif pattern_type == "Diamond_Pattern":
        period = 400 * freq_jitter
        val = (np.abs(((x + dx_offset) % period) - period/2) +
               np.abs(((y + dy_offset) % period) - period/2)) / period
    elif pattern_type == "Honeycomb":
        spacing = 200 * freq_jitter
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2) + dx_offset
                hy = j * spacing * 0.866 + dy_offset
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4 * size_scale), 0, 1))
        val = 1 - val

    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        shifted_dist = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        shifted_norm = shifted_dist / (shifted_dist.max() + 1e-8)
        scaled_norm = np.clip(shifted_norm / size_scale, 0, 1)
        layer_idx = np.clip((scaled_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed)
        sigma = width * 0.05 * size_scale
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1) + dx_offset
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1) + dy_offset
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Edge_Enhancement":
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        target = 0.8 * size_scale + phase_offset * 0.05
        target = np.clip(target, 0.3, 1.2)
        band_width = 0.15 * size_scale
        val = 1 - np.exp(-((shifted_norm - target)**2) / (2 * band_width**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Core_Shell":
        core_sigma = width * 0.1 * size_scale
        shell_r    = max_dist * (0.5 + phase_offset * 0.05) * size_scale
        shell_sigma = width * 0.08 * size_scale
        core  = np.exp(-dist_center_s**2 / (2 * core_sigma**2))
        shell = np.exp(-((dist_center_s - shell_r)**2) / (2 * shell_sigma**2))
        val = 0.7 * core + 0.5 * shell
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Branching":
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        branch_width = 0.3 * size_scale
        rotated_angle = angle_s + phase_offset * 0.5
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(rotated_angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
    elif pattern_type == "Laminar_Curved":
        curve_amp  = 200 * (1 + phase_offset * 0.3)
        curve_freq = 400 * freq_jitter
        band_thick = 300 * size_scale
        curved_y = (y + dy_offset) - curve_amp * np.sin((x + dx_offset) / curve_freq)
        layer_idx = (curved_y // band_thick).astype(int) % 2
        val = layer_idx.astype(float)
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed)
        n_seeds = max(5, int(20 / size_scale))
        seed_x = rng.uniform(x.min(), x.max(), n_seeds) + dx_offset
        seed_y = rng.uniform(y.min(), y.max(), n_seeds) + dy_offset
        seed_val = rng.rand(n_seeds)
        nearest_idx = np.zeros(n, dtype=int)
        min_dist_arr = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist_arr
            nearest_idx[closer] = i
            min_dist_arr[closer] = dist[closer]
        val = seed_val[nearest_idx]
    elif pattern_type == "Gradient_Sigmoid":
        shift = 0.5 + phase_offset * 0.1
        steepness = 10.0 / size_scale
        val = 1 / (1 + np.exp(-steepness * (x_norm - shift)))
    elif pattern_type == "Bimodal_Distribution":
        region1 = (x < cx_s).astype(float) * 0.8
        region2 = (x >= cx_s).astype(float) * 0.3
        val = region1 + region2
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed)
        punctum_sigma = 40 * size_scale
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max()) + dx_offset
            py = rng.uniform(y.min(), y.max()) + dy_offset
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * punctum_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Periventricular":
        void_r = 0.15 * size_scale + phase_offset * 0.02
        ring_r = 0.25 * size_scale + phase_offset * 0.02
        ring_w = 0.08 * size_scale
        shifted_norm = dist_center_s / (dist_center_s.max() + 1e-8)
        void_mask = shifted_norm < void_r
        ring_val  = np.exp(-((shifted_norm - ring_r)**2) / (2 * ring_w**2))
        val = ring_val * ~void_mask
    elif pattern_type == "Asymmetric_Lobe":
        lobe1_x = cx - width*0.2 + dx_offset
        lobe1_y = cy              + dy_offset
        lobe2_x = cx + width*0.15 + dx_offset
        lobe2_y = cy + height*0.1 + dy_offset
        sigma1 = width  * 0.15 * size_scale
        sigma2 = width  * 0.10 * size_scale
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * sigma1**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * sigma2**2))
        val = val / (val.max() + 1e-8)
    else:
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)

    val = val * amp_scale

    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def _apply_dissimilarity_distortion(val, coords, noise_idx, sample_seed,
                                    dissimilarity, width, height):
    """
    Post-processing distortion scaled by NOISE_DISSIMILARITY (1-10).

    CRITICAL: All operations use smooth continuous spatial functions evaluated
    at (x, y) coordinates.  The RNG draws only scalar parameters (frequencies,
    phases, exponents) — never n-dependent arrays.  This guarantees identical
    distortion surfaces on RNA and MSI grids, preserving cross-modal correlation.

    Mechanisms (scaled by d = (dissimilarity-1)/9, range 0-1):
      1. Multiplicative spatial modulation — very low-frequency sinusoidal
         field (wavelength 1500-3000um, well above both grid spacings).
      2. Power-law warping — spatially uniform exponent (trivially grid-safe).
      3. Smooth additive field — low-frequency sinusoid at different angle.
    """
    d = (dissimilarity - 1) / 9.0

    if d < 0.01:
        return val

    x, y = coords[:, 0], coords[:, 1]

    # RNG for scalar params only — no n-dependent draws
    rng_d = np.random.RandomState((noise_idx * 31 + 997) % 2**31)

    # 1. Multiplicative spatial modulation (very low frequency)
    mod_freq_x = rng_d.uniform(1500, 3000)
    mod_freq_y = rng_d.uniform(1500, 3000)
    mod_phase  = rng_d.uniform(0, 2 * np.pi)
    modulation = (np.sin(x / mod_freq_x + mod_phase) *
                  np.cos(y / mod_freq_y + mod_phase * 0.7) + 1) / 2
    val = val * (1 - d * 0.4) + val * modulation * (d * 0.4)

    # 2. Power-law warping (spatially uniform)
    exponent = 1.0 + d * 1.5 * rng_d.choice([-1, 1])
    exponent = np.clip(exponent, 0.3, 2.5)
    val = np.clip(val, 0, 1) ** exponent

    # 3. Smooth additive field (low frequency)
    add_freq  = rng_d.uniform(1200, 2500)
    add_theta = rng_d.uniform(0, 2 * np.pi)
    add_phase = rng_d.uniform(0, 2 * np.pi)
    proj = x * np.cos(add_theta) + y * np.sin(add_theta)
    structured = (np.sin(proj / add_freq + add_phase) + 1) / 2
    val = val + structured * d * 0.2

    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000,
                             sample_seed=None, size_scale_range=(0.70, 1.40),
                             dissimilarity=5):
    """
    Generate noise features that are correlated across animals for the same
    noise_idx, with modest per-sample perturbations.

    CRITICAL INVARIANT: Every noise type produces a smooth continuous spatial
    field f(x, y).  This guarantees that RNA hexagonal and MSI Cartesian grids
    sampling the same underlying field stay correlated — the gene and its
    corresponding m/z feature are always the same pattern.

    Forbidden operations (would break RNA<->MSI correspondence):
      - rng.rand(n) where n = len(coords)
      - floor/modulo discretisation at scales near grid spacing
      - Nearest-neighbor assignment with few seeds

    Uses N_NOISE_TYPES=20 covering all GT pattern families:
       0: Rotated gradient              (Gradients)
       1: Gaussian blob                 (Blobs)
       2: Directional stripe            (Stripes)
       3: Ring                          (Rings)
       4: Mixture of two GT patterns    (Complex combinations)
       5: Inverted GT pattern           (Inversions)
       6: Soft-thresholded GT pattern   (Threshold-based)
       7: 2-D sinusoidal field          (Waves periodic)
       8: Scattered Gaussian spots      (Spots)
       9: Smooth multi-frequency field  (Smooth noise)  [FIXED]
      10: Smooth checkerboard analogue  (Checkerboard/Geometric)  [NEW]
      11: Smooth sector pattern         (Sectors)  [NEW]
      12: Damped concentric waves       (Waves_Concentric/Ripple)  [NEW]
      13: Spiral pattern                (Waves_Spiral)  [NEW]
      14: Elliptical ring               (Ring_Elliptical)  [NEW]
      15: Smooth concentric layers      (Cortical_Layers)  [NEW]
      16: Smooth branching/radial       (Branching)  [NEW]
      17: Multi-blob mosaic             (Mosaic_Irregular)  [NEW]
      18: Sigmoid along random axis     (Gradient_Sigmoid/Bimodal)  [NEW]
      19: Multi-source interference     (Waves_Interference)  [NEW]
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    cx_n, cy_n = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx_n)**2 + (y - cy_n)**2)
    max_dist = dist_center.max() + 1e-8

    # ------------------------------------------------------------------
    # Structure RNG — noise_idx only, identical across all animals.
    # ALL draws are scalars or fixed-size arrays (never n-dependent).
    # ------------------------------------------------------------------
    struct_seed = noise_idx * 7 + 42
    rng_struct  = np.random.RandomState(struct_seed)

    noise_type = noise_idx % N_NOISE_TYPES

    # Pre-draw ALL structural parameters unconditionally (fixed draw order)
    base_theta     = rng_struct.uniform(0, 2*np.pi)
    base_bx_frac   = rng_struct.uniform(0.1, 0.9)
    base_by_frac   = rng_struct.uniform(0.1, 0.9)
    base_sigma_b   = rng_struct.uniform(300, 700)
    base_freq      = rng_struct.uniform(80, 300)
    base_angle_d   = rng_struct.uniform(0, np.pi)
    base_target_r  = rng_struct.uniform(0.15, 0.55)
    base_sigma_r   = rng_struct.uniform(0.04, 0.12)
    base_p1_idx    = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_p2_idx    = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_mix_w     = rng_struct.uniform(0.3, 0.7)
    base_inv_idx   = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_thr_idx   = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_threshold = rng_struct.uniform(0.35, 0.65)
    base_freq_x    = rng_struct.uniform(300, 800)
    base_freq_y    = rng_struct.uniform(300, 800)
    base_phase_x   = rng_struct.uniform(0, 2*np.pi)
    base_phase_y   = rng_struct.uniform(0, 2*np.pi)
    base_n_spots   = rng_struct.randint(8, 25)
    base_spot_xs   = rng_struct.uniform(0.05, 0.95, 30) * width   # fixed-30
    base_spot_ys   = rng_struct.uniform(0.05, 0.95, 30) * height
    base_spot_sigs = rng_struct.uniform(100, 300, 30)
    # Type 9: smooth multi-frequency field params (fixed-size)
    base_sf_freqs  = rng_struct.uniform(200, 800, (5, 2))
    base_sf_phases = rng_struct.uniform(0, 2*np.pi, 5)
    base_sf_amps   = rng_struct.uniform(0.1, 0.4, 5)
    # Types 10-19 structural params (all scalar or fixed-size)
    base_checker_freq   = rng_struct.uniform(150, 500)
    base_n_sectors      = rng_struct.choice([3, 5, 6, 7, 9, 10, 11])
    base_conc_freq      = rng_struct.uniform(30, 80)
    base_spiral_rate    = rng_struct.uniform(100, 250)
    base_ellipse_a_frac = rng_struct.uniform(0.3, 0.6)
    base_ellipse_b_frac = rng_struct.uniform(0.15, 0.40)
    base_n_layers       = rng_struct.randint(3, 8)
    base_layer_vals     = rng_struct.rand(8)
    base_n_branches     = rng_struct.randint(3, 9)
    base_branch_width   = rng_struct.uniform(0.15, 0.45)
    base_n_mosaic       = rng_struct.randint(15, 40)
    base_mosaic_xs      = rng_struct.uniform(0.05, 0.95, 40) * width
    base_mosaic_ys      = rng_struct.uniform(0.05, 0.95, 40) * height
    base_mosaic_sigs    = rng_struct.uniform(200, 600, 40)
    base_mosaic_vals    = rng_struct.rand(40)
    base_sigmoid_shift  = rng_struct.uniform(0.3, 0.7)
    base_sigmoid_steep  = rng_struct.uniform(6, 18)
    base_intf_src1_x    = rng_struct.uniform(0.15, 0.40) * width
    base_intf_src1_y    = rng_struct.uniform(0.3, 0.7) * height
    base_intf_src2_x    = rng_struct.uniform(0.60, 0.85) * width
    base_intf_src2_y    = rng_struct.uniform(0.3, 0.7) * height
    base_intf_freq      = rng_struct.uniform(30, 80)

    # ------------------------------------------------------------------
    # Per-sample perturbation RNG
    # ------------------------------------------------------------------
    if sample_seed is not None:
        perturb_seed = (struct_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)
        dx_offset     = rng_p.uniform(-0.05, 0.05) * width
        dy_offset     = rng_p.uniform(-0.05, 0.05) * height
        freq_jitter   = rng_p.uniform(0.90, 1.10)
        size_scale    = rng_p.uniform(size_scale_range[0], size_scale_range[1])
        rot_jitter    = rng_p.uniform(-0.15, 0.15)
        amp_scale     = rng_p.uniform(0.80, 1.05)
        bio_noise_std = rng_p.uniform(0.0, 0.08)
    else:
        dx_offset = dy_offset = 0.0
        freq_jitter   = 1.0
        size_scale    = 1.0
        rot_jitter    = 0.0
        amp_scale     = 1.0
        bio_noise_std = 0.0

    # Shifted centre
    cx_s = cx_n + dx_offset
    cy_s = cy_n + dy_offset
    dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
    angle_s = np.arctan2(y - cy_s, x - cx_s)

    # ------------------------------------------------------------------
    # Pattern construction — ALL types use continuous f(x,y) only
    # ------------------------------------------------------------------
    if noise_type == 0:
        # Rotated gradient
        theta = base_theta + rot_jitter
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)

    elif noise_type == 1:
        # Gaussian blob — position relative to coordinate extent so the blob
        # always lands within the tissue regardless of grid type
        bx = x.min() + base_bx_frac * (x.max() - x.min()) + dx_offset
        by = y.min() + base_by_frac * (y.max() - y.min()) + dy_offset
        sigma = base_sigma_b * size_scale
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))

    elif noise_type == 2:
        # Directional stripe (wide frequency)
        angle_dir = base_angle_d + rot_jitter * 0.3
        proj = x * np.cos(angle_dir) + y * np.sin(angle_dir)
        val = (np.sin(proj / (base_freq * 4.0)) + 1) / 2

    elif noise_type == 3:
        # Ring
        cx_r = cx_n + dx_offset
        cy_r = cy_n + dy_offset
        dist = np.sqrt((x - cx_r)**2 + (y - cy_r)**2)
        max_r = dist.max() + 1e-8
        target_r = max_r * base_target_r * np.clip(size_scale, 0.9, 1.1)
        sigma = max_r * base_sigma_r * np.clip(size_scale, 0.9, 1.1)
        val = np.exp(-((dist - target_r)**2) / (2 * sigma**2))

    elif noise_type == 4:
        # Mixture of two GT patterns (structural base, seed=None)
        p1 = generate_pattern_values(coords, gt_patterns_list[base_p1_idx],
                                     width, height, sample_seed=None)
        p2 = generate_pattern_values(coords, gt_patterns_list[base_p2_idx],
                                     width, height, sample_seed=None)
        val = base_mix_w * p1 + (1 - base_mix_w) * p2

    elif noise_type == 5:
        # Inverted GT pattern
        base_pattern = generate_pattern_values(coords, gt_patterns_list[base_inv_idx],
                                               width, height, sample_seed=None)
        val = 1 - base_pattern

    elif noise_type == 6:
        # Soft-thresholded GT pattern
        base_pattern = generate_pattern_values(coords, gt_patterns_list[base_thr_idx],
                                               width, height, sample_seed=None)
        val = 1.0 / (1.0 + np.exp(-12.0 * (base_pattern - base_threshold)))

    elif noise_type == 7:
        # 2-D sinusoidal field
        val = (np.sin(x / (base_freq_x / freq_jitter) + base_phase_x) *
               np.sin(y / (base_freq_y / freq_jitter) + base_phase_y) + 1) / 2

    elif noise_type == 8:
        # Scattered Gaussian spots (fixed positions, continuous Gaussians)
        val = np.zeros(n)
        for k in range(base_n_spots):
            sx = base_spot_xs[k]
            sy = base_spot_ys[k]
            sigma = base_spot_sigs[k] * size_scale
            dist = np.sqrt((x - sx)**2 + (y - sy)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)

    elif noise_type == 9:
        # Smooth multi-frequency field [FIXED: was rng.rand(n)]
        # Sum of 5 sinusoidal components — purely continuous f(x,y)
        val = np.zeros(n)
        for k in range(5):
            fx, fy = base_sf_freqs[k]
            ph = base_sf_phases[k]
            amp = base_sf_amps[k]
            val += amp * (np.sin(x / (fx * freq_jitter) + ph) *
                          np.cos(y / (fy * freq_jitter) + ph * 1.3) + 1) / 2
        val += 0.15 * (x - x.min()) / (x.max() - x.min() + 1e-8)
        val += 0.10 * (y - y.min()) / (y.max() - y.min() + 1e-8)
        val = (val - val.min()) / (val.max() - val.min() + 1e-8)

    # === NEW NOISE TYPES (10-19) — all continuous f(x,y) ===

    elif noise_type == 10:
        # Smooth checkerboard analogue — product of two sinusoids
        freq = base_checker_freq * freq_jitter
        val = (np.sin(x / freq + dx_offset / freq) *
               np.sin(y / freq + dy_offset / freq) + 1) / 2

    elif noise_type == 11:
        # Smooth sector pattern — cos(N * angle) for N-fold symmetry
        val = (np.cos(base_n_sectors * (angle_s + rot_jitter)) + 1) / 2

    elif noise_type == 12:
        # Damped concentric waves
        freq = base_conc_freq * freq_jitter
        dist_n = dist_center_s / (dist_center_s.max() + 1e-8)
        val = np.exp(-dist_n * 1.5) * (np.sin(dist_center_s / freq) + 1) / 2

    elif noise_type == 13:
        # Spiral pattern
        spiral = angle_s + dist_center_s / (base_spiral_rate * freq_jitter) + rot_jitter
        val = (np.sin(spiral * 3) + 1) / 2

    elif noise_type == 14:
        # Elliptical ring
        a = max_dist * base_ellipse_a_frac * size_scale
        b = max_dist * base_ellipse_b_frac * size_scale
        ellipse_dist = np.sqrt(((x - cx_s) / (a + 1e-8))**2 +
                               ((y - cy_s) / (b + 1e-8))**2)
        ring_w = 0.12 * size_scale
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * ring_w**2))

    elif noise_type == 15:
        # Smooth concentric layers — Gaussian bumps at fixed radii
        dist_n = dist_center_s / (dist_center_s.max() + 1e-8)
        scaled = np.clip(dist_n / size_scale, 0, 1)
        val = np.zeros(n)
        for k in range(base_n_layers):
            target = (k + 0.5) / base_n_layers
            sigma_l = 0.5 / base_n_layers
            val += base_layer_vals[k] * np.exp(-((scaled - target)**2) / (2 * sigma_l**2))
        val = val / (val.max() + 1e-8)

    elif noise_type == 16:
        # Smooth branching/radial — exp(cos(N*angle))
        bw = base_branch_width * size_scale
        dist_n = dist_center_s / (dist_center_s.max() + 1e-8)
        rotated_angle = angle_s + rot_jitter
        arm_val = np.exp(np.cos(base_n_branches * rotated_angle) / (bw + 0.01))
        arm_val = (arm_val - arm_val.min()) / (arm_val.max() - arm_val.min() + 1e-8)
        val = arm_val * (1 - np.exp(-dist_n * 3))

    elif noise_type == 17:
        # Multi-blob mosaic — softmax-weighted Gaussians (smooth, no Voronoi)
        nb = base_n_mosaic
        val = np.zeros(n)
        weight_sum = np.zeros(n) + 1e-8
        for k in range(nb):
            bx = base_mosaic_xs[k] + dx_offset
            by = base_mosaic_ys[k] + dy_offset
            sig = base_mosaic_sigs[k] * size_scale
            w = np.exp(-((x - bx)**2 + (y - by)**2) / (2 * sig**2))
            val += w * base_mosaic_vals[k]
            weight_sum += w
        val = val / weight_sum

    elif noise_type == 18:
        # Sigmoid transition along random axis
        theta = base_theta + rot_jitter
        proj = x * np.cos(theta) + y * np.sin(theta)
        proj_norm = (proj - proj.min()) / (proj.max() - proj.min() + 1e-8)
        val = 1 / (1 + np.exp(-base_sigmoid_steep * (proj_norm - base_sigmoid_shift)))

    elif noise_type == 19:
        # Multi-source interference
        s1x = base_intf_src1_x + dx_offset
        s1y = base_intf_src1_y + dy_offset
        s2x = base_intf_src2_x + dx_offset
        s2y = base_intf_src2_y + dy_offset
        d1 = np.sqrt((x - s1x)**2 + (y - s1y)**2)
        d2 = np.sqrt((x - s2x)**2 + (y - s2y)**2)
        freq = base_intf_freq * freq_jitter
        val = (np.sin(d1 / freq) + np.sin(d2 / freq) + 2) / 4

    else:
        val = np.zeros(n) + 0.5

    # Amplitude scaling
    val = val * amp_scale

    # Per-sample biological noise
    if sample_seed is not None and bio_noise_std > 0:
        rng_bio = np.random.RandomState((struct_seed ^ sample_seed) + 2)
        val = val + rng_bio.normal(0, bio_noise_std, size=n)

    val = np.clip(val, 0, 1)

    # Apply dissimilarity-based distortion (smooth post-processing)
    val = _apply_dissimilarity_distortion(val, coords, noise_idx, sample_seed,
                                          dissimilarity, width, height)

    return val


def get_visium_hex_grid(width, height, spacing=100):
    """Generates coordinates for a Visium-like hexagonal grid."""
    dy = spacing * np.sqrt(3) / 2
    coords = []
    row = 0
    y = 0
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        while x < width:
            coords.append([x, y])
            x += spacing
        y += dy
        row += 1
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# USER INPUT
# =============================================================================
def get_dissimilarity_from_user():
    """Prompt for noise dissimilarity level (1-10)."""
    default = NOISE_DISSIMILARITY
    try:
        raw = input(f"Enter noise dissimilarity level (1-10, default={default}): ").strip()
        if raw == "":
            return default
        val = int(raw)
        if val < 1 or val > 10:
            print(f"  Value out of range. Using default={default}.")
            return default
        return val
    except (ValueError, EOFError):
        print(f"  Invalid input. Using default={default}.")
        return default


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000

if __name__ == "__main__":
    dissimilarity = get_dissimilarity_from_user()

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Half-brain shapes, Visium-like RNA grid, per-sample variation")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {len(GT_PATTERNS)}")
    print(f"Noise Features: {NOISE_FEATURES}")
    print(f"Noise Types: {N_NOISE_TYPES} (covering all pattern families)")
    print(f"Noise Dissimilarity Level: {dissimilarity}/10")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} um")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}um spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}um spacing")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}-{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"
            seed = hash(sample_id) % 2**32
            print(f"  Processing {sample_id} (seed={seed})...")

            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)

            # MSI Data
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]

            n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []

            for i, pat in enumerate(GT_PATTERNS):
                msi_data[:, i] = generate_pattern_values(
                    msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_msi.append(f"MZ_{pat}")

            for i in range(NOISE_FEATURES):
                msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                    msi_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                    dissimilarity=dissimilarity)
                var_names_msi.append(f"MZ_Noise_{i}")

            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]

            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))

            # RNA Data
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]

            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []

            for i, pat in enumerate(GT_PATTERNS):
                rna_data[:, i] = generate_pattern_values(
                    rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_rna.append(f"Gene_{pat}")
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))

            for i in range(NOISE_FEATURES):
                rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                    rna_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                    dissimilarity=dissimilarity)
                var_names_rna.append(f"Gene_Noise_{i}")

            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]

            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print("\n50 GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")
    print(f"Noise dissimilarity level: {dissimilarity}/10")
    print(f"Noise types: {N_NOISE_TYPES} (covering all {len(GT_PATTERNS)} pattern families)")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, per-sample variation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Noise Types: 50 (covering all pattern families)
Noise Dissimilarity Level: 2/10
Field size: 6000x6000 um
RNA grid: Visium-like hexagonal, 100um spacing
MSI grid: Cartesian, 60um spacing
Pattern size scale range: 0.90-1.10
Output: dummy_data_50
  Processing YC_1 (seed=994770162)...
  Processing YC_2 (seed=1682408368)...
  Processing YC_3 (seed=2152946093)...
  Processing YC_4 (seed=2799160652)...
  Processing YAD_1 (seed=629658747)...
  Processing YAD_2 (seed=2750943403)...
  Processing YAD_3 (seed=3651348702)...
  Processing YAD_4 (seed=2044655894)...
  Processing AC_1 (seed=3946498238)...
  Processing AC_2 (seed=1278239180)...
  Processing AC_3 (seed=1963447578)...
  Processing AC_4 (seed=2690307881)...
  Processing AAD_1 (seed=1260648889)...
  Processing AAD_2 (seed=2488950731)...
  Processing AAD_3 (s

In [5]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100
MSI_PIXEL_SIZE = 60

PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

NOISE_DISSIMILARITY = 2

GT_PATTERNS = [
    "Gradient_X", "Gradient_Y", "Gradient_Diagonal_NE", "Gradient_Diagonal_NW",
    "Gradient_Radial_In", "Gradient_Radial_Out",
    "Stripes_Vertical", "Stripes_Horizontal", "Stripes_Diagonal_45",
    "Stripes_Diagonal_135", "Waves_Concentric", "Waves_Spiral",
    "Waves_Interference", "Waves_Ripple",
    "Blob_Center", "Blob_TopRight", "Blob_TopLeft", "Blob_BottomRight",
    "Blob_BottomLeft", "Spots_Grid_Dense", "Spots_Grid_Sparse",
    "Spots_Random_Large", "Spots_Triangular", "Spots_Hexagonal",
    "Ring_Inner", "Ring_Outer", "Ring_Double", "Ring_Eccentric",
    "Ring_Elliptical", "Ring_Partial",
    "Checkerboard_Fine", "Checkerboard_Coarse", "Quadrant_Alternating",
    "Sectors_4", "Sectors_8", "Triangle_Pattern", "Diamond_Pattern", "Honeycomb",
    "Cortical_Layers", "Hotspot_Cluster", "Edge_Enhancement", "Core_Shell",
    "Branching", "Laminar_Curved", "Mosaic_Irregular", "Gradient_Sigmoid",
    "Bimodal_Distribution", "Punctate_Dense", "Periventricular", "Asymmetric_Lobe",
]

NOISE_FEATURES = 500

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

N_NOISE_TYPES = len(GT_PATTERNS)


def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    np.random.seed(sample_seed)
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height

    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        if x_rot < left_edge: return False
        if y_rot < bottom_edge: return False
        if y_rot > top_edge: return False
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        right_edge = left_edge + right_width
        if x_rot > right_edge: return False
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            if y_norm > max_y_norm: return False
        return True
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    angle = np.arctan2(y - cy, x - cx)
    pattern_seed = hash(pattern_type) % 2**31

    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height
        phase_offset = rng_s.uniform(-1.0, 1.0)
        freq_jitter = rng_s.uniform(0.75, 1.25)
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])
        amp_scale = rng_s.uniform(0.75, 1.05)
        bio_noise_std = rng_s.uniform(0.0, 0.10)
        cx_s = cx + dx_offset
        cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0; freq_jitter = 1.0; size_scale = 1.0
        amp_scale = 1.0; bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center; angle_s = angle

    if pattern_type == "Gradient_X": val = x_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Y": val = y_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Diagonal_NE": val = (x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Diagonal_NW": val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Radial_In": val = 1 - dist_norm + phase_offset * 0.1
    elif pattern_type == "Gradient_Radial_Out": val = dist_norm + phase_offset * 0.1
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
    elif pattern_type == "Waves_Interference":
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset; by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed); sigma = width * 0.1 * size_scale; val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset; by = rng.uniform(y.min(), y.max()) + dy_offset
            val += np.exp(-((x-bx)**2+(y-by)**2) / (2*sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter; spot_sigma = 80 * size_scale; val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter; spot_sigma = 60 * size_scale; val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2*size_scale+phase_offset*0.05); ring_width = width*0.04*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7*size_scale+phase_offset*0.05); ring_width = width*0.05*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Double":
        r1 = max_dist*(0.25*size_scale+phase_offset*0.03); r2 = max_dist*(0.55*size_scale+phase_offset*0.03)
        rw = width*0.03*size_scale
        val = np.exp(-((dist_center_s-r1)**2)/(2*rw**2))+np.exp(-((dist_center_s-r2)**2)/(2*rw**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx+width*0.15+dx_offset; off_cy = cy+height*0.1+dy_offset
        dist_off = np.sqrt((x-off_cx)**2+(y-off_cy)**2)
        target_r = max_dist*(0.35*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        val = np.exp(-((dist_off-target_r)**2)/(2*rw**2))
    elif pattern_type == "Ring_Elliptical":
        a = max_dist*(0.5*size_scale+phase_offset*0.05); b = max_dist*(0.3*size_scale+phase_offset*0.03)
        ed = np.sqrt(((x-cx_s)/a)**2+((y-cy_s)/b)**2); rw = 0.1*size_scale
        val = np.exp(-((ed-1)**2)/(2*rw**2))
    elif pattern_type == "Ring_Partial":
        target_r = max_dist*(0.4*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        rv = np.exp(-((dist_center_s-target_r)**2)/(2*rw**2))
        ah = (np.pi/2)*size_scale; a_s = -ah+phase_offset*0.3; a_e = ah+phase_offset*0.3
        val = rv * ((angle_s > a_s) & (angle_s < a_e))
    elif pattern_type == "Checkerboard_Fine":
        ss = 200*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Checkerboard_Coarse":
        ss = 500*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Quadrant_Alternating":
        val = ((x>cx_s)&(y>cy_s) | (x<cx_s)&(y<cy_s)).astype(float)
    elif pattern_type == "Sectors_4":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/2)).astype(int)%4; val = (sector%2).astype(float)
    elif pattern_type == "Sectors_8":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/4)).astype(int)%8; val = (sector%2).astype(float)
    elif pattern_type == "Triangle_Pattern":
        period = 300*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)/(period/2)+np.abs(((y+dy_offset)%period)-period/2)/(period/2))/2
    elif pattern_type == "Diamond_Pattern":
        period = 400*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)+np.abs(((y+dy_offset)%period)-period/2))/period
    elif pattern_type == "Honeycomb":
        spacing = 200*freq_jitter; val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i*spacing+(j%2)*(spacing/2)+dx_offset; hy = j*spacing*0.866+dy_offset
                dist = np.sqrt((x-hx)**2+(y-hy)**2)
                val = np.maximum(val, 1-np.clip(dist/(spacing*0.4*size_scale), 0, 1))
        val = 1-val
    elif pattern_type == "Cortical_Layers":
        n_layers = 5; layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        sd = np.sqrt((x-cx_s)**2+(y-cy_s)**2); sn = sd/(sd.max()+1e-8)
        sc = np.clip(sn/size_scale, 0, 1); li = np.clip((sc*n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in li])
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed); sigma = width*0.05*size_scale; val = np.zeros(n)
        ccx = cx+rng.uniform(-width*0.1, width*0.1)+dx_offset; ccy = cy+rng.uniform(-height*0.1, height*0.1)+dy_offset
        for _ in range(8):
            bx = ccx+rng.normal(0, width*0.08); by = ccy+rng.normal(0, height*0.08)
            val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sigma**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Edge_Enhancement":
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        target = np.clip(0.8*size_scale+phase_offset*0.05, 0.3, 1.2); bw = 0.15*size_scale
        val = np.clip(1-np.exp(-((sn-target)**2)/(2*bw**2)), 0, 1)
    elif pattern_type == "Core_Shell":
        cs = width*0.1*size_scale; sr = max_dist*(0.5+phase_offset*0.05)*size_scale; ss = width*0.08*size_scale
        val = 0.7*np.exp(-dist_center_s**2/(2*cs**2))+0.5*np.exp(-((dist_center_s-sr)**2)/(2*ss**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Branching":
        nb = 5; ba = 2*np.pi/nb; bw = 0.3*size_scale; ra = angle_s+phase_offset*0.5
        mdb = np.inf*np.ones(n)
        for i in range(nb):
            ta = i*ba; ad2 = np.abs(np.mod(ra-ta+np.pi, 2*np.pi)-np.pi)
            mdb = np.minimum(mdb, ad2)
        val = np.exp(-mdb**2/(2*bw**2))*(1-np.exp(-dist_norm*3))
    elif pattern_type == "Laminar_Curved":
        ca = 200*(1+phase_offset*0.3); cf = 400*freq_jitter; bt = 300*size_scale
        cy2 = (y+dy_offset)-ca*np.sin((x+dx_offset)/cf)
        val = ((cy2//bt).astype(int)%2).astype(float)
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed); ns = max(5, int(20/size_scale))
        sx = rng.uniform(x.min(), x.max(), ns)+dx_offset; sy = rng.uniform(y.min(), y.max(), ns)+dy_offset
        sv = rng.rand(ns); ni = np.zeros(n, dtype=int); md = np.inf*np.ones(n)
        for i in range(ns):
            d = np.sqrt((x-sx[i])**2+(y-sy[i])**2); c = d<md; ni[c] = i; md[c] = d[c]
        val = sv[ni]
    elif pattern_type == "Gradient_Sigmoid":
        shift = 0.5+phase_offset*0.1; steep = 10.0/size_scale
        val = 1/(1+np.exp(-steep*(x_norm-shift)))
    elif pattern_type == "Bimodal_Distribution":
        val = (x<cx_s).astype(float)*0.8+(x>=cx_s).astype(float)*0.3
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed); ps = 40*size_scale; val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())+dx_offset; py = rng.uniform(y.min(), y.max())+dy_offset
            val += np.exp(-((x-px)**2+(y-py)**2)/(2*ps**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Periventricular":
        vr = 0.15*size_scale+phase_offset*0.02; rr = 0.25*size_scale+phase_offset*0.02; rw = 0.08*size_scale
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        val = np.exp(-((sn-rr)**2)/(2*rw**2))*(sn>=vr)
    elif pattern_type == "Asymmetric_Lobe":
        l1x = cx-width*0.2+dx_offset; l1y = cy+dy_offset; l2x = cx+width*0.15+dx_offset; l2y = cy+height*0.1+dy_offset
        s1 = width*0.15*size_scale; s2 = width*0.10*size_scale
        val = 0.7*np.exp(-((x-l1x)**2+(y-l1y)**2)/(2*s1**2))+0.5*np.exp(-((x-l2x)**2+(y-l2y)**2)/(2*s2**2))
        val = val/(val.max()+1e-8)
    else:
        rng = np.random.RandomState(pattern_seed); val = rng.rand(n)

    val = val * amp_scale
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)
    return np.clip(val, 0, 1)


def _apply_dissimilarity_distortion(val, coords, noise_idx, sample_seed,
                                    dissimilarity, width, height):
    d = (dissimilarity - 1) / 9.0
    if d < 0.01: return val
    x, y = coords[:, 0], coords[:, 1]
    rng_d = np.random.RandomState((noise_idx * 31 + 997) % 2**31)
    mod_freq_x = rng_d.uniform(1500, 3000); mod_freq_y = rng_d.uniform(1500, 3000)
    mod_phase = rng_d.uniform(0, 2*np.pi)
    modulation = (np.sin(x/mod_freq_x+mod_phase)*np.cos(y/mod_freq_y+mod_phase*0.7)+1)/2
    val = val*(1-d*0.4)+val*modulation*(d*0.4)
    exponent = 1.0+d*1.5*rng_d.choice([-1,1]); exponent = np.clip(exponent, 0.3, 2.5)
    val = np.clip(val, 0, 1)**exponent
    add_freq = rng_d.uniform(1200, 2500); add_theta = rng_d.uniform(0, 2*np.pi)
    add_phase = rng_d.uniform(0, 2*np.pi)
    proj = x*np.cos(add_theta)+y*np.sin(add_theta)
    val = val+(np.sin(proj/add_freq+add_phase)+1)/2*d*0.2
    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000,
                             sample_seed=None, size_scale_range=(0.70, 1.40),
                             dissimilarity=5):
    """
    Generate noise features with N_NOISE_TYPES = len(GT_PATTERNS) = 50 distinct
    noise types. Each noise type is a smooth continuous f(x,y) variation close
    to a corresponding GT pattern family.

    CRITICAL INVARIANT: All operations use continuous spatial functions.
    No rng.rand(n), no grid-scale discretisation, no nearest-neighbor with few seeds.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    cx_n, cy_n = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx_n)**2 + (y - cy_n)**2)
    max_dist = dist_center.max() + 1e-8

    struct_seed = noise_idx * 7 + 42
    rng_struct = np.random.RandomState(struct_seed)
    noise_type = noise_idx % N_NOISE_TYPES

    # Pre-draw ALL structural parameters unconditionally (fixed draw order)
    base_theta     = rng_struct.uniform(0, 2*np.pi)
    base_bx_frac   = rng_struct.uniform(0.1, 0.9)
    base_by_frac   = rng_struct.uniform(0.1, 0.9)
    base_sigma_b   = rng_struct.uniform(300, 700)
    base_freq      = rng_struct.uniform(80, 300)
    base_angle_d   = rng_struct.uniform(0, np.pi)
    base_target_r  = rng_struct.uniform(0.15, 0.55)
    base_sigma_r   = rng_struct.uniform(0.04, 0.12)
    base_p1_idx    = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_p2_idx    = rng_struct.randint(0, min(20, len(gt_patterns_list)))
    base_mix_w     = rng_struct.uniform(0.3, 0.7)
    base_inv_idx   = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_thr_idx   = rng_struct.randint(0, min(15, len(gt_patterns_list)))
    base_threshold = rng_struct.uniform(0.35, 0.65)
    base_freq_x    = rng_struct.uniform(300, 800)
    base_freq_y    = rng_struct.uniform(300, 800)
    base_phase_x   = rng_struct.uniform(0, 2*np.pi)
    base_phase_y   = rng_struct.uniform(0, 2*np.pi)
    base_n_spots   = rng_struct.randint(8, 25)
    base_spot_xs   = rng_struct.uniform(0.05, 0.95, 30) * width
    base_spot_ys   = rng_struct.uniform(0.05, 0.95, 30) * height
    base_spot_sigs = rng_struct.uniform(100, 300, 30)
    base_sf_freqs  = rng_struct.uniform(200, 800, (5, 2))
    base_sf_phases = rng_struct.uniform(0, 2*np.pi, 5)
    base_sf_amps   = rng_struct.uniform(0.1, 0.4, 5)
    base_checker_freq   = rng_struct.uniform(150, 500)
    base_n_sectors      = rng_struct.choice([3, 5, 6, 7, 9, 10, 11])
    base_conc_freq      = rng_struct.uniform(30, 80)
    base_spiral_rate    = rng_struct.uniform(100, 250)
    base_ellipse_a_frac = rng_struct.uniform(0.3, 0.6)
    base_ellipse_b_frac = rng_struct.uniform(0.15, 0.40)
    base_n_layers       = rng_struct.randint(3, 8)
    base_layer_vals     = rng_struct.rand(8)
    base_n_branches     = rng_struct.randint(3, 9)
    base_branch_width   = rng_struct.uniform(0.15, 0.45)
    base_n_mosaic       = rng_struct.randint(15, 40)
    base_mosaic_xs      = rng_struct.uniform(0.05, 0.95, 40) * width
    base_mosaic_ys      = rng_struct.uniform(0.05, 0.95, 40) * height
    base_mosaic_sigs    = rng_struct.uniform(200, 600, 40)
    base_mosaic_vals    = rng_struct.rand(40)
    base_sigmoid_shift  = rng_struct.uniform(0.3, 0.7)
    base_sigmoid_steep  = rng_struct.uniform(6, 18)
    base_intf_src1_x    = rng_struct.uniform(0.15, 0.40) * width
    base_intf_src1_y    = rng_struct.uniform(0.3, 0.7) * height
    base_intf_src2_x    = rng_struct.uniform(0.60, 0.85) * width
    base_intf_src2_y    = rng_struct.uniform(0.3, 0.7) * height
    base_intf_freq      = rng_struct.uniform(30, 80)

    # Types 20-49 additional structural params
    base_asym_theta     = rng_struct.uniform(0, 2*np.pi)
    base_asym_skew      = rng_struct.uniform(0.2, 0.8)
    base_radial_off_x   = rng_struct.uniform(0.2, 0.8)
    base_radial_off_y   = rng_struct.uniform(0.2, 0.8)
    base_dbl_freq       = rng_struct.uniform(60, 200)
    base_cross_freq1    = rng_struct.uniform(80, 250)
    base_cross_freq2    = rng_struct.uniform(80, 250)
    base_cross_angle    = rng_struct.uniform(0, np.pi)
    base_decay_edge_dir = rng_struct.uniform(0, 2*np.pi)
    base_decay_freq     = rng_struct.uniform(40, 120)
    base_standing_fx    = rng_struct.uniform(60, 200)
    base_standing_fy    = rng_struct.uniform(60, 200)
    base_elong_theta    = rng_struct.uniform(0, np.pi)
    base_elong_ratio    = rng_struct.uniform(2.0, 5.0)
    base_elong_sigma    = rng_struct.uniform(300, 700)
    base_mclust_n       = rng_struct.randint(4, 12)
    base_mclust_xs      = rng_struct.uniform(0.1, 0.9, 12) * width
    base_mclust_ys      = rng_struct.uniform(0.1, 0.9, 12) * height
    base_mclust_sigs    = rng_struct.uniform(150, 500, 12)
    base_tri_spot_sp    = rng_struct.uniform(300, 600)
    base_tri_spot_sig   = rng_struct.uniform(50, 150)
    base_hex_spot_sp    = rng_struct.uniform(250, 500)
    base_hex_spot_sig   = rng_struct.uniform(40, 120)
    base_arc_start      = rng_struct.uniform(-np.pi, np.pi)
    base_arc_span       = rng_struct.uniform(np.pi/3, np.pi)
    base_arc_r          = rng_struct.uniform(0.25, 0.55)
    base_arc_w          = rng_struct.uniform(0.03, 0.08)
    base_dblring_r1     = rng_struct.uniform(0.15, 0.35)
    base_dblring_r2     = rng_struct.uniform(0.45, 0.65)
    base_dblring_off_x  = rng_struct.uniform(-0.1, 0.1)
    base_dblring_off_y  = rng_struct.uniform(-0.1, 0.1)
    base_dblring_w      = rng_struct.uniform(0.03, 0.07)
    base_diamond_freq   = rng_struct.uniform(200, 600)
    base_triwave_freq   = rng_struct.uniform(200, 500)
    base_quad_steepness = rng_struct.uniform(5, 20)
    base_hcomb_freq     = rng_struct.uniform(150, 400)
    base_cs_core_sig    = rng_struct.uniform(0.08, 0.15)
    base_cs_shell_r     = rng_struct.uniform(0.35, 0.60)
    base_cs_shell_sig   = rng_struct.uniform(0.05, 0.12)
    base_edge_target    = rng_struct.uniform(0.5, 0.9)
    base_edge_bw        = rng_struct.uniform(0.1, 0.25)
    base_curve_amp      = rng_struct.uniform(100, 400)
    base_curve_freq2    = rng_struct.uniform(300, 700)
    base_curve_thick    = rng_struct.uniform(200, 500)
    base_punct_n        = rng_struct.randint(30, 70)
    base_punct_xs       = rng_struct.uniform(0.05, 0.95, 70) * width
    base_punct_ys       = rng_struct.uniform(0.05, 0.95, 70) * height
    base_punct_sig      = rng_struct.uniform(30, 80)
    base_peri_void_r    = rng_struct.uniform(0.10, 0.20)
    base_peri_ring_r    = rng_struct.uniform(0.22, 0.35)
    base_peri_ring_w    = rng_struct.uniform(0.05, 0.12)
    base_lobe1_off_x    = rng_struct.uniform(-0.25, -0.10)
    base_lobe1_off_y    = rng_struct.uniform(-0.10, 0.10)
    base_lobe2_off_x    = rng_struct.uniform(0.10, 0.25)
    base_lobe2_off_y    = rng_struct.uniform(-0.10, 0.10)
    base_lobe1_sig      = rng_struct.uniform(0.10, 0.20)
    base_lobe2_sig      = rng_struct.uniform(0.08, 0.15)
    base_bimod_steep    = rng_struct.uniform(8, 20)
    base_bimod_shift    = rng_struct.uniform(0.3, 0.7)
    base_bimod_theta    = rng_struct.uniform(0, np.pi)
    base_rotcheck_theta = rng_struct.uniform(np.pi/8, np.pi/3)
    base_rotcheck_freq  = rng_struct.uniform(150, 400)
    base_taper_n        = rng_struct.randint(4, 10)
    base_taper_width    = rng_struct.uniform(0.15, 0.40)
    base_taper_decay    = rng_struct.uniform(1.5, 4.0)
    base_svor_n         = rng_struct.randint(20, 50)
    base_svor_xs        = rng_struct.uniform(0.05, 0.95, 50) * width
    base_svor_ys        = rng_struct.uniform(0.05, 0.95, 50) * height
    base_svor_sigs      = rng_struct.uniform(200, 600, 50)
    base_svor_vals      = rng_struct.rand(50)
    base_exp_decay_rate = rng_struct.uniform(1.5, 4.0)
    base_aniso_theta    = rng_struct.uniform(0, np.pi)
    base_aniso_sig_maj  = rng_struct.uniform(400, 900)
    base_aniso_sig_min  = rng_struct.uniform(100, 300)
    base_modring_r      = rng_struct.uniform(0.3, 0.6)
    base_modring_w      = rng_struct.uniform(0.04, 0.10)
    base_modring_n      = rng_struct.randint(3, 8)
    base_warp_freq      = rng_struct.uniform(60, 200)
    base_warp_theta     = rng_struct.uniform(0, np.pi)
    base_warp_mod_freq  = rng_struct.uniform(500, 1500)

    # Per-sample perturbation
    if sample_seed is not None:
        perturb_seed = (struct_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)
        dx_offset = rng_p.uniform(-0.05, 0.05) * width
        dy_offset = rng_p.uniform(-0.05, 0.05) * height
        freq_jitter = rng_p.uniform(0.90, 1.10)
        size_scale = rng_p.uniform(size_scale_range[0], size_scale_range[1])
        rot_jitter = rng_p.uniform(-0.15, 0.15)
        amp_scale = rng_p.uniform(0.80, 1.05)
        bio_noise_std = rng_p.uniform(0.0, 0.08)
    else:
        dx_offset = dy_offset = 0.0
        freq_jitter = 1.0; size_scale = 1.0; rot_jitter = 0.0
        amp_scale = 1.0; bio_noise_std = 0.0

    cx_s = cx_n + dx_offset; cy_s = cy_n + dy_offset
    dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
    max_dist_s = dist_center_s.max() + 1e-8
    angle_s = np.arctan2(y - cy_s, x - cx_s)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)

    # === NOISE TYPES 0-19 ===
    if noise_type == 0:  # Rotated gradient
        theta = base_theta + rot_jitter
        xr = x*np.cos(theta)+y*np.sin(theta); val = (xr-xr.min())/(xr.max()-xr.min()+1e-8)
    elif noise_type == 1:  # Gaussian blob
        bx = x.min()+base_bx_frac*(x.max()-x.min())+dx_offset
        by = y.min()+base_by_frac*(y.max()-y.min())+dy_offset
        sig = base_sigma_b*size_scale; d = np.sqrt((x-bx)**2+(y-by)**2)
        val = np.exp(-d**2/(2*sig**2))
    elif noise_type == 2:  # Directional stripe
        ad = base_angle_d+rot_jitter*0.3; proj = x*np.cos(ad)+y*np.sin(ad)
        val = (np.sin(proj/(base_freq*4.0))+1)/2
    elif noise_type == 3:  # Ring
        cxr = cx_n+dx_offset; cyr = cy_n+dy_offset
        d = np.sqrt((x-cxr)**2+(y-cyr)**2); mr = d.max()+1e-8
        tr = mr*base_target_r*np.clip(size_scale,0.9,1.1)
        sig = mr*base_sigma_r*np.clip(size_scale,0.9,1.1)
        val = np.exp(-((d-tr)**2)/(2*sig**2))
    elif noise_type == 4:  # Mixture of two GT patterns
        p1 = generate_pattern_values(coords, gt_patterns_list[base_p1_idx], width, height, sample_seed=None)
        p2 = generate_pattern_values(coords, gt_patterns_list[base_p2_idx], width, height, sample_seed=None)
        val = base_mix_w*p1+(1-base_mix_w)*p2
    elif noise_type == 5:  # Inverted GT pattern
        val = 1-generate_pattern_values(coords, gt_patterns_list[base_inv_idx], width, height, sample_seed=None)
    elif noise_type == 6:  # Soft-thresholded GT pattern
        bp = generate_pattern_values(coords, gt_patterns_list[base_thr_idx], width, height, sample_seed=None)
        val = 1.0/(1.0+np.exp(-12.0*(bp-base_threshold)))
    elif noise_type == 7:  # 2-D sinusoidal field
        val = (np.sin(x/(base_freq_x/freq_jitter)+base_phase_x)*np.sin(y/(base_freq_y/freq_jitter)+base_phase_y)+1)/2
    elif noise_type == 8:  # Scattered Gaussian spots
        val = np.zeros(n)
        for k in range(base_n_spots):
            sig = base_spot_sigs[k]*size_scale; d = np.sqrt((x-base_spot_xs[k])**2+(y-base_spot_ys[k])**2)
            val += np.exp(-d**2/(2*sig**2))
        val = np.clip(val, 0, 1)
    elif noise_type == 9:  # Smooth multi-frequency field
        val = np.zeros(n)
        for k in range(5):
            fx, fy = base_sf_freqs[k]; ph = base_sf_phases[k]; amp = base_sf_amps[k]
            val += amp*(np.sin(x/(fx*freq_jitter)+ph)*np.cos(y/(fy*freq_jitter)+ph*1.3)+1)/2
        val += 0.15*x_norm+0.10*y_norm; val = (val-val.min())/(val.max()-val.min()+1e-8)
    elif noise_type == 10:  # Smooth checkerboard
        freq = base_checker_freq*freq_jitter
        val = (np.sin(x/freq+dx_offset/freq)*np.sin(y/freq+dy_offset/freq)+1)/2
    elif noise_type == 11:  # Smooth sector pattern
        val = (np.cos(base_n_sectors*(angle_s+rot_jitter))+1)/2
    elif noise_type == 12:  # Damped concentric waves
        freq = base_conc_freq*freq_jitter; dn = dist_center_s/max_dist_s
        val = np.exp(-dn*1.5)*(np.sin(dist_center_s/freq)+1)/2
    elif noise_type == 13:  # Spiral pattern
        sp = angle_s+dist_center_s/(base_spiral_rate*freq_jitter)+rot_jitter
        val = (np.sin(sp*3)+1)/2
    elif noise_type == 14:  # Elliptical ring
        a = max_dist*base_ellipse_a_frac*size_scale; b = max_dist*base_ellipse_b_frac*size_scale
        ed = np.sqrt(((x-cx_s)/(a+1e-8))**2+((y-cy_s)/(b+1e-8))**2)
        val = np.exp(-((ed-1)**2)/(2*(0.12*size_scale)**2))
    elif noise_type == 15:  # Smooth concentric layers
        dn = dist_center_s/max_dist_s; sc = np.clip(dn/size_scale, 0, 1); val = np.zeros(n)
        for k in range(base_n_layers):
            t = (k+0.5)/base_n_layers; sl = 0.5/base_n_layers
            val += base_layer_vals[k]*np.exp(-((sc-t)**2)/(2*sl**2))
        val = val/(val.max()+1e-8)
    elif noise_type == 16:  # Smooth branching/radial
        bw = base_branch_width*size_scale; dn = dist_center_s/max_dist_s
        ra = angle_s+rot_jitter; av = np.exp(np.cos(base_n_branches*ra)/(bw+0.01))
        av = (av-av.min())/(av.max()-av.min()+1e-8); val = av*(1-np.exp(-dn*3))
    elif noise_type == 17:  # Multi-blob mosaic
        val = np.zeros(n); ws = np.zeros(n)+1e-8
        for k in range(base_n_mosaic):
            bx = base_mosaic_xs[k]+dx_offset; by = base_mosaic_ys[k]+dy_offset
            sig = base_mosaic_sigs[k]*size_scale
            w = np.exp(-((x-bx)**2+(y-by)**2)/(2*sig**2)); val += w*base_mosaic_vals[k]; ws += w
        val = val/ws
    elif noise_type == 18:  # Sigmoid along random axis
        theta = base_theta+rot_jitter; proj = x*np.cos(theta)+y*np.sin(theta)
        pn = (proj-proj.min())/(proj.max()-proj.min()+1e-8)
        val = 1/(1+np.exp(-base_sigmoid_steep*(pn-base_sigmoid_shift)))
    elif noise_type == 19:  # Multi-source interference
        s1x = base_intf_src1_x+dx_offset; s1y = base_intf_src1_y+dy_offset
        s2x = base_intf_src2_x+dx_offset; s2y = base_intf_src2_y+dy_offset
        d1 = np.sqrt((x-s1x)**2+(y-s1y)**2); d2 = np.sqrt((x-s2x)**2+(y-s2y)**2)
        freq = base_intf_freq*freq_jitter; val = (np.sin(d1/freq)+np.sin(d2/freq)+2)/4

    # === NEW NOISE TYPES 20-49 ===
    elif noise_type == 20:  # Asymmetric gradient
        theta = base_asym_theta+rot_jitter; proj = x*np.cos(theta)+y*np.sin(theta)
        pn = (proj-proj.min())/(proj.max()-proj.min()+1e-8); val = pn**(base_asym_skew+0.5)
    elif noise_type == 21:  # Radial gradient offset center
        ocx = x.min()+base_radial_off_x*(x.max()-x.min())+dx_offset
        ocy = y.min()+base_radial_off_y*(y.max()-y.min())+dy_offset
        do = np.sqrt((x-ocx)**2+(y-ocy)**2); val = 1-do/(do.max()+1e-8)
    elif noise_type == 22:  # Double-frequency stripes
        ad = base_angle_d+rot_jitter*0.3; proj = x*np.cos(ad)+y*np.sin(ad)
        f1 = base_dbl_freq*freq_jitter; f2 = f1*2.3
        val = (np.sin(proj/f1)+0.5*np.sin(proj/f2)+1.5)/3
    elif noise_type == 23:  # Crossed stripes
        a1 = base_cross_angle+rot_jitter; a2 = a1+np.pi/3
        p1 = x*np.cos(a1)+y*np.sin(a1); p2 = x*np.cos(a2)+y*np.sin(a2)
        f1 = base_cross_freq1*freq_jitter; f2 = base_cross_freq2*freq_jitter
        val = ((np.sin(p1/f1)+1)/2+(np.sin(p2/f2)+1)/2)/2
    elif noise_type == 24:  # Decaying waves from edge
        theta = base_decay_edge_dir+rot_jitter
        proj = x*np.cos(theta)+y*np.sin(theta)
        pn = (proj-proj.min())/(proj.max()-proj.min()+1e-8)
        perp = -x*np.sin(theta)+y*np.cos(theta); freq = base_decay_freq*freq_jitter
        val = np.exp(-pn*2)*(np.sin(perp/freq)+1)/2
    elif noise_type == 25:  # Standing wave pattern
        fx = base_standing_fx*freq_jitter; fy = base_standing_fy*freq_jitter
        val = (np.cos(x/fx+dx_offset/fx)*np.cos(y/fy+dy_offset/fy)+1)/2
    elif noise_type == 26:  # Elongated/anisotropic blob
        theta = base_elong_theta+rot_jitter; ct = np.cos(theta); st = np.sin(theta)
        dxr = (x-cx_s)*ct+(y-cy_s)*st; dyr = -(x-cx_s)*st+(y-cy_s)*ct
        sm = base_elong_sigma*size_scale; sn = sm/base_elong_ratio
        val = np.exp(-(dxr**2/(2*sm**2)+dyr**2/(2*sn**2)))
    elif noise_type == 27:  # Multi-blob cluster
        val = np.zeros(n)
        for k in range(base_mclust_n):
            bx = base_mclust_xs[k]+dx_offset; by = base_mclust_ys[k]+dy_offset
            sig = base_mclust_sigs[k]*size_scale; d = np.sqrt((x-bx)**2+(y-by)**2)
            val += np.exp(-d**2/(2*sig**2))
        val = val/(val.max()+1e-8)
    elif noise_type == 28:  # Triangular spot lattice
        sp = base_tri_spot_sp*freq_jitter; sig = base_tri_spot_sig*size_scale; val = np.zeros(n)
        for i in range(15):
            for j in range(15):
                bx = i*sp+(j%2)*(sp/2)+dx_offset; by = j*sp*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sig**2))
        val = np.clip(val, 0, 1)
    elif noise_type == 29:  # Hexagonal spot lattice
        sp = base_hex_spot_sp*freq_jitter; sig = base_hex_spot_sig*size_scale; val = np.zeros(n)
        for i in range(15):
            for j in range(15):
                bx = i*sp+(j%2)*(sp/2)+dx_offset; by = j*sp*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sig**2))
        val = np.clip(val, 0, 1)
    elif noise_type == 30:  # Partial arc ring
        tr = max_dist_s*base_arc_r*size_scale; rw = max_dist_s*base_arc_w*size_scale
        rv = np.exp(-((dist_center_s-tr)**2)/(2*rw**2))
        ac = base_arc_start+rot_jitter; adiff = np.mod(angle_s-ac+np.pi, 2*np.pi)-np.pi
        am = (np.cos(np.clip(adiff/base_arc_span*np.pi, -np.pi, np.pi))+1)/2
        val = rv*am
    elif noise_type == 31:  # Double ring with offset center
        ocx = cx_n+base_dblring_off_x*width+dx_offset; ocy = cy_n+base_dblring_off_y*height+dy_offset
        do = np.sqrt((x-ocx)**2+(y-ocy)**2); mr = do.max()+1e-8
        r1 = mr*base_dblring_r1*size_scale; r2 = mr*base_dblring_r2*size_scale
        rw = mr*base_dblring_w*size_scale
        val = np.exp(-((do-r1)**2)/(2*rw**2))+np.exp(-((do-r2)**2)/(2*rw**2))
        val = val/(val.max()+1e-8)
    elif noise_type == 32:  # Smooth diamond pattern
        freq = base_diamond_freq*freq_jitter
        val = (np.abs(np.sin((x+dx_offset)/freq))+np.abs(np.sin((y+dy_offset)/freq)))/2
    elif noise_type == 33:  # Smooth triangle wave
        freq = base_triwave_freq*freq_jitter; theta = base_angle_d+rot_jitter
        proj = x*np.cos(theta)+y*np.sin(theta)
        val = (np.arcsin(np.sin(proj/freq))/(np.pi/2)+1)/2
    elif noise_type == 34:  # Smooth quadrant
        steep = base_quad_steepness/size_scale
        sx = 1/(1+np.exp(-steep*(x_norm-0.5))); sy = 1/(1+np.exp(-steep*(y_norm-0.5)))
        val = sx*sy+(1-sx)*(1-sy)
    elif noise_type == 35:  # Honeycomb-like smooth
        freq = base_hcomb_freq*freq_jitter; val = np.zeros(n)
        for a in [0, np.pi/3, 2*np.pi/3]:
            proj = (x+dx_offset)*np.cos(a+rot_jitter)+(y+dy_offset)*np.sin(a+rot_jitter)
            val += np.cos(proj/freq)
        val = (val-val.min())/(val.max()-val.min()+1e-8)
    elif noise_type == 36:  # Core-shell variant
        cs = width*base_cs_core_sig*size_scale; sr = max_dist_s*base_cs_shell_r*size_scale
        ss = width*base_cs_shell_sig*size_scale
        val = 0.6*np.exp(-dist_center_s**2/(2*cs**2))+0.6*np.exp(-((dist_center_s-sr)**2)/(2*ss**2))
        val = val/(val.max()+1e-8)
    elif noise_type == 37:  # Edge enhancement variant
        dn = dist_center_s/max_dist_s; t = base_edge_target*size_scale; bw = base_edge_bw*size_scale
        val = 1-np.exp(-((dn-t)**2)/(2*bw**2))
    elif noise_type == 38:  # Curved laminar smooth
        cy2 = (y+dy_offset)-base_curve_amp*np.sin((x+dx_offset)/(base_curve_freq2*freq_jitter))
        val = (np.sin(cy2/(base_curve_thick*size_scale))+1)/2
    elif noise_type == 39:  # Dense punctate variant
        val = np.zeros(n)
        for k in range(base_punct_n):
            px = base_punct_xs[k]+dx_offset; py = base_punct_ys[k]+dy_offset
            sig = base_punct_sig*size_scale; d = np.sqrt((x-px)**2+(y-py)**2)
            val += np.exp(-d**2/(2*sig**2))
        val = np.clip(val, 0, 1)
    elif noise_type == 40:  # Periventricular variant
        dn = dist_center_s/max_dist_s; vr = base_peri_void_r*size_scale
        rr = base_peri_ring_r*size_scale; rw = base_peri_ring_w*size_scale
        vv = 1/(1+np.exp(-30*(dn-vr))); rv = np.exp(-((dn-rr)**2)/(2*rw**2))
        val = rv*vv
    elif noise_type == 41:  # Asymmetric double lobe
        l1x = cx_n+base_lobe1_off_x*width+dx_offset; l1y = cy_n+base_lobe1_off_y*height+dy_offset
        l2x = cx_n+base_lobe2_off_x*width+dx_offset; l2y = cy_n+base_lobe2_off_y*height+dy_offset
        s1 = width*base_lobe1_sig*size_scale; s2 = width*base_lobe2_sig*size_scale
        d1 = np.sqrt((x-l1x)**2+(y-l1y)**2); d2 = np.sqrt((x-l2x)**2+(y-l2y)**2)
        val = 0.65*np.exp(-d1**2/(2*s1**2))+0.55*np.exp(-d2**2/(2*s2**2))
        val = val/(val.max()+1e-8)
    elif noise_type == 42:  # Bimodal sigmoid step
        theta = base_bimod_theta+rot_jitter; proj = x*np.cos(theta)+y*np.sin(theta)
        pn = (proj-proj.min())/(proj.max()-proj.min()+1e-8)
        val = 0.3+0.5/(1+np.exp(-base_bimod_steep*(pn-base_bimod_shift)))
    elif noise_type == 43:  # Rotated checkerboard
        theta = base_rotcheck_theta+rot_jitter; ct = np.cos(theta); st = np.sin(theta)
        xr = (x+dx_offset)*ct+(y+dy_offset)*st; yr = -(x+dx_offset)*st+(y+dy_offset)*ct
        freq = base_rotcheck_freq*freq_jitter; val = (np.sin(xr/freq)*np.sin(yr/freq)+1)/2
    elif noise_type == 44:  # Tapered radial arms
        dn = dist_center_s/max_dist_s; ra = angle_s+rot_jitter
        av = np.exp(np.cos(base_taper_n*ra)/(base_taper_width*size_scale+0.01))
        av = (av-av.min())/(av.max()-av.min()+1e-8)
        tp = np.exp(-dn*base_taper_decay); val = av*(1-tp)*tp
    elif noise_type == 45:  # Smooth Voronoi-like
        val = np.zeros(n); ws = np.zeros(n)+1e-8
        for k in range(base_svor_n):
            bx = base_svor_xs[k]+dx_offset; by = base_svor_ys[k]+dy_offset
            sig = base_svor_sigs[k]*size_scale
            w = np.exp(-((x-bx)**2+(y-by)**2)/(2*sig**2)); val += w*base_svor_vals[k]; ws += w
        val = val/ws
    elif noise_type == 46:  # Exponential radial decay
        dn = dist_center_s/max_dist_s; val = np.exp(-base_exp_decay_rate*dn*size_scale)
    elif noise_type == 47:  # Anisotropic Gaussian
        theta = base_aniso_theta+rot_jitter; ct = np.cos(theta); st = np.sin(theta)
        dxr = (x-cx_s)*ct+(y-cy_s)*st; dyr = -(x-cx_s)*st+(y-cy_s)*ct
        sm = base_aniso_sig_maj*size_scale; sn = base_aniso_sig_min*size_scale
        val = np.exp(-(dxr**2/(2*sm**2)+dyr**2/(2*sn**2)))
    elif noise_type == 48:  # Modulated ring
        tr = max_dist_s*base_modring_r*size_scale; rw = max_dist_s*base_modring_w*size_scale
        rv = np.exp(-((dist_center_s-tr)**2)/(2*rw**2))
        am = (np.cos(base_modring_n*(angle_s+rot_jitter))+1)/2
        val = rv*(0.3+0.7*am)
    elif noise_type == 49:  # Warped stripes
        theta = base_warp_theta+rot_jitter
        proj = x*np.cos(theta)+y*np.sin(theta)
        perp = -x*np.sin(theta)+y*np.cos(theta)
        lf = base_warp_freq*freq_jitter*(1+0.3*np.sin(perp/(base_warp_mod_freq*freq_jitter)))
        val = (np.sin(proj/lf)+1)/2
    else:
        val = np.zeros(n) + 0.5

    val = val * amp_scale
    if sample_seed is not None and bio_noise_std > 0:
        rng_bio = np.random.RandomState((struct_seed ^ sample_seed) + 2)
        val = val + rng_bio.normal(0, bio_noise_std, size=n)
    val = np.clip(val, 0, 1)
    val = _apply_dissimilarity_distortion(val, coords, noise_idx, sample_seed, dissimilarity, width, height)
    return val


def get_visium_hex_grid(width, height, spacing=100):
    dy = spacing * np.sqrt(3) / 2
    coords = []
    row = 0; y = 0
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        while x < width:
            coords.append([x, y]); x += spacing
        y += dy; row += 1
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    x = np.arange(0, width, spacing); y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000

if __name__ == "__main__":

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Half-brain shapes, Visium-like RNA grid, per-sample variation")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {len(GT_PATTERNS)}")
    print(f"Noise Features: {NOISE_FEATURES}")
    print(f"Noise Types: {N_NOISE_TYPES} (= len(GT_PATTERNS), covering all pattern families)")
    print(f"Noise Dissimilarity Level: {NOISE_DISSIMILARITY}/10")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} um")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}um spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}um pixels")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}-{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"
            seed = hash(sample_id) % 2**32
            print(f"  Processing {sample_id} (seed={seed})...")

            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)

            # MSI Data
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]

            n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []

            for i, pat in enumerate(GT_PATTERNS):
                msi_data[:, i] = generate_pattern_values(
                    msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_msi.append(f"MZ_{pat}")

            for i in range(NOISE_FEATURES):
                msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                    msi_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                    dissimilarity=NOISE_DISSIMILARITY)
                var_names_msi.append(f"MZ_Noise_{i}")

            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]

            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))

            # RNA Data
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]

            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []

            for i, pat in enumerate(GT_PATTERNS):
                rna_data[:, i] = generate_pattern_values(
                    rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_rna.append(f"Gene_{pat}")
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))

            for i in range(NOISE_FEATURES):
                rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                    rna_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                    dissimilarity=dissimilarity)
                var_names_rna.append(f"Gene_Noise_{i}")

            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]

            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print("\n50 GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")
    print(f"Noise dissimilarity level: {dissimilarity}/10")
    print(f"Noise types: {N_NOISE_TYPES} (= len(GT_PATTERNS), covering all pattern families)")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, per-sample variation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Noise Types: 50 (= len(GT_PATTERNS), covering all pattern families)
Noise Dissimilarity Level: 2/10
Field size: 6000x6000 um
RNA grid: Visium-like hexagonal, 100um spacing
MSI grid: Cartesian, 60um pixels
Pattern size scale range: 0.90-1.10
Output: dummy_data_50
  Processing YC_1 (seed=994770162)...
  Processing YC_2 (seed=1682408368)...


KeyboardInterrupt: 

In [7]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100
MSI_PIXEL_SIZE = 60

PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

# =============================================================================
# NOISE DISSIMILARITY CONTROL
# =============================================================================
# Controls how different noise features are from their parent GT pattern.
# Range: 1 (minimal difference, very close to GT) to 10 (maximal difference).
# Default 2 provides subtle separation.
#
# All distortions use smooth continuous spatial functions so that RNA and MSI
# grids sampling the same underlying field stay perfectly correlated.
NOISE_DISSIMILARITY = 3  # <-- USER-ADJUSTABLE: integer 1-10

# Number of noise variants per GT pattern.
# Total noise features = NOISE_PER_PATTERN * len(GT_PATTERNS)
NOISE_PER_PATTERN = 10

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X", "Gradient_Y", "Gradient_Diagonal_NE", "Gradient_Diagonal_NW",
    "Gradient_Radial_In", "Gradient_Radial_Out",
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical", "Stripes_Horizontal", "Stripes_Diagonal_45",
    "Stripes_Diagonal_135", "Waves_Concentric", "Waves_Spiral",
    "Waves_Interference", "Waves_Ripple",
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center", "Blob_TopRight", "Blob_TopLeft", "Blob_BottomRight",
    "Blob_BottomLeft", "Spots_Grid_Dense", "Spots_Grid_Sparse",
    "Spots_Random_Large", "Spots_Triangular", "Spots_Hexagonal",
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner", "Ring_Outer", "Ring_Double", "Ring_Eccentric",
    "Ring_Elliptical", "Ring_Partial",
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine", "Checkerboard_Coarse", "Quadrant_Alternating",
    "Sectors_4", "Sectors_8", "Triangle_Pattern", "Diamond_Pattern", "Honeycomb",
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers", "Hotspot_Cluster", "Edge_Enhancement", "Core_Shell",
    "Branching", "Laminar_Curved", "Mosaic_Irregular", "Gradient_Sigmoid",
    "Bimodal_Distribution", "Punctate_Dense", "Periventricular", "Asymmetric_Lobe",
]

TOTAL_NOISE_FEATURES = NOISE_PER_PATTERN * len(GT_PATTERNS)

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    np.random.seed(sample_seed)
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height

    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        dx = x - cx; dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        if x_rot < left_edge: return False
        if y_rot < bottom_edge: return False
        if y_rot > top_edge: return False
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        right_edge = left_edge + right_width
        if x_rot > right_edge: return False
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            if x_rel < 0.05: max_y_norm = 1.0
            else: max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            if y_norm > max_y_norm: return False
        return True
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    angle = np.arctan2(y - cy, x - cx)
    pattern_seed = hash(pattern_type) % 2**31

    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height
        phase_offset = rng_s.uniform(-1.0, 1.0)
        freq_jitter = rng_s.uniform(0.75, 1.25)
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])
        amp_scale = rng_s.uniform(0.75, 1.05)
        bio_noise_std = rng_s.uniform(0.0, 0.10)
        cx_s = cx + dx_offset; cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0; freq_jitter = 1.0; size_scale = 1.0
        amp_scale = 1.0; bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center; angle_s = angle

    if pattern_type == "Gradient_X": val = x_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Y": val = y_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Diagonal_NE": val = (x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Diagonal_NW": val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Radial_In": val = 1 - dist_norm + phase_offset * 0.1
    elif pattern_type == "Gradient_Radial_Out": val = dist_norm + phase_offset * 0.1
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
    elif pattern_type == "Waves_Interference":
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset; by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed); sigma = width * 0.1 * size_scale; val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset; by = rng.uniform(y.min(), y.max()) + dy_offset
            val += np.exp(-((x-bx)**2+(y-by)**2) / (2*sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter; spot_sigma = 80 * size_scale; val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter; spot_sigma = 60 * size_scale; val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2*size_scale+phase_offset*0.05); ring_width = width*0.04*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7*size_scale+phase_offset*0.05); ring_width = width*0.05*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Double":
        r1 = max_dist*(0.25*size_scale+phase_offset*0.03); r2 = max_dist*(0.55*size_scale+phase_offset*0.03)
        rw = width*0.03*size_scale
        val = np.exp(-((dist_center_s-r1)**2)/(2*rw**2))+np.exp(-((dist_center_s-r2)**2)/(2*rw**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx+width*0.15+dx_offset; off_cy = cy+height*0.1+dy_offset
        dist_off = np.sqrt((x-off_cx)**2+(y-off_cy)**2)
        target_r = max_dist*(0.35*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        val = np.exp(-((dist_off-target_r)**2)/(2*rw**2))
    elif pattern_type == "Ring_Elliptical":
        a = max_dist*(0.5*size_scale+phase_offset*0.05); b = max_dist*(0.3*size_scale+phase_offset*0.03)
        ed = np.sqrt(((x-cx_s)/a)**2+((y-cy_s)/b)**2); rw = 0.1*size_scale
        val = np.exp(-((ed-1)**2)/(2*rw**2))
    elif pattern_type == "Ring_Partial":
        target_r = max_dist*(0.4*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        rv = np.exp(-((dist_center_s-target_r)**2)/(2*rw**2))
        ah = (np.pi/2)*size_scale; a_s = -ah+phase_offset*0.3; a_e = ah+phase_offset*0.3
        val = rv * ((angle_s > a_s) & (angle_s < a_e))
    elif pattern_type == "Checkerboard_Fine":
        ss = 200*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Checkerboard_Coarse":
        ss = 500*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Quadrant_Alternating":
        val = ((x>cx_s)&(y>cy_s) | (x<cx_s)&(y<cy_s)).astype(float)
    elif pattern_type == "Sectors_4":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/2)).astype(int)%4; val = (sector%2).astype(float)
    elif pattern_type == "Sectors_8":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/4)).astype(int)%8; val = (sector%2).astype(float)
    elif pattern_type == "Triangle_Pattern":
        period = 300*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)/(period/2)+np.abs(((y+dy_offset)%period)-period/2)/(period/2))/2
    elif pattern_type == "Diamond_Pattern":
        period = 400*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)+np.abs(((y+dy_offset)%period)-period/2))/period
    elif pattern_type == "Honeycomb":
        spacing = 200*freq_jitter; val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i*spacing+(j%2)*(spacing/2)+dx_offset; hy = j*spacing*0.866+dy_offset
                dist = np.sqrt((x-hx)**2+(y-hy)**2)
                val = np.maximum(val, 1-np.clip(dist/(spacing*0.4*size_scale), 0, 1))
        val = 1-val
    elif pattern_type == "Cortical_Layers":
        n_layers = 5; layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        sd = np.sqrt((x-cx_s)**2+(y-cy_s)**2); sn = sd/(sd.max()+1e-8)
        sc = np.clip(sn/size_scale, 0, 1); li = np.clip((sc*n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in li])
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed); sigma = width*0.05*size_scale; val = np.zeros(n)
        ccx = cx+rng.uniform(-width*0.1, width*0.1)+dx_offset; ccy = cy+rng.uniform(-height*0.1, height*0.1)+dy_offset
        for _ in range(8):
            bx = ccx+rng.normal(0, width*0.08); by = ccy+rng.normal(0, height*0.08)
            val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sigma**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Edge_Enhancement":
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        target = np.clip(0.8*size_scale+phase_offset*0.05, 0.3, 1.2); bw = 0.15*size_scale
        val = np.clip(1-np.exp(-((sn-target)**2)/(2*bw**2)), 0, 1)
    elif pattern_type == "Core_Shell":
        cs = width*0.1*size_scale; sr = max_dist*(0.5+phase_offset*0.05)*size_scale; ss = width*0.08*size_scale
        val = 0.7*np.exp(-dist_center_s**2/(2*cs**2))+0.5*np.exp(-((dist_center_s-sr)**2)/(2*ss**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Branching":
        nb = 5; ba = 2*np.pi/nb; bw = 0.3*size_scale; ra = angle_s+phase_offset*0.5
        mdb = np.inf*np.ones(n)
        for i in range(nb):
            ta = i*ba; ad2 = np.abs(np.mod(ra-ta+np.pi, 2*np.pi)-np.pi)
            mdb = np.minimum(mdb, ad2)
        val = np.exp(-mdb**2/(2*bw**2))*(1-np.exp(-dist_norm*3))
    elif pattern_type == "Laminar_Curved":
        ca = 200*(1+phase_offset*0.3); cf = 400*freq_jitter; bt = 300*size_scale
        cy2 = (y+dy_offset)-ca*np.sin((x+dx_offset)/cf)
        val = ((cy2//bt).astype(int)%2).astype(float)
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed); ns = max(5, int(20/size_scale))
        sx = rng.uniform(x.min(), x.max(), ns)+dx_offset; sy = rng.uniform(y.min(), y.max(), ns)+dy_offset
        sv = rng.rand(ns); ni = np.zeros(n, dtype=int); md = np.inf*np.ones(n)
        for i in range(ns):
            d = np.sqrt((x-sx[i])**2+(y-sy[i])**2); c = d<md; ni[c] = i; md[c] = d[c]
        val = sv[ni]
    elif pattern_type == "Gradient_Sigmoid":
        shift = 0.5+phase_offset*0.1; steep = 10.0/size_scale
        val = 1/(1+np.exp(-steep*(x_norm-shift)))
    elif pattern_type == "Bimodal_Distribution":
        val = (x<cx_s).astype(float)*0.8+(x>=cx_s).astype(float)*0.3
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed); ps = 40*size_scale; val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())+dx_offset; py = rng.uniform(y.min(), y.max())+dy_offset
            val += np.exp(-((x-px)**2+(y-py)**2)/(2*ps**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Periventricular":
        vr = 0.15*size_scale+phase_offset*0.02; rr = 0.25*size_scale+phase_offset*0.02; rw = 0.08*size_scale
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        val = np.exp(-((sn-rr)**2)/(2*rw**2))*(sn>=vr)
    elif pattern_type == "Asymmetric_Lobe":
        l1x = cx-width*0.2+dx_offset; l1y = cy+dy_offset; l2x = cx+width*0.15+dx_offset; l2y = cy+height*0.1+dy_offset
        s1 = width*0.15*size_scale; s2 = width*0.10*size_scale
        val = 0.7*np.exp(-((x-l1x)**2+(y-l1y)**2)/(2*s1**2))+0.5*np.exp(-((x-l2x)**2+(y-l2y)**2)/(2*s2**2))
        val = val/(val.max()+1e-8)
    else:
        rng = np.random.RandomState(pattern_seed); val = rng.rand(n)

    val = val * amp_scale
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)
    return np.clip(val, 0, 1)


def generate_noise_from_parent(coords, parent_pattern, variant_idx,
                               width=3000, height=3000,
                               sample_seed=None, size_scale_range=(0.70, 1.40),
                               dissimilarity=2):
    """
    Generate a noise feature as a distorted variant of a specific parent GT pattern.

    Each noise feature starts from the parent pattern (generated with sample_seed=None
    to get the structural base) and then applies smooth continuous distortions
    controlled by NOISE_DISSIMILARITY (1-10).

    variant_idx (0..NOISE_PER_PATTERN-1) determines which unique distortion is applied.
    Different variant_idx values produce different distortions of the same parent.

    CRITICAL INVARIANT: All distortions use smooth continuous spatial functions
    f(x, y). The RNG draws only scalar parameters (never n-dependent arrays).
    This guarantees identical distortion on RNA and MSI grids.

    Distortion mechanisms (scaled by d = (dissimilarity-1)/9, range 0-1):
      1. Multiplicative spatial modulation — low-frequency sinusoidal field
      2. Power-law warping — spatially uniform exponent
      3. Smooth additive field — low-frequency sinusoid at unique angle
      4. Spatial shift/warp — smooth coordinate remapping
      5. Frequency modulation — local frequency perturbation
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    d = (dissimilarity - 1) / 9.0  # 0 to 1

    # Generate the parent pattern (structural base, no per-sample variation)
    val = generate_pattern_values(coords, parent_pattern, width, height,
                                  sample_seed=None, add_noise=False,
                                  size_scale_range=size_scale_range)

    if d < 0.001:
        # Even at dissimilarity=1, add tiny per-variant uniqueness
        d = 0.01

    # Unique RNG for this parent+variant combination — scalar draws only
    parent_hash = hash(parent_pattern) % 2**31
    variant_seed = (parent_hash * 1000 + variant_idx * 7 + 42) % 2**31
    rng = np.random.RandomState(variant_seed)

    # --- Distortion 1: Multiplicative spatial modulation ---
    mod_freq_x = rng.uniform(800, 2500)
    mod_freq_y = rng.uniform(800, 2500)
    mod_phase  = rng.uniform(0, 2 * np.pi)
    mod_amp    = d * rng.uniform(0.2, 0.5)
    modulation = (np.sin(x / mod_freq_x + mod_phase) *
                  np.cos(y / mod_freq_y + mod_phase * 0.7) + 1) / 2
    val = val * (1 - mod_amp) + val * modulation * mod_amp

    # --- Distortion 2: Power-law warping ---
    exp_direction = rng.choice([-1, 1])
    exponent = 1.0 + d * rng.uniform(0.5, 2.0) * exp_direction
    exponent = np.clip(exponent, 0.3, 3.0)
    val = np.clip(val, 0, 1) ** exponent

    # --- Distortion 3: Smooth additive field ---
    add_freq  = rng.uniform(600, 2000)
    add_theta = rng.uniform(0, 2 * np.pi)
    add_phase = rng.uniform(0, 2 * np.pi)
    add_amp   = d * rng.uniform(0.1, 0.3)
    proj = x * np.cos(add_theta) + y * np.sin(add_theta)
    additive = (np.sin(proj / add_freq + add_phase) + 1) / 2
    val = val + additive * add_amp

    # --- Distortion 4: Spatial coordinate warp (smooth) ---
    # Shifts the effective sampling coordinates via low-frequency sinusoidal warp
    warp_amp_x = d * rng.uniform(50, 300)  # um of shift
    warp_amp_y = d * rng.uniform(50, 300)
    warp_freq_x = rng.uniform(1000, 3000)
    warp_freq_y = rng.uniform(1000, 3000)
    warp_phase_x = rng.uniform(0, 2 * np.pi)
    warp_phase_y = rng.uniform(0, 2 * np.pi)
    # We re-generate the parent at warped coordinates for stronger distortion
    if d > 0.15:
        x_warped = x + warp_amp_x * np.sin(y / warp_freq_y + warp_phase_y)
        y_warped = y + warp_amp_y * np.sin(x / warp_freq_x + warp_phase_x)
        warped_coords = np.column_stack([x_warped, y_warped])
        val_warped = generate_pattern_values(warped_coords, parent_pattern, width, height,
                                              sample_seed=None, add_noise=False,
                                              size_scale_range=size_scale_range)
        # Blend original and warped based on dissimilarity
        blend = min(d * 1.5, 0.8)
        val = val * (1 - blend) + val_warped * blend

    # --- Distortion 5: Secondary frequency component ---
    sec_freq  = rng.uniform(100, 500)
    sec_theta = rng.uniform(0, 2 * np.pi)
    sec_phase = rng.uniform(0, 2 * np.pi)
    sec_amp   = d * rng.uniform(0.05, 0.2)
    proj2 = x * np.cos(sec_theta) + y * np.sin(sec_theta)
    secondary = (np.sin(proj2 / sec_freq + sec_phase) + 1) / 2
    val = val + secondary * sec_amp

    val = np.clip(val, 0, 1)

    # --- Per-sample variation (same mechanism as GT patterns) ---
    if sample_seed is not None:
        perturb_seed = (variant_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)
        amp_scale     = rng_p.uniform(0.80, 1.05)
        bio_noise_std = rng_p.uniform(0.0, 0.08)

        val = val * amp_scale
        if bio_noise_std > 0:
            rng_bio = np.random.RandomState((variant_seed ^ sample_seed) + 2)
            val = val + rng_bio.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    dy = spacing * np.sqrt(3) / 2
    coords = []; row = 0; y = 0
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        while x < width:
            coords.append([x, y]); x += spacing
        y += dy; row += 1
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    x = np.arange(0, width, spacing); y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================
if __name__ == "__main__":

    n_gt = len(GT_PATTERNS)
    total_noise = NOISE_PER_PATTERN * n_gt
    n_total_features = n_gt + total_noise

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Each GT pattern produces noise variants controlled by dissimilarity")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {n_gt}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Total noise features: {total_noise}")
    print(f"Total features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise Dissimilarity Level: {NOISE_DISSIMILARITY}/10")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} um")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}um spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}um pixels")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}-{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"
            seed = hash(sample_id) % 2**32
            print(f"\n  Processing {sample_id} (seed={seed})...")

            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)

            # === MSI Data ===
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(xi, yi) for xi, yi in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]

            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    MSI GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                msi_data[:, col] = generate_pattern_values(
                    msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_msi.append(f"MZ_{pat}")
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    msi_data[:, col] = generate_noise_from_parent(
                        msi_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        dissimilarity=NOISE_DISSIMILARITY)
                    var_names_msi.append(f"MZ_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]

            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))

            # === RNA Data ===
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(xi, yi) for xi, yi in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]

            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    RNA GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                rna_data[:, col] = generate_pattern_values(
                    rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_rna.append(f"Gene_{pat}")
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    rna_data[:, col] = generate_noise_from_parent(
                        rna_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        dissimilarity=NOISE_DISSIMILARITY)
                    var_names_rna.append(f"Gene_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]

            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print(f"\n{n_gt} GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Noise dissimilarity level: {NOISE_DISSIMILARITY}/10")
    print(f"\nNoise naming: MZ_Noise_<PatternName>_<VariantIdx> / Gene_Noise_<PatternName>_<VariantIdx>")
    print(f"Example: MZ_Noise_Gradient_X_0 is variant 0 of Gradient_X noise")

Generating Ground Truth Dummy Data with 50 PATTERNS
Each GT pattern produces noise variants controlled by dissimilarity
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise per pattern: 10
Total noise features: 500
Total features per sample: 50 GT + 500 Noise = 550
Noise Dissimilarity Level: 3/10
Field size: 6000x6000 um
RNA grid: Visium-like hexagonal, 100um spacing
MSI grid: Cartesian, 60um pixels
Pattern size scale range: 0.90-1.10
Output: dummy_data_50

  Processing YC_1 (seed=994770162)...
    MSI GT  1/50: Gradient_X + 10 noise variants
    MSI GT  2/50: Gradient_Y + 10 noise variants
    MSI GT  3/50: Gradient_Diagonal_NE + 10 noise variants
    MSI GT  4/50: Gradient_Diagonal_NW + 10 noise variants
    MSI GT  5/50: Gradient_Radial_In + 10 noise variants
    MSI GT  6/50: Gradient_Radial_Out + 10 noise variants
    MSI GT  7/50: Stripes_Vertical + 10 noise variants
    MSI GT  8/50: Stripes_Horizontal + 10 noise variants
    MSI GT  9/50: Stripes_Diagonal_45 + 10 noise vari

In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100
MSI_PIXEL_SIZE = 60

PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

# =============================================================================
# NOISE DISSIMILARITY CONTROL
# =============================================================================
# Controls how different noise features are from their parent GT pattern.
# Range: 1 (minimal difference, very close to GT) to 10 (maximal difference).
# Default 2 provides subtle separation.
#
# All distortions use smooth continuous spatial functions so that RNA and MSI
# grids sampling the same underlying field stay perfectly correlated.
NOISE_DISSIMILARITY = 5  # <-- USER-ADJUSTABLE: integer 1-10

# Number of noise variants per GT pattern.
# Total noise features = NOISE_PER_PATTERN * len(GT_PATTERNS)
NOISE_PER_PATTERN = 10

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X", "Gradient_Y", "Gradient_Diagonal_NE", "Gradient_Diagonal_NW",
    "Gradient_Radial_In", "Gradient_Radial_Out",
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical", "Stripes_Horizontal", "Stripes_Diagonal_45",
    "Stripes_Diagonal_135", "Waves_Concentric", "Waves_Spiral",
    "Waves_Interference", "Waves_Ripple",
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center", "Blob_TopRight", "Blob_TopLeft", "Blob_BottomRight",
    "Blob_BottomLeft", "Spots_Grid_Dense", "Spots_Grid_Sparse",
    "Spots_Random_Large", "Spots_Triangular", "Spots_Hexagonal",
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner", "Ring_Outer", "Ring_Double", "Ring_Eccentric",
    "Ring_Elliptical", "Ring_Partial",
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine", "Checkerboard_Coarse", "Quadrant_Alternating",
    "Sectors_4", "Sectors_8", "Triangle_Pattern", "Diamond_Pattern", "Honeycomb",
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers", "Hotspot_Cluster", "Edge_Enhancement", "Core_Shell",
    "Branching", "Laminar_Curved", "Mosaic_Irregular", "Gradient_Sigmoid",
    "Bimodal_Distribution", "Punctate_Dense", "Periventricular", "Asymmetric_Lobe",
]

TOTAL_NOISE_FEATURES = NOISE_PER_PATTERN * len(GT_PATTERNS)

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    np.random.seed(sample_seed)
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height

    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        dx = x - cx; dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        if x_rot < left_edge: return False
        if y_rot < bottom_edge: return False
        if y_rot > top_edge: return False
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        right_edge = left_edge + right_width
        if x_rot > right_edge: return False
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            if x_rel < 0.05: max_y_norm = 1.0
            else: max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            if y_norm > max_y_norm: return False
        return True
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    angle = np.arctan2(y - cy, x - cx)
    pattern_seed = hash(pattern_type) % 2**31

    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height
        phase_offset = rng_s.uniform(-1.0, 1.0)
        freq_jitter = rng_s.uniform(0.75, 1.25)
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])
        amp_scale = rng_s.uniform(0.75, 1.05)
        bio_noise_std = rng_s.uniform(0.0, 0.10)
        cx_s = cx + dx_offset; cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0; freq_jitter = 1.0; size_scale = 1.0
        amp_scale = 1.0; bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center; angle_s = angle

    if pattern_type == "Gradient_X": val = x_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Y": val = y_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Diagonal_NE": val = (x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Diagonal_NW": val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Radial_In": val = 1 - dist_norm + phase_offset * 0.1
    elif pattern_type == "Gradient_Radial_Out": val = dist_norm + phase_offset * 0.1
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
    elif pattern_type == "Waves_Interference":
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset; by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed); sigma = width * 0.1 * size_scale; val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset; by = rng.uniform(y.min(), y.max()) + dy_offset
            val += np.exp(-((x-bx)**2+(y-by)**2) / (2*sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter; spot_sigma = 80 * size_scale; val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter; spot_sigma = 60 * size_scale; val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2*size_scale+phase_offset*0.05); ring_width = width*0.04*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7*size_scale+phase_offset*0.05); ring_width = width*0.05*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Double":
        r1 = max_dist*(0.25*size_scale+phase_offset*0.03); r2 = max_dist*(0.55*size_scale+phase_offset*0.03)
        rw = width*0.03*size_scale
        val = np.exp(-((dist_center_s-r1)**2)/(2*rw**2))+np.exp(-((dist_center_s-r2)**2)/(2*rw**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx+width*0.15+dx_offset; off_cy = cy+height*0.1+dy_offset
        dist_off = np.sqrt((x-off_cx)**2+(y-off_cy)**2)
        target_r = max_dist*(0.35*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        val = np.exp(-((dist_off-target_r)**2)/(2*rw**2))
    elif pattern_type == "Ring_Elliptical":
        a = max_dist*(0.5*size_scale+phase_offset*0.05); b = max_dist*(0.3*size_scale+phase_offset*0.03)
        ed = np.sqrt(((x-cx_s)/a)**2+((y-cy_s)/b)**2); rw = 0.1*size_scale
        val = np.exp(-((ed-1)**2)/(2*rw**2))
    elif pattern_type == "Ring_Partial":
        target_r = max_dist*(0.4*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        rv = np.exp(-((dist_center_s-target_r)**2)/(2*rw**2))
        ah = (np.pi/2)*size_scale; a_s = -ah+phase_offset*0.3; a_e = ah+phase_offset*0.3
        val = rv * ((angle_s > a_s) & (angle_s < a_e))
    elif pattern_type == "Checkerboard_Fine":
        ss = 200*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Checkerboard_Coarse":
        ss = 500*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Quadrant_Alternating":
        val = ((x>cx_s)&(y>cy_s) | (x<cx_s)&(y<cy_s)).astype(float)
    elif pattern_type == "Sectors_4":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/2)).astype(int)%4; val = (sector%2).astype(float)
    elif pattern_type == "Sectors_8":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/4)).astype(int)%8; val = (sector%2).astype(float)
    elif pattern_type == "Triangle_Pattern":
        period = 300*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)/(period/2)+np.abs(((y+dy_offset)%period)-period/2)/(period/2))/2
    elif pattern_type == "Diamond_Pattern":
        period = 400*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)+np.abs(((y+dy_offset)%period)-period/2))/period
    elif pattern_type == "Honeycomb":
        spacing = 200*freq_jitter; val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i*spacing+(j%2)*(spacing/2)+dx_offset; hy = j*spacing*0.866+dy_offset
                dist = np.sqrt((x-hx)**2+(y-hy)**2)
                val = np.maximum(val, 1-np.clip(dist/(spacing*0.4*size_scale), 0, 1))
        val = 1-val
    elif pattern_type == "Cortical_Layers":
        n_layers = 5; layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        sd = np.sqrt((x-cx_s)**2+(y-cy_s)**2); sn = sd/(sd.max()+1e-8)
        sc = np.clip(sn/size_scale, 0, 1); li = np.clip((sc*n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in li])
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed); sigma = width*0.05*size_scale; val = np.zeros(n)
        ccx = cx+rng.uniform(-width*0.1, width*0.1)+dx_offset; ccy = cy+rng.uniform(-height*0.1, height*0.1)+dy_offset
        for _ in range(8):
            bx = ccx+rng.normal(0, width*0.08); by = ccy+rng.normal(0, height*0.08)
            val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sigma**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Edge_Enhancement":
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        target = np.clip(0.8*size_scale+phase_offset*0.05, 0.3, 1.2); bw = 0.15*size_scale
        val = np.clip(1-np.exp(-((sn-target)**2)/(2*bw**2)), 0, 1)
    elif pattern_type == "Core_Shell":
        cs = width*0.1*size_scale; sr = max_dist*(0.5+phase_offset*0.05)*size_scale; ss = width*0.08*size_scale
        val = 0.7*np.exp(-dist_center_s**2/(2*cs**2))+0.5*np.exp(-((dist_center_s-sr)**2)/(2*ss**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Branching":
        nb = 5; ba = 2*np.pi/nb; bw = 0.3*size_scale; ra = angle_s+phase_offset*0.5
        mdb = np.inf*np.ones(n)
        for i in range(nb):
            ta = i*ba; ad2 = np.abs(np.mod(ra-ta+np.pi, 2*np.pi)-np.pi)
            mdb = np.minimum(mdb, ad2)
        val = np.exp(-mdb**2/(2*bw**2))*(1-np.exp(-dist_norm*3))
    elif pattern_type == "Laminar_Curved":
        ca = 200*(1+phase_offset*0.3); cf = 400*freq_jitter; bt = 300*size_scale
        cy2 = (y+dy_offset)-ca*np.sin((x+dx_offset)/cf)
        val = ((cy2//bt).astype(int)%2).astype(float)
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed); ns = max(5, int(20/size_scale))
        sx = rng.uniform(x.min(), x.max(), ns)+dx_offset; sy = rng.uniform(y.min(), y.max(), ns)+dy_offset
        sv = rng.rand(ns); ni = np.zeros(n, dtype=int); md = np.inf*np.ones(n)
        for i in range(ns):
            d = np.sqrt((x-sx[i])**2+(y-sy[i])**2); c = d<md; ni[c] = i; md[c] = d[c]
        val = sv[ni]
    elif pattern_type == "Gradient_Sigmoid":
        shift = 0.5+phase_offset*0.1; steep = 10.0/size_scale
        val = 1/(1+np.exp(-steep*(x_norm-shift)))
    elif pattern_type == "Bimodal_Distribution":
        val = (x<cx_s).astype(float)*0.8+(x>=cx_s).astype(float)*0.3
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed); ps = 40*size_scale; val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())+dx_offset; py = rng.uniform(y.min(), y.max())+dy_offset
            val += np.exp(-((x-px)**2+(y-py)**2)/(2*ps**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Periventricular":
        vr = 0.15*size_scale+phase_offset*0.02; rr = 0.25*size_scale+phase_offset*0.02; rw = 0.08*size_scale
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        val = np.exp(-((sn-rr)**2)/(2*rw**2))*(sn>=vr)
    elif pattern_type == "Asymmetric_Lobe":
        l1x = cx-width*0.2+dx_offset; l1y = cy+dy_offset; l2x = cx+width*0.15+dx_offset; l2y = cy+height*0.1+dy_offset
        s1 = width*0.15*size_scale; s2 = width*0.10*size_scale
        val = 0.7*np.exp(-((x-l1x)**2+(y-l1y)**2)/(2*s1**2))+0.5*np.exp(-((x-l2x)**2+(y-l2y)**2)/(2*s2**2))
        val = val/(val.max()+1e-8)
    else:
        rng = np.random.RandomState(pattern_seed); val = rng.rand(n)

    val = val * amp_scale
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)
    return np.clip(val, 0, 1)


def generate_noise_from_parent(coords, parent_pattern, variant_idx,
                               width=3000, height=3000,
                               sample_seed=None, size_scale_range=(0.70, 1.40),
                               dissimilarity=2):
    """
    Generate a noise feature as a distorted variant of a specific parent GT pattern.

    Each noise feature starts from the parent pattern (generated with the same
    sample_seed so it matches that sample's GT pattern) and then applies smooth
    continuous distortions controlled by NOISE_DISSIMILARITY (1-10).

    variant_idx (0..NOISE_PER_PATTERN-1) determines which unique distortion is applied.
    Different variant_idx values produce different distortions of the same parent.

    CRITICAL INVARIANT: All distortions use smooth continuous spatial functions
    f(x, y). The RNG draws only scalar parameters (never n-dependent arrays).
    This guarantees identical distortion on RNA and MSI grids.

    Distortion mechanisms (scaled by d = (dissimilarity-1)/9, range 0-1):
      1. Multiplicative spatial modulation — low-frequency sinusoidal field
      2. Power-law warping — spatially uniform exponent
      3. Smooth additive field — low-frequency sinusoid at unique angle
      4. Spatial shift/warp — smooth coordinate remapping
      5. Frequency modulation — local frequency perturbation
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    d = (dissimilarity - 1) / 9.0  # 0 to 1

    # Generate the parent pattern WITH per-sample variation.
    # Each sample's noise is derived from that sample's own GT pattern.
    val = generate_pattern_values(coords, parent_pattern, width, height,
                                  sample_seed=sample_seed, add_noise=False,
                                  size_scale_range=size_scale_range)

    if d < 0.001:
        # Even at dissimilarity=1, add tiny per-variant uniqueness
        d = 0.01

    # Unique RNG for this parent+variant combination — scalar draws only
    parent_hash = hash(parent_pattern) % 2**31
    variant_seed = (parent_hash * 1000 + variant_idx * 7 + 42) % 2**31
    rng = np.random.RandomState(variant_seed)

    # --- Distortion 1: Multiplicative spatial modulation ---
    mod_freq_x = rng.uniform(800, 2500)
    mod_freq_y = rng.uniform(800, 2500)
    mod_phase  = rng.uniform(0, 2 * np.pi)
    mod_amp    = d * rng.uniform(0.2, 0.5)
    modulation = (np.sin(x / mod_freq_x + mod_phase) *
                  np.cos(y / mod_freq_y + mod_phase * 0.7) + 1) / 2
    val = val * (1 - mod_amp) + val * modulation * mod_amp

    # --- Distortion 2: Power-law warping ---
    exp_direction = rng.choice([-1, 1])
    exponent = 1.0 + d * rng.uniform(0.5, 2.0) * exp_direction
    exponent = np.clip(exponent, 0.3, 3.0)
    val = np.clip(val, 0, 1) ** exponent

    # --- Distortion 3: Smooth additive field ---
    add_freq  = rng.uniform(600, 2000)
    add_theta = rng.uniform(0, 2 * np.pi)
    add_phase = rng.uniform(0, 2 * np.pi)
    add_amp   = d * rng.uniform(0.1, 0.3)
    proj = x * np.cos(add_theta) + y * np.sin(add_theta)
    additive = (np.sin(proj / add_freq + add_phase) + 1) / 2
    val = val + additive * add_amp

    # --- Distortion 4: Spatial coordinate warp (smooth) ---
    # Shifts the effective sampling coordinates via low-frequency sinusoidal warp
    warp_amp_x = d * rng.uniform(50, 300)  # um of shift
    warp_amp_y = d * rng.uniform(50, 300)
    warp_freq_x = rng.uniform(1000, 3000)
    warp_freq_y = rng.uniform(1000, 3000)
    warp_phase_x = rng.uniform(0, 2 * np.pi)
    warp_phase_y = rng.uniform(0, 2 * np.pi)
    # We re-generate the parent at warped coordinates for stronger distortion
    if d > 0.15:
        x_warped = x + warp_amp_x * np.sin(y / warp_freq_y + warp_phase_y)
        y_warped = y + warp_amp_y * np.sin(x / warp_freq_x + warp_phase_x)
        warped_coords = np.column_stack([x_warped, y_warped])
        val_warped = generate_pattern_values(warped_coords, parent_pattern, width, height,
                                              sample_seed=sample_seed, add_noise=False,
                                              size_scale_range=size_scale_range)
        # Blend original and warped based on dissimilarity
        blend = min(d * 1.5, 0.8)
        val = val * (1 - blend) + val_warped * blend

    # --- Distortion 5: Secondary frequency component ---
    sec_freq  = rng.uniform(100, 500)
    sec_theta = rng.uniform(0, 2 * np.pi)
    sec_phase = rng.uniform(0, 2 * np.pi)
    sec_amp   = d * rng.uniform(0.05, 0.2)
    proj2 = x * np.cos(sec_theta) + y * np.sin(sec_theta)
    secondary = (np.sin(proj2 / sec_freq + sec_phase) + 1) / 2
    val = val + secondary * sec_amp

    val = np.clip(val, 0, 1)

    # --- Per-sample variation (same mechanism as GT patterns) ---
    if sample_seed is not None:
        perturb_seed = (variant_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)
        amp_scale     = rng_p.uniform(0.80, 1.05)
        bio_noise_std = rng_p.uniform(0.0, 0.08)

        val = val * amp_scale
        if bio_noise_std > 0:
            rng_bio = np.random.RandomState((variant_seed ^ sample_seed) + 2)
            val = val + rng_bio.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    dy = spacing * np.sqrt(3) / 2
    coords = []; row = 0; y = 0
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        while x < width:
            coords.append([x, y]); x += spacing
        y += dy; row += 1
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    x = np.arange(0, width, spacing); y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================
if __name__ == "__main__":

    n_gt = len(GT_PATTERNS)
    total_noise = NOISE_PER_PATTERN * n_gt
    n_total_features = n_gt + total_noise

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Each GT pattern produces noise variants controlled by dissimilarity")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {n_gt}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Total noise features: {total_noise}")
    print(f"Total features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise Dissimilarity Level: {NOISE_DISSIMILARITY}/10")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} um")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}um spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}um pixels")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}-{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"
            seed = hash(sample_id) % 2**32
            print(f"\n  Processing {sample_id} (seed={seed})...")

            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)

            # === MSI Data ===
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(xi, yi) for xi, yi in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]

            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    MSI GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                msi_data[:, col] = generate_pattern_values(
                    msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_msi.append(f"MZ_{pat}")
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    msi_data[:, col] = generate_noise_from_parent(
                        msi_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        dissimilarity=NOISE_DISSIMILARITY)
                    var_names_msi.append(f"MZ_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]

            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))

            # === RNA Data ===
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(xi, yi) for xi, yi in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]

            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    RNA GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                rna_data[:, col] = generate_pattern_values(
                    rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_rna.append(f"Gene_{pat}")
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    rna_data[:, col] = generate_noise_from_parent(
                        rna_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        dissimilarity=NOISE_DISSIMILARITY)
                    var_names_rna.append(f"Gene_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]

            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print(f"\n{n_gt} GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Noise dissimilarity level: {NOISE_DISSIMILARITY}/10")
    print(f"\nNoise naming: MZ_Noise_<PatternName>_<VariantIdx> / Gene_Noise_<PatternName>_<VariantIdx>")
    print(f"Example: MZ_Noise_Gradient_X_0 is variant 0 of Gradient_X noise")

Generating Ground Truth Dummy Data with 50 PATTERNS
Each GT pattern produces noise variants controlled by dissimilarity
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise per pattern: 10
Total noise features: 500
Total features per sample: 50 GT + 500 Noise = 550
Noise Dissimilarity Level: 5/10
Field size: 6000x6000 um
RNA grid: Visium-like hexagonal, 100um spacing
MSI grid: Cartesian, 60um pixels
Pattern size scale range: 0.90-1.10
Output: dummy_data_50

  Processing YC_1 (seed=2956852261)...
    MSI GT  1/50: Gradient_X + 10 noise variants
    MSI GT  2/50: Gradient_Y + 10 noise variants
    MSI GT  3/50: Gradient_Diagonal_NE + 10 noise variants
    MSI GT  4/50: Gradient_Diagonal_NW + 10 noise variants
    MSI GT  5/50: Gradient_Radial_In + 10 noise variants
    MSI GT  6/50: Gradient_Radial_Out + 10 noise variants
    MSI GT  7/50: Stripes_Vertical + 10 noise variants
    MSI GT  8/50: Stripes_Horizontal + 10 noise variants
    MSI GT  9/50: Stripes_Diagonal_45 + 10 noise var

In [3]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100
MSI_PIXEL_SIZE = 60

PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

# =============================================================================
# NOISE DISSIMILARITY CONTROL
# =============================================================================
# Controls how different noise features are from their parent GT pattern.
# Range: 1 (minimal difference, very close to GT) to 10 (maximal difference).
# Default 2 provides subtle separation.
#
# All distortions use smooth continuous spatial functions so that RNA and MSI
# grids sampling the same underlying field stay perfectly correlated.
NOISE_DISSIMILARITY = 4  # <-- USER-ADJUSTABLE: integer 1-10

# Number of noise variants per GT pattern.
# Total noise features = NOISE_PER_PATTERN * len(GT_PATTERNS)
NOISE_PER_PATTERN = 10

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X", "Gradient_Y", "Gradient_Diagonal_NE", "Gradient_Diagonal_NW",
    "Gradient_Radial_In", "Gradient_Radial_Out",
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical", "Stripes_Horizontal", "Stripes_Diagonal_45",
    "Stripes_Diagonal_135", "Waves_Concentric", "Waves_Spiral",
    "Waves_Interference", "Waves_Ripple",
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center", "Blob_TopRight", "Blob_TopLeft", "Blob_BottomRight",
    "Blob_BottomLeft", "Spots_Grid_Dense", "Spots_Grid_Sparse",
    "Spots_Random_Large", "Spots_Triangular", "Spots_Hexagonal",
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner", "Ring_Outer", "Ring_Double", "Ring_Eccentric",
    "Ring_Elliptical", "Ring_Partial",
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine", "Checkerboard_Coarse", "Quadrant_Alternating",
    "Sectors_4", "Sectors_8", "Triangle_Pattern", "Diamond_Pattern", "Honeycomb",
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers", "Hotspot_Cluster", "Edge_Enhancement", "Core_Shell",
    "Branching", "Laminar_Curved", "Mosaic_Irregular", "Gradient_Sigmoid",
    "Bimodal_Distribution", "Punctate_Dense", "Periventricular", "Asymmetric_Lobe",
]

TOTAL_NOISE_FEATURES = NOISE_PER_PATTERN * len(GT_PATTERNS)

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    np.random.seed(sample_seed)
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height

    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        dx = x - cx; dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        if x_rot < left_edge: return False
        if y_rot < bottom_edge: return False
        if y_rot > top_edge: return False
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        right_edge = left_edge + right_width
        if x_rot > right_edge: return False
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            if x_rel < 0.05: max_y_norm = 1.0
            else: max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            if y_norm > max_y_norm: return False
        return True
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    angle = np.arctan2(y - cy, x - cx)
    pattern_seed = hash(pattern_type) % 2**31

    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height
        phase_offset = rng_s.uniform(-1.0, 1.0)
        freq_jitter = rng_s.uniform(0.75, 1.25)
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])
        amp_scale = rng_s.uniform(0.75, 1.05)
        bio_noise_std = rng_s.uniform(0.0, 0.10)
        cx_s = cx + dx_offset; cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0; freq_jitter = 1.0; size_scale = 1.0
        amp_scale = 1.0; bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center; angle_s = angle

    if pattern_type == "Gradient_X": val = x_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Y": val = y_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Diagonal_NE": val = (x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Diagonal_NW": val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Radial_In": val = 1 - dist_norm + phase_offset * 0.1
    elif pattern_type == "Gradient_Radial_Out": val = dist_norm + phase_offset * 0.1
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
    elif pattern_type == "Waves_Interference":
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset; by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed); sigma = width * 0.1 * size_scale; val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset; by = rng.uniform(y.min(), y.max()) + dy_offset
            val += np.exp(-((x-bx)**2+(y-by)**2) / (2*sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter; spot_sigma = 80 * size_scale; val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter; spot_sigma = 60 * size_scale; val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2*size_scale+phase_offset*0.05); ring_width = width*0.04*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7*size_scale+phase_offset*0.05); ring_width = width*0.05*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Double":
        r1 = max_dist*(0.25*size_scale+phase_offset*0.03); r2 = max_dist*(0.55*size_scale+phase_offset*0.03)
        rw = width*0.03*size_scale
        val = np.exp(-((dist_center_s-r1)**2)/(2*rw**2))+np.exp(-((dist_center_s-r2)**2)/(2*rw**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx+width*0.15+dx_offset; off_cy = cy+height*0.1+dy_offset
        dist_off = np.sqrt((x-off_cx)**2+(y-off_cy)**2)
        target_r = max_dist*(0.35*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        val = np.exp(-((dist_off-target_r)**2)/(2*rw**2))
    elif pattern_type == "Ring_Elliptical":
        a = max_dist*(0.5*size_scale+phase_offset*0.05); b = max_dist*(0.3*size_scale+phase_offset*0.03)
        ed = np.sqrt(((x-cx_s)/a)**2+((y-cy_s)/b)**2); rw = 0.1*size_scale
        val = np.exp(-((ed-1)**2)/(2*rw**2))
    elif pattern_type == "Ring_Partial":
        target_r = max_dist*(0.4*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        rv = np.exp(-((dist_center_s-target_r)**2)/(2*rw**2))
        ah = (np.pi/2)*size_scale; a_s = -ah+phase_offset*0.3; a_e = ah+phase_offset*0.3
        val = rv * ((angle_s > a_s) & (angle_s < a_e))
    elif pattern_type == "Checkerboard_Fine":
        ss = 200*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Checkerboard_Coarse":
        ss = 500*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Quadrant_Alternating":
        val = ((x>cx_s)&(y>cy_s) | (x<cx_s)&(y<cy_s)).astype(float)
    elif pattern_type == "Sectors_4":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/2)).astype(int)%4; val = (sector%2).astype(float)
    elif pattern_type == "Sectors_8":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/4)).astype(int)%8; val = (sector%2).astype(float)
    elif pattern_type == "Triangle_Pattern":
        period = 300*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)/(period/2)+np.abs(((y+dy_offset)%period)-period/2)/(period/2))/2
    elif pattern_type == "Diamond_Pattern":
        period = 400*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)+np.abs(((y+dy_offset)%period)-period/2))/period
    elif pattern_type == "Honeycomb":
        spacing = 200*freq_jitter; val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i*spacing+(j%2)*(spacing/2)+dx_offset; hy = j*spacing*0.866+dy_offset
                dist = np.sqrt((x-hx)**2+(y-hy)**2)
                val = np.maximum(val, 1-np.clip(dist/(spacing*0.4*size_scale), 0, 1))
        val = 1-val
    elif pattern_type == "Cortical_Layers":
        n_layers = 5; layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        sd = np.sqrt((x-cx_s)**2+(y-cy_s)**2); sn = sd/(sd.max()+1e-8)
        sc = np.clip(sn/size_scale, 0, 1); li = np.clip((sc*n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in li])
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed); sigma = width*0.05*size_scale; val = np.zeros(n)
        ccx = cx+rng.uniform(-width*0.1, width*0.1)+dx_offset; ccy = cy+rng.uniform(-height*0.1, height*0.1)+dy_offset
        for _ in range(8):
            bx = ccx+rng.normal(0, width*0.08); by = ccy+rng.normal(0, height*0.08)
            val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sigma**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Edge_Enhancement":
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        target = np.clip(0.8*size_scale+phase_offset*0.05, 0.3, 1.2); bw = 0.15*size_scale
        val = np.clip(1-np.exp(-((sn-target)**2)/(2*bw**2)), 0, 1)
    elif pattern_type == "Core_Shell":
        cs = width*0.1*size_scale; sr = max_dist*(0.5+phase_offset*0.05)*size_scale; ss = width*0.08*size_scale
        val = 0.7*np.exp(-dist_center_s**2/(2*cs**2))+0.5*np.exp(-((dist_center_s-sr)**2)/(2*ss**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Branching":
        nb = 5; ba = 2*np.pi/nb; bw = 0.3*size_scale; ra = angle_s+phase_offset*0.5
        mdb = np.inf*np.ones(n)
        for i in range(nb):
            ta = i*ba; ad2 = np.abs(np.mod(ra-ta+np.pi, 2*np.pi)-np.pi)
            mdb = np.minimum(mdb, ad2)
        val = np.exp(-mdb**2/(2*bw**2))*(1-np.exp(-dist_norm*3))
    elif pattern_type == "Laminar_Curved":
        ca = 200*(1+phase_offset*0.3); cf = 400*freq_jitter; bt = 300*size_scale
        cy2 = (y+dy_offset)-ca*np.sin((x+dx_offset)/cf)
        val = ((cy2//bt).astype(int)%2).astype(float)
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed); ns = max(5, int(20/size_scale))
        sx = rng.uniform(x.min(), x.max(), ns)+dx_offset; sy = rng.uniform(y.min(), y.max(), ns)+dy_offset
        sv = rng.rand(ns); ni = np.zeros(n, dtype=int); md = np.inf*np.ones(n)
        for i in range(ns):
            d = np.sqrt((x-sx[i])**2+(y-sy[i])**2); c = d<md; ni[c] = i; md[c] = d[c]
        val = sv[ni]
    elif pattern_type == "Gradient_Sigmoid":
        shift = 0.5+phase_offset*0.1; steep = 10.0/size_scale
        val = 1/(1+np.exp(-steep*(x_norm-shift)))
    elif pattern_type == "Bimodal_Distribution":
        val = (x<cx_s).astype(float)*0.8+(x>=cx_s).astype(float)*0.3
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed); ps = 40*size_scale; val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())+dx_offset; py = rng.uniform(y.min(), y.max())+dy_offset
            val += np.exp(-((x-px)**2+(y-py)**2)/(2*ps**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Periventricular":
        vr = 0.15*size_scale+phase_offset*0.02; rr = 0.25*size_scale+phase_offset*0.02; rw = 0.08*size_scale
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        val = np.exp(-((sn-rr)**2)/(2*rw**2))*(sn>=vr)
    elif pattern_type == "Asymmetric_Lobe":
        l1x = cx-width*0.2+dx_offset; l1y = cy+dy_offset; l2x = cx+width*0.15+dx_offset; l2y = cy+height*0.1+dy_offset
        s1 = width*0.15*size_scale; s2 = width*0.10*size_scale
        val = 0.7*np.exp(-((x-l1x)**2+(y-l1y)**2)/(2*s1**2))+0.5*np.exp(-((x-l2x)**2+(y-l2y)**2)/(2*s2**2))
        val = val/(val.max()+1e-8)
    else:
        rng = np.random.RandomState(pattern_seed); val = rng.rand(n)

    val = val * amp_scale
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)
    return np.clip(val, 0, 1)


def generate_noise_from_parent(coords, parent_pattern, variant_idx,
                               width=3000, height=3000,
                               sample_seed=None, size_scale_range=(0.70, 1.40),
                               dissimilarity=2):
    """
    Generate a noise feature as a spatially-distorted variant of a parent GT pattern.

    Each noise feature re-generates the parent pattern with MODIFIED spatial
    parameters (shifted center, different frequency, rotated angle, warped
    coordinates) so the resulting pattern is structurally different — not just
    a color/intensity change.

    dissimilarity (1-10) controls the magnitude of spatial distortions.
    Even at dissimilarity=1, each variant is visibly distinct from the parent.

    variant_idx (0..NOISE_PER_PATTERN-1) determines which unique set of
    spatial perturbations is applied.

    CRITICAL INVARIANT: All distortions use smooth continuous spatial functions
    f(x, y). The RNG draws only scalar parameters (never n-dependent arrays).
    This guarantees identical distortion on RNA and MSI grids.

    Distortion mechanisms (ALL always active, magnitude scaled by dissimilarity):
      1. Coordinate warp — sinusoidal spatial remapping (changes where features appear)
      2. Center shift — moves the effective center of the pattern
      3. Frequency/scale perturbation — changes feature sizes and spacings
      4. Rotation — rotates the effective coordinate frame
      5. Additive spatial field — adds a unique low-frequency spatial signal
      6. Intensity modulation — spatially-varying brightness change
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]

    # d controls distortion magnitude: 0.05 (dissimilarity=1) to 1.0 (dissimilarity=10)
    # We use a minimum floor of 0.05 so even dissimilarity=1 produces visible changes
    d = max(0.05, (dissimilarity - 1) / 9.0)

    # Unique RNG for this parent+variant combination — scalar draws only
    parent_hash = hash(parent_pattern) % 2**31
    variant_seed = (parent_hash * 1000 + variant_idx * 7 + 42) % 2**31
    rng = np.random.RandomState(variant_seed)

    # =====================================================================
    # DISTORTION 1: Coordinate warp (ALWAYS active)
    # Sinusoidal spatial remapping — physically moves pattern features.
    # This is the primary mechanism that changes spatial structure.
    # =====================================================================
    # Base warp: 50-150 um even at lowest dissimilarity, up to 600 um at max
    warp_base = 50 + d * 550  # um of maximum displacement
    warp_amp_x = rng.uniform(0.5, 1.0) * warp_base
    warp_amp_y = rng.uniform(0.5, 1.0) * warp_base
    warp_freq_x = rng.uniform(1500, 4000)
    warp_freq_y = rng.uniform(1500, 4000)
    warp_phase_x = rng.uniform(0, 2 * np.pi)
    warp_phase_y = rng.uniform(0, 2 * np.pi)

    x_warped = x + warp_amp_x * np.sin(y / warp_freq_y + warp_phase_y)
    y_warped = y + warp_amp_y * np.sin(x / warp_freq_x + warp_phase_x)

    # =====================================================================
    # DISTORTION 2: Center shift
    # Moves the effective pattern center, causing features to appear in
    # different locations.
    # =====================================================================
    # 3-12% of field size depending on dissimilarity
    shift_mag = (0.03 + d * 0.09)
    center_shift_x = rng.uniform(-shift_mag, shift_mag) * width
    center_shift_y = rng.uniform(-shift_mag, shift_mag) * height

    x_warped = x_warped + center_shift_x
    y_warped = y_warped + center_shift_y

    # =====================================================================
    # DISTORTION 3: Frequency/scale perturbation
    # Applies smooth spatial scaling — stretches or compresses features.
    # Implemented as coordinate scaling around the field center.
    # =====================================================================
    # 3-20% frequency change depending on dissimilarity
    freq_change = 1.0 + rng.uniform(-1, 1) * (0.03 + d * 0.17)
    cx_field = width / 2.0
    cy_field = height / 2.0

    x_warped = cx_field + (x_warped - cx_field) * freq_change
    y_warped = cy_field + (y_warped - cy_field) * freq_change

    # =====================================================================
    # DISTORTION 4: Rotation
    # Rotates the coordinate frame — shifts angular features and tilts
    # directional patterns.
    # =====================================================================
    # 3-25 degrees depending on dissimilarity
    rot_angle = rng.uniform(-1, 1) * (0.05 + d * 0.39)  # radians
    cos_r, sin_r = np.cos(rot_angle), np.sin(rot_angle)
    x_rot = cx_field + (x_warped - cx_field) * cos_r - (y_warped - cy_field) * sin_r
    y_rot = cy_field + (x_warped - cx_field) * sin_r + (y_warped - cy_field) * cos_r

    # =====================================================================
    # Generate parent pattern at the distorted coordinates
    # This is the KEY difference: pattern is evaluated at warped positions,
    # producing actual spatial structure changes.
    # =====================================================================
    warped_coords = np.column_stack([x_rot, y_rot])
    val = generate_pattern_values(warped_coords, parent_pattern, width, height,
                                  sample_seed=sample_seed, add_noise=False,
                                  size_scale_range=size_scale_range)

    # =====================================================================
    # DISTORTION 5: Additive spatial field
    # Adds a unique low-frequency signal that modifies the pattern's
    # spatial distribution (not just intensity).
    # =====================================================================
    add_freq  = rng.uniform(400, 1500)
    add_theta = rng.uniform(0, 2 * np.pi)
    add_phase = rng.uniform(0, 2 * np.pi)
    # 5-25% additive strength
    add_amp   = 0.05 + d * 0.20
    proj = x * np.cos(add_theta) + y * np.sin(add_theta)
    additive = (np.sin(proj / add_freq + add_phase) + 1) / 2
    val = val * (1 - add_amp * 0.5) + additive * add_amp * 0.5

    # =====================================================================
    # DISTORTION 6: Spatially-varying intensity modulation
    # A smooth 2D field that locally brightens/darkens the pattern.
    # =====================================================================
    mod_freq_x = rng.uniform(600, 2000)
    mod_freq_y = rng.uniform(600, 2000)
    mod_phase  = rng.uniform(0, 2 * np.pi)
    # 5-30% modulation depth
    mod_amp    = 0.05 + d * 0.25
    modulation = (np.sin(x / mod_freq_x + mod_phase) *
                  np.cos(y / mod_freq_y + mod_phase * 0.7) + 1) / 2
    val = val * (1 - mod_amp) + val * modulation * mod_amp * 2

    val = np.clip(val, 0, 1)

    # =====================================================================
    # Per-sample variation (same mechanism as GT patterns)
    # =====================================================================
    if sample_seed is not None:
        perturb_seed = (variant_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)
        amp_scale     = rng_p.uniform(0.85, 1.05)
        bio_noise_std = rng_p.uniform(0.0, 0.06)

        val = val * amp_scale
        if bio_noise_std > 0:
            rng_bio = np.random.RandomState((variant_seed ^ sample_seed) + 2)
            val = val + rng_bio.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    dy = spacing * np.sqrt(3) / 2
    coords = []; row = 0; y = 0
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        while x < width:
            coords.append([x, y]); x += spacing
        y += dy; row += 1
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    x = np.arange(0, width, spacing); y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================
if __name__ == "__main__":

    n_gt = len(GT_PATTERNS)
    total_noise = NOISE_PER_PATTERN * n_gt
    n_total_features = n_gt + total_noise

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Each GT pattern produces noise variants controlled by dissimilarity")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {n_gt}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Total noise features: {total_noise}")
    print(f"Total features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise Dissimilarity Level: {NOISE_DISSIMILARITY}/10")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} um")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}um spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}um pixels")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}-{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"
            seed = hash(sample_id) % 2**32
            print(f"\n  Processing {sample_id} (seed={seed})...")

            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)

            # === MSI Data ===
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(xi, yi) for xi, yi in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]

            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    MSI GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                msi_data[:, col] = generate_pattern_values(
                    msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_msi.append(f"MZ_{pat}")
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    msi_data[:, col] = generate_noise_from_parent(
                        msi_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        dissimilarity=NOISE_DISSIMILARITY)
                    var_names_msi.append(f"MZ_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]

            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))

            # === RNA Data ===
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(xi, yi) for xi, yi in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]

            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    RNA GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                rna_data[:, col] = generate_pattern_values(
                    rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_rna.append(f"Gene_{pat}")
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    rna_data[:, col] = generate_noise_from_parent(
                        rna_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        dissimilarity=NOISE_DISSIMILARITY)
                    var_names_rna.append(f"Gene_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]

            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print(f"\n{n_gt} GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Noise dissimilarity level: {NOISE_DISSIMILARITY}/10")
    print(f"\nNoise naming: MZ_Noise_<PatternName>_<VariantIdx> / Gene_Noise_<PatternName>_<VariantIdx>")
    print(f"Example: MZ_Noise_Gradient_X_0 is variant 0 of Gradient_X noise")

Generating Ground Truth Dummy Data with 50 PATTERNS
Each GT pattern produces noise variants controlled by dissimilarity
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise per pattern: 10
Total noise features: 500
Total features per sample: 50 GT + 500 Noise = 550
Noise Dissimilarity Level: 4/10
Field size: 6000x6000 um
RNA grid: Visium-like hexagonal, 100um spacing
MSI grid: Cartesian, 60um pixels
Pattern size scale range: 0.90-1.10
Output: dummy_data_50

  Processing YC_1 (seed=2956852261)...
    MSI GT  1/50: Gradient_X + 10 noise variants
    MSI GT  2/50: Gradient_Y + 10 noise variants
    MSI GT  3/50: Gradient_Diagonal_NE + 10 noise variants
    MSI GT  4/50: Gradient_Diagonal_NW + 10 noise variants
    MSI GT  5/50: Gradient_Radial_In + 10 noise variants
    MSI GT  6/50: Gradient_Radial_Out + 10 noise variants
    MSI GT  7/50: Stripes_Vertical + 10 noise variants
    MSI GT  8/50: Stripes_Horizontal + 10 noise variants
    MSI GT  9/50: Stripes_Diagonal_45 + 10 noise var

In [4]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100
MSI_PIXEL_SIZE = 60

PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

# =============================================================================
# NOISE DISTORTION CONTROLS (5 independent parameters, each 1-10)
# =============================================================================
# Each parameter controls how much one spatial property of noise variants
# deviates from the parent GT pattern.
#   1 = minimal deviation (noise very close to parent)
#  10 = maximal deviation (noise substantially different from parent)
#
# All distortions use smooth continuous spatial functions so that RNA and MSI
# grids sampling the same underlying field stay perfectly correlated.
NOISE_SHAPE       = 8   # Sinusoidal coordinate warp — deforms the spatial structure
NOISE_LOCATION    = 8   # Center shift — translates where features appear
NOISE_SCALE       = 8   # Size/frequency change — stretches or compresses features
NOISE_FREQUENCY   = 8   # Additive spatial frequency field — overlays new spatial signal
NOISE_ORIENTATION = 8   # Rotation of the coordinate frame

# Number of noise variants per GT pattern.
# Total noise features = NOISE_PER_PATTERN * len(GT_PATTERNS)
NOISE_PER_PATTERN = 10

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X", "Gradient_Y", "Gradient_Diagonal_NE", "Gradient_Diagonal_NW",
    "Gradient_Radial_In", "Gradient_Radial_Out",
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical", "Stripes_Horizontal", "Stripes_Diagonal_45",
    "Stripes_Diagonal_135", "Waves_Concentric", "Waves_Spiral",
    "Waves_Interference", "Waves_Ripple",
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center", "Blob_TopRight", "Blob_TopLeft", "Blob_BottomRight",
    "Blob_BottomLeft", "Spots_Grid_Dense", "Spots_Grid_Sparse",
    "Spots_Random_Large", "Spots_Triangular", "Spots_Hexagonal",
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner", "Ring_Outer", "Ring_Double", "Ring_Eccentric",
    "Ring_Elliptical", "Ring_Partial",
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine", "Checkerboard_Coarse", "Quadrant_Alternating",
    "Sectors_4", "Sectors_8", "Triangle_Pattern", "Diamond_Pattern", "Honeycomb",
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers", "Hotspot_Cluster", "Edge_Enhancement", "Core_Shell",
    "Branching", "Laminar_Curved", "Mosaic_Irregular", "Gradient_Sigmoid",
    "Bimodal_Distribution", "Punctate_Dense", "Periventricular", "Asymmetric_Lobe",
]

TOTAL_NOISE_FEATURES = NOISE_PER_PATTERN * len(GT_PATTERNS)

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    np.random.seed(sample_seed)
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height

    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        dx = x - cx; dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        if x_rot < left_edge: return False
        if y_rot < bottom_edge: return False
        if y_rot > top_edge: return False
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        right_edge = left_edge + right_width
        if x_rot > right_edge: return False
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            if x_rel < 0.05: max_y_norm = 1.0
            else: max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            if y_norm > max_y_norm: return False
        return True
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    angle = np.arctan2(y - cy, x - cx)
    pattern_seed = hash(pattern_type) % 2**31

    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height
        phase_offset = rng_s.uniform(-1.0, 1.0)
        freq_jitter = rng_s.uniform(0.75, 1.25)
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])
        amp_scale = rng_s.uniform(0.75, 1.05)
        bio_noise_std = rng_s.uniform(0.0, 0.10)
        cx_s = cx + dx_offset; cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0; freq_jitter = 1.0; size_scale = 1.0
        amp_scale = 1.0; bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center; angle_s = angle

    if pattern_type == "Gradient_X": val = x_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Y": val = y_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Diagonal_NE": val = (x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Diagonal_NW": val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Radial_In": val = 1 - dist_norm + phase_offset * 0.1
    elif pattern_type == "Gradient_Radial_Out": val = dist_norm + phase_offset * 0.1
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
    elif pattern_type == "Waves_Interference":
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset; by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed); sigma = width * 0.1 * size_scale; val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset; by = rng.uniform(y.min(), y.max()) + dy_offset
            val += np.exp(-((x-bx)**2+(y-by)**2) / (2*sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter; spot_sigma = 80 * size_scale; val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter; spot_sigma = 60 * size_scale; val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2*size_scale+phase_offset*0.05); ring_width = width*0.04*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7*size_scale+phase_offset*0.05); ring_width = width*0.05*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Double":
        r1 = max_dist*(0.25*size_scale+phase_offset*0.03); r2 = max_dist*(0.55*size_scale+phase_offset*0.03)
        rw = width*0.03*size_scale
        val = np.exp(-((dist_center_s-r1)**2)/(2*rw**2))+np.exp(-((dist_center_s-r2)**2)/(2*rw**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx+width*0.15+dx_offset; off_cy = cy+height*0.1+dy_offset
        dist_off = np.sqrt((x-off_cx)**2+(y-off_cy)**2)
        target_r = max_dist*(0.35*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        val = np.exp(-((dist_off-target_r)**2)/(2*rw**2))
    elif pattern_type == "Ring_Elliptical":
        a = max_dist*(0.5*size_scale+phase_offset*0.05); b = max_dist*(0.3*size_scale+phase_offset*0.03)
        ed = np.sqrt(((x-cx_s)/a)**2+((y-cy_s)/b)**2); rw = 0.1*size_scale
        val = np.exp(-((ed-1)**2)/(2*rw**2))
    elif pattern_type == "Ring_Partial":
        target_r = max_dist*(0.4*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        rv = np.exp(-((dist_center_s-target_r)**2)/(2*rw**2))
        ah = (np.pi/2)*size_scale; a_s = -ah+phase_offset*0.3; a_e = ah+phase_offset*0.3
        val = rv * ((angle_s > a_s) & (angle_s < a_e))
    elif pattern_type == "Checkerboard_Fine":
        ss = 200*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Checkerboard_Coarse":
        ss = 500*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Quadrant_Alternating":
        val = ((x>cx_s)&(y>cy_s) | (x<cx_s)&(y<cy_s)).astype(float)
    elif pattern_type == "Sectors_4":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/2)).astype(int)%4; val = (sector%2).astype(float)
    elif pattern_type == "Sectors_8":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/4)).astype(int)%8; val = (sector%2).astype(float)
    elif pattern_type == "Triangle_Pattern":
        period = 300*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)/(period/2)+np.abs(((y+dy_offset)%period)-period/2)/(period/2))/2
    elif pattern_type == "Diamond_Pattern":
        period = 400*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)+np.abs(((y+dy_offset)%period)-period/2))/period
    elif pattern_type == "Honeycomb":
        spacing = 200*freq_jitter; val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i*spacing+(j%2)*(spacing/2)+dx_offset; hy = j*spacing*0.866+dy_offset
                dist = np.sqrt((x-hx)**2+(y-hy)**2)
                val = np.maximum(val, 1-np.clip(dist/(spacing*0.4*size_scale), 0, 1))
        val = 1-val
    elif pattern_type == "Cortical_Layers":
        n_layers = 5; layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        sd = np.sqrt((x-cx_s)**2+(y-cy_s)**2); sn = sd/(sd.max()+1e-8)
        sc = np.clip(sn/size_scale, 0, 1); li = np.clip((sc*n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in li])
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed); sigma = width*0.05*size_scale; val = np.zeros(n)
        ccx = cx+rng.uniform(-width*0.1, width*0.1)+dx_offset; ccy = cy+rng.uniform(-height*0.1, height*0.1)+dy_offset
        for _ in range(8):
            bx = ccx+rng.normal(0, width*0.08); by = ccy+rng.normal(0, height*0.08)
            val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sigma**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Edge_Enhancement":
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        target = np.clip(0.8*size_scale+phase_offset*0.05, 0.3, 1.2); bw = 0.15*size_scale
        val = np.clip(1-np.exp(-((sn-target)**2)/(2*bw**2)), 0, 1)
    elif pattern_type == "Core_Shell":
        cs = width*0.1*size_scale; sr = max_dist*(0.5+phase_offset*0.05)*size_scale; ss = width*0.08*size_scale
        val = 0.7*np.exp(-dist_center_s**2/(2*cs**2))+0.5*np.exp(-((dist_center_s-sr)**2)/(2*ss**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Branching":
        nb = 5; ba = 2*np.pi/nb; bw = 0.3*size_scale; ra = angle_s+phase_offset*0.5
        mdb = np.inf*np.ones(n)
        for i in range(nb):
            ta = i*ba; ad2 = np.abs(np.mod(ra-ta+np.pi, 2*np.pi)-np.pi)
            mdb = np.minimum(mdb, ad2)
        val = np.exp(-mdb**2/(2*bw**2))*(1-np.exp(-dist_norm*3))
    elif pattern_type == "Laminar_Curved":
        ca = 200*(1+phase_offset*0.3); cf = 400*freq_jitter; bt = 300*size_scale
        cy2 = (y+dy_offset)-ca*np.sin((x+dx_offset)/cf)
        val = ((cy2//bt).astype(int)%2).astype(float)
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed); ns = max(5, int(20/size_scale))
        sx = rng.uniform(x.min(), x.max(), ns)+dx_offset; sy = rng.uniform(y.min(), y.max(), ns)+dy_offset
        sv = rng.rand(ns); ni = np.zeros(n, dtype=int); md = np.inf*np.ones(n)
        for i in range(ns):
            d = np.sqrt((x-sx[i])**2+(y-sy[i])**2); c = d<md; ni[c] = i; md[c] = d[c]
        val = sv[ni]
    elif pattern_type == "Gradient_Sigmoid":
        shift = 0.5+phase_offset*0.1; steep = 10.0/size_scale
        val = 1/(1+np.exp(-steep*(x_norm-shift)))
    elif pattern_type == "Bimodal_Distribution":
        val = (x<cx_s).astype(float)*0.8+(x>=cx_s).astype(float)*0.3
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed); ps = 40*size_scale; val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())+dx_offset; py = rng.uniform(y.min(), y.max())+dy_offset
            val += np.exp(-((x-px)**2+(y-py)**2)/(2*ps**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Periventricular":
        vr = 0.15*size_scale+phase_offset*0.02; rr = 0.25*size_scale+phase_offset*0.02; rw = 0.08*size_scale
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        val = np.exp(-((sn-rr)**2)/(2*rw**2))*(sn>=vr)
    elif pattern_type == "Asymmetric_Lobe":
        l1x = cx-width*0.2+dx_offset; l1y = cy+dy_offset; l2x = cx+width*0.15+dx_offset; l2y = cy+height*0.1+dy_offset
        s1 = width*0.15*size_scale; s2 = width*0.10*size_scale
        val = 0.7*np.exp(-((x-l1x)**2+(y-l1y)**2)/(2*s1**2))+0.5*np.exp(-((x-l2x)**2+(y-l2y)**2)/(2*s2**2))
        val = val/(val.max()+1e-8)
    else:
        rng = np.random.RandomState(pattern_seed); val = rng.rand(n)

    val = val * amp_scale
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)
    return np.clip(val, 0, 1)


def generate_noise_from_parent(coords, parent_pattern, variant_idx,
                               width=3000, height=3000,
                               sample_seed=None, size_scale_range=(0.70, 1.40),
                               shape=2, location=2, scale=2,
                               frequency=2, orientation=2):
    """
    Generate a noise feature as a spatially-distorted variant of a parent GT pattern.

    Five independent parameters (each 1-10) control distinct spatial properties:

      shape       — Sinusoidal coordinate warp that deforms the spatial structure.
                    Physically displaces features via smooth remapping.
      location    — Translates the effective pattern center so features appear
                    at shifted positions.
      scale       — Stretches or compresses the pattern by rescaling coordinates
                    around the field center (changes feature sizes/spacings).
      frequency   — Overlays a unique low-frequency additive spatial signal,
                    changing the pattern's spatial frequency content.
      orientation — Rotates the coordinate frame, tilting directional patterns
                    and shifting angular features.

    variant_idx (0..NOISE_PER_PATTERN-1) determines which unique set of
    perturbations is applied.  Different variant_idx values produce different
    random draws for each distortion axis.

    CRITICAL INVARIANT: All distortions use smooth continuous spatial functions
    f(x, y). The RNG draws only scalar parameters (never n-dependent arrays).
    This guarantees identical results on RNA and MSI grids.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]

    # Convert 1-10 integer controls to 0-1 continuous magnitudes.
    # Floor of 0.05 ensures even level=1 produces a visible (though small) change.
    d_shape  = max(0.05, (shape - 1) / 9.0)
    d_loc    = max(0.05, (location - 1) / 9.0)
    d_scale  = max(0.05, (scale - 1) / 9.0)
    d_freq   = max(0.05, (frequency - 1) / 9.0)
    d_orient = max(0.05, (orientation - 1) / 9.0)

    # Unique RNG for this parent+variant combination — scalar draws only.
    parent_hash = hash(parent_pattern) % 2**31
    variant_seed = (parent_hash * 1000 + variant_idx * 7 + 42) % 2**31
    rng = np.random.RandomState(variant_seed)

    cx_field = width / 2.0
    cy_field = height / 2.0

    # =====================================================================
    # 1. SHAPE — Sinusoidal coordinate warp
    #    Displaces each (x, y) by a smooth sinusoidal field.
    #    Level 1 ≈ 50 µm max shift;  Level 10 ≈ 600 µm max shift.
    # =====================================================================
    warp_base = 50 + d_shape * 550  # µm of maximum displacement
    warp_amp_x  = rng.uniform(0.5, 1.0) * warp_base
    warp_amp_y  = rng.uniform(0.5, 1.0) * warp_base
    warp_freq_x = rng.uniform(1500, 4000)
    warp_freq_y = rng.uniform(1500, 4000)
    warp_ph_x   = rng.uniform(0, 2 * np.pi)
    warp_ph_y   = rng.uniform(0, 2 * np.pi)

    x_w = x + warp_amp_x * np.sin(y / warp_freq_y + warp_ph_y)
    y_w = y + warp_amp_y * np.sin(x / warp_freq_x + warp_ph_x)

    # =====================================================================
    # 2. LOCATION — Center shift / translation
    #    Moves the effective origin of the pattern.
    #    Level 1 ≈ ±3 % of field;  Level 10 ≈ ±12 % of field.
    # =====================================================================
    shift_mag = 0.03 + d_loc * 0.09
    shift_x = rng.uniform(-shift_mag, shift_mag) * width
    shift_y = rng.uniform(-shift_mag, shift_mag) * height

    x_w = x_w + shift_x
    y_w = y_w + shift_y

    # =====================================================================
    # 3. SCALE — Coordinate rescaling around field center
    #    Stretches or compresses features uniformly.
    #    Level 1 ≈ ±3 % change;  Level 10 ≈ ±20 % change.
    # =====================================================================
    scale_factor = 1.0 + rng.uniform(-1, 1) * (0.03 + d_scale * 0.17)

    x_w = cx_field + (x_w - cx_field) * scale_factor
    y_w = cy_field + (y_w - cy_field) * scale_factor

    # =====================================================================
    # 4. ORIENTATION — Rotation around field center
    #    Tilts directional patterns and shifts angular features.
    #    Level 1 ≈ ±3°;  Level 10 ≈ ±25°.
    # =====================================================================
    rot_angle = rng.uniform(-1, 1) * (0.05 + d_orient * 0.39)  # radians
    cos_r, sin_r = np.cos(rot_angle), np.sin(rot_angle)
    dx_c = x_w - cx_field
    dy_c = y_w - cy_field
    x_r = cx_field + dx_c * cos_r - dy_c * sin_r
    y_r = cy_field + dx_c * sin_r + dy_c * cos_r

    # =====================================================================
    # Generate parent pattern at the distorted coordinates.
    # The pattern is evaluated at warped positions, producing actual
    # spatial structure changes (not just intensity/color shifts).
    # =====================================================================
    warped_coords = np.column_stack([x_r, y_r])
    val = generate_pattern_values(warped_coords, parent_pattern, width, height,
                                  sample_seed=sample_seed, add_noise=False,
                                  size_scale_range=size_scale_range)

    # =====================================================================
    # 5. FREQUENCY — Additive low-frequency spatial signal
    #    Overlays a unique sinusoidal field that changes the spatial
    #    frequency content of the pattern.
    #    Level 1 ≈ 5 % amplitude;  Level 10 ≈ 25 % amplitude.
    # =====================================================================
    add_freq  = rng.uniform(400, 1500)
    add_theta = rng.uniform(0, 2 * np.pi)
    add_phase = rng.uniform(0, 2 * np.pi)
    add_amp   = 0.05 + d_freq * 0.20

    proj = x * np.cos(add_theta) + y * np.sin(add_theta)
    additive = (np.sin(proj / add_freq + add_phase) + 1) / 2
    val = val * (1 - add_amp * 0.5) + additive * add_amp * 0.5

    val = np.clip(val, 0, 1)

    # =====================================================================
    # Per-sample variation (same mechanism as GT patterns)
    # =====================================================================
    if sample_seed is not None:
        perturb_seed = (variant_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)
        amp_scale     = rng_p.uniform(0.85, 1.05)
        bio_noise_std = rng_p.uniform(0.0, 0.06)

        val = val * amp_scale
        if bio_noise_std > 0:
            rng_bio = np.random.RandomState((variant_seed ^ sample_seed) + 2)
            val = val + rng_bio.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    dy = spacing * np.sqrt(3) / 2
    coords = []; row = 0; y = 0
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        while x < width:
            coords.append([x, y]); x += spacing
        y += dy; row += 1
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    x = np.arange(0, width, spacing); y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================
if __name__ == "__main__":

    n_gt = len(GT_PATTERNS)
    total_noise = NOISE_PER_PATTERN * n_gt
    n_total_features = n_gt + total_noise

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Each GT pattern produces noise variants with 5 distortion controls")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {n_gt}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Total noise features: {total_noise}")
    print(f"Total features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise distortion controls (1-10):")
    print(f"  Shape:       {NOISE_SHAPE:2d}  (coordinate warp)")
    print(f"  Location:    {NOISE_LOCATION:2d}  (center shift)")
    print(f"  Scale:       {NOISE_SCALE:2d}  (size/frequency change)")
    print(f"  Frequency:   {NOISE_FREQUENCY:2d}  (additive spatial signal)")
    print(f"  Orientation: {NOISE_ORIENTATION:2d}  (rotation)")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} um")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}um spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}um pixels")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}-{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"
            seed = hash(sample_id) % 2**32
            print(f"\n  Processing {sample_id} (seed={seed})...")

            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)

            # === MSI Data ===
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(xi, yi) for xi, yi in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]

            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    MSI GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                msi_data[:, col] = generate_pattern_values(
                    msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_msi.append(f"MZ_{pat}")
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    msi_data[:, col] = generate_noise_from_parent(
                        msi_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        shape=NOISE_SHAPE, location=NOISE_LOCATION,
                        scale=NOISE_SCALE, frequency=NOISE_FREQUENCY,
                        orientation=NOISE_ORIENTATION)
                    var_names_msi.append(f"MZ_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]

            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))

            # === RNA Data ===
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(xi, yi) for xi, yi in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]

            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    RNA GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                rna_data[:, col] = generate_pattern_values(
                    rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_rna.append(f"Gene_{pat}")
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    rna_data[:, col] = generate_noise_from_parent(
                        rna_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        shape=NOISE_SHAPE, location=NOISE_LOCATION,
                        scale=NOISE_SCALE, frequency=NOISE_FREQUENCY,
                        orientation=NOISE_ORIENTATION)
                    var_names_rna.append(f"Gene_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]

            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print(f"\n{n_gt} GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Noise distortion controls: shape={NOISE_SHAPE}, location={NOISE_LOCATION}, "
          f"scale={NOISE_SCALE}, frequency={NOISE_FREQUENCY}, orientation={NOISE_ORIENTATION}")
    print(f"\nNoise naming: MZ_Noise_<PatternName>_<VariantIdx> / Gene_Noise_<PatternName>_<VariantIdx>")
    print(f"Example: MZ_Noise_Gradient_X_0 is variant 0 of Gradient_X noise")

Generating Ground Truth Dummy Data with 50 PATTERNS
Each GT pattern produces noise variants with 5 distortion controls
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise per pattern: 10
Total noise features: 500
Total features per sample: 50 GT + 500 Noise = 550
Noise distortion controls (1-10):
  Shape:        8  (coordinate warp)
  Location:     8  (center shift)
  Scale:        8  (size/frequency change)
  Frequency:    8  (additive spatial signal)
  Orientation:  8  (rotation)
Field size: 6000x6000 um
RNA grid: Visium-like hexagonal, 100um spacing
MSI grid: Cartesian, 60um pixels
Pattern size scale range: 0.90-1.10
Output: dummy_data_50

  Processing YC_1 (seed=4166224011)...
    MSI GT  1/50: Gradient_X + 10 noise variants
    MSI GT  2/50: Gradient_Y + 10 noise variants
    MSI GT  3/50: Gradient_Diagonal_NE + 10 noise variants
    MSI GT  4/50: Gradient_Diagonal_NW + 10 noise variants
    MSI GT  5/50: Gradient_Radial_In + 10 noise variants
    MSI GT  6/50: Gradient_Radia

In [5]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100
MSI_PIXEL_SIZE = 60

PATTERN_SIZE_SCALE_RANGE = (0.9, 1.1)

# =============================================================================
# NOISE DISTORTION CONTROLS (5 independent parameters, each 1-20)
# =============================================================================
# Each parameter controls how much one spatial property of noise variants
# deviates from the parent GT pattern.
#   1  = minimal deviation (noise very close to parent)
#  10  = moderate deviation
#  20  = extreme deviation (noise barely recognisable as the parent)
#
# All distortions use smooth continuous spatial functions so that RNA and MSI
# grids sampling the same underlying field stay perfectly correlated.
NOISE_SHAPE       = 8   # Sinusoidal coordinate warp — deforms the spatial structure
NOISE_LOCATION    = 8   # Center shift — translates where features appear
NOISE_SCALE       = 8   # Size/frequency change — stretches or compresses features
NOISE_FREQUENCY   = 8   # Additive spatial frequency field — overlays new spatial signal
NOISE_ORIENTATION = 8   # Rotation of the coordinate frame

# Number of noise variants per GT pattern.
# Total noise features = NOISE_PER_PATTERN * len(GT_PATTERNS)
NOISE_PER_PATTERN = 10

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X", "Gradient_Y", "Gradient_Diagonal_NE", "Gradient_Diagonal_NW",
    "Gradient_Radial_In", "Gradient_Radial_Out",
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical", "Stripes_Horizontal", "Stripes_Diagonal_45",
    "Stripes_Diagonal_135", "Waves_Concentric", "Waves_Spiral",
    "Waves_Interference", "Waves_Ripple",
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center", "Blob_TopRight", "Blob_TopLeft", "Blob_BottomRight",
    "Blob_BottomLeft", "Spots_Grid_Dense", "Spots_Grid_Sparse",
    "Spots_Random_Large", "Spots_Triangular", "Spots_Hexagonal",
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner", "Ring_Outer", "Ring_Double", "Ring_Eccentric",
    "Ring_Elliptical", "Ring_Partial",
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine", "Checkerboard_Coarse", "Quadrant_Alternating",
    "Sectors_4", "Sectors_8", "Triangle_Pattern", "Diamond_Pattern", "Honeycomb",
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers", "Hotspot_Cluster", "Edge_Enhancement", "Core_Shell",
    "Branching", "Laminar_Curved", "Mosaic_Irregular", "Gradient_Sigmoid",
    "Bimodal_Distribution", "Punctate_Dense", "Periventricular", "Asymmetric_Lobe",
]

TOTAL_NOISE_FEATURES = NOISE_PER_PATTERN * len(GT_PATTERNS)

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)

FIELD_WIDTH  = 6000
FIELD_HEIGHT = 6000


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    np.random.seed(sample_seed)
    brain_height = np.random.uniform(4900, 5400)
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    max_width_fraction = np.random.uniform(0.5, 0.9)
    rotation = np.random.uniform(-0.05, 0.05)
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    left_edge = (width - brain_max_width) / 2 + offset_x
    bottom_edge = (height - brain_height) / 2 + offset_y
    bottom_width = brain_max_width * 0.70
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height

    def is_in_halfbrain(x, y):
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        dx = x - cx; dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        if x_rot < left_edge: return False
        if y_rot < bottom_edge: return False
        if y_rot > top_edge: return False
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        if y_norm <= max_width_fraction:
            t = y_norm / max_width_fraction
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        right_edge = left_edge + right_width
        if x_rot > right_edge: return False
        if y_norm > max_width_fraction:
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            if x_rel < 0.05: max_y_norm = 1.0
            else: max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            if y_norm > max_y_norm: return False
        return True
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000,
                            sample_seed=None, add_noise=True,
                            size_scale_range=(0.70, 1.40)):
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    angle = np.arctan2(y - cy, x - cx)
    pattern_seed = hash(pattern_type) % 2**31

    if sample_seed is not None:
        rng_s = np.random.RandomState(sample_seed ^ pattern_seed)
        dx_offset = rng_s.uniform(-0.15, 0.15) * width
        dy_offset = rng_s.uniform(-0.15, 0.15) * height
        phase_offset = rng_s.uniform(-1.0, 1.0)
        freq_jitter = rng_s.uniform(0.75, 1.25)
        size_scale = rng_s.uniform(size_scale_range[0], size_scale_range[1])
        amp_scale = rng_s.uniform(0.75, 1.05)
        bio_noise_std = rng_s.uniform(0.0, 0.10)
        cx_s = cx + dx_offset; cy_s = cy + dy_offset
        dist_center_s = np.sqrt((x - cx_s)**2 + (y - cy_s)**2)
        angle_s = np.arctan2(y - cy_s, x - cx_s)
    else:
        dx_offset = dy_offset = 0.0
        phase_offset = 0.0; freq_jitter = 1.0; size_scale = 1.0
        amp_scale = 1.0; bio_noise_std = 0.0
        cx_s, cy_s = cx, cy
        dist_center_s = dist_center; angle_s = angle

    if pattern_type == "Gradient_X": val = x_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Y": val = y_norm + phase_offset * 0.2
    elif pattern_type == "Gradient_Diagonal_NE": val = (x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Diagonal_NW": val = (1 - x_norm + y_norm) / 2 + phase_offset * 0.15
    elif pattern_type == "Gradient_Radial_In": val = 1 - dist_norm + phase_offset * 0.1
    elif pattern_type == "Gradient_Radial_Out": val = dist_norm + phase_offset * 0.1
    elif pattern_type == "Stripes_Vertical":
        base_freq = 80 / freq_jitter
        val = (np.sin(x / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Horizontal":
        base_freq = 80 / freq_jitter
        val = (np.sin(y / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_45":
        base_freq = 100 / freq_jitter
        val = (np.sin((x + y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Stripes_Diagonal_135":
        base_freq = 100 / freq_jitter
        val = (np.sin((x - y) / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Concentric":
        base_freq = 60 / freq_jitter
        val = (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Waves_Spiral":
        spiral = angle_s + dist_center_s / (150 * freq_jitter) + phase_offset
        val = (np.sin(spiral * 3) + 1) / 2
    elif pattern_type == "Waves_Interference":
        d1 = np.sqrt((x - (width*0.3 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        d2 = np.sqrt((x - (width*0.7 + dx_offset))**2 + (y - (height*0.5 + dy_offset))**2)
        base_freq = 50 / freq_jitter
        val = (np.sin(d1/base_freq + phase_offset) + np.sin(d2/base_freq + phase_offset) + 2) / 4
    elif pattern_type == "Waves_Ripple":
        base_freq = 40 / freq_jitter
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center_s / base_freq + phase_offset * np.pi) + 1) / 2
    elif pattern_type == "Blob_Center":
        sigma = width * 0.2 * size_scale
        val = np.exp(-dist_center_s**2 / (2 * sigma**2))
    elif pattern_type == "Blob_TopRight":
        bx = x.max() * 0.75 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_TopLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset; by = y.max() * 0.75 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomRight":
        bx = x.max() * 0.75 + dx_offset; by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Blob_BottomLeft":
        bx = x.min() + (x.max()-x.min()) * 0.25 + dx_offset
        by = y.min() + (y.max()-y.min()) * 0.25 + dy_offset
        sigma = width * 0.15 * size_scale
        val = np.exp(-((x-bx)**2 + (y-by)**2) / (2 * sigma**2))
    elif pattern_type == "Spots_Grid_Dense":
        base_freq = 100 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Grid_Sparse":
        base_freq = 200 / freq_jitter
        raw = np.clip(np.sin(x/base_freq+phase_offset)*np.sin(y/base_freq+phase_offset), 0, 1)
        val = raw ** (1.0 / size_scale)
    elif pattern_type == "Spots_Random_Large":
        rng = np.random.RandomState(pattern_seed); sigma = width * 0.1 * size_scale; val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max()) + dx_offset; by = rng.uniform(y.min(), y.max()) + dy_offset
            val += np.exp(-((x-bx)**2+(y-by)**2) / (2*sigma**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Spots_Triangular":
        spacing = 400 * freq_jitter; spot_sigma = 80 * size_scale; val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350 * freq_jitter; spot_sigma = 60 * size_scale; val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i*spacing+(j%2)*(spacing/2)+dx_offset; by = j*spacing*0.866+dy_offset
                val += np.exp(-((x-bx)**2+(y-by)**2)/(2*spot_sigma**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * (0.2*size_scale+phase_offset*0.05); ring_width = width*0.04*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * (0.7*size_scale+phase_offset*0.05); ring_width = width*0.05*size_scale
        val = np.exp(-((dist_center_s-target_r)**2)/(2*ring_width**2))
    elif pattern_type == "Ring_Double":
        r1 = max_dist*(0.25*size_scale+phase_offset*0.03); r2 = max_dist*(0.55*size_scale+phase_offset*0.03)
        rw = width*0.03*size_scale
        val = np.exp(-((dist_center_s-r1)**2)/(2*rw**2))+np.exp(-((dist_center_s-r2)**2)/(2*rw**2))
        val = val / (val.max() + 1e-8)
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx+width*0.15+dx_offset; off_cy = cy+height*0.1+dy_offset
        dist_off = np.sqrt((x-off_cx)**2+(y-off_cy)**2)
        target_r = max_dist*(0.35*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        val = np.exp(-((dist_off-target_r)**2)/(2*rw**2))
    elif pattern_type == "Ring_Elliptical":
        a = max_dist*(0.5*size_scale+phase_offset*0.05); b = max_dist*(0.3*size_scale+phase_offset*0.03)
        ed = np.sqrt(((x-cx_s)/a)**2+((y-cy_s)/b)**2); rw = 0.1*size_scale
        val = np.exp(-((ed-1)**2)/(2*rw**2))
    elif pattern_type == "Ring_Partial":
        target_r = max_dist*(0.4*size_scale+phase_offset*0.04); rw = width*0.05*size_scale
        rv = np.exp(-((dist_center_s-target_r)**2)/(2*rw**2))
        ah = (np.pi/2)*size_scale; a_s = -ah+phase_offset*0.3; a_e = ah+phase_offset*0.3
        val = rv * ((angle_s > a_s) & (angle_s < a_e))
    elif pattern_type == "Checkerboard_Fine":
        ss = 200*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Checkerboard_Coarse":
        ss = 500*freq_jitter; xi = ((x+dx_offset)//ss).astype(int); yi = ((y+dy_offset)//ss).astype(int)
        val = ((xi+yi)%2).astype(float)
    elif pattern_type == "Quadrant_Alternating":
        val = ((x>cx_s)&(y>cy_s) | (x<cx_s)&(y<cy_s)).astype(float)
    elif pattern_type == "Sectors_4":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/2)).astype(int)%4; val = (sector%2).astype(float)
    elif pattern_type == "Sectors_8":
        sector = ((angle_s+np.pi+phase_offset)/(np.pi/4)).astype(int)%8; val = (sector%2).astype(float)
    elif pattern_type == "Triangle_Pattern":
        period = 300*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)/(period/2)+np.abs(((y+dy_offset)%period)-period/2)/(period/2))/2
    elif pattern_type == "Diamond_Pattern":
        period = 400*freq_jitter
        val = (np.abs(((x+dx_offset)%period)-period/2)+np.abs(((y+dy_offset)%period)-period/2))/period
    elif pattern_type == "Honeycomb":
        spacing = 200*freq_jitter; val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i*spacing+(j%2)*(spacing/2)+dx_offset; hy = j*spacing*0.866+dy_offset
                dist = np.sqrt((x-hx)**2+(y-hy)**2)
                val = np.maximum(val, 1-np.clip(dist/(spacing*0.4*size_scale), 0, 1))
        val = 1-val
    elif pattern_type == "Cortical_Layers":
        n_layers = 5; layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        sd = np.sqrt((x-cx_s)**2+(y-cy_s)**2); sn = sd/(sd.max()+1e-8)
        sc = np.clip(sn/size_scale, 0, 1); li = np.clip((sc*n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in li])
    elif pattern_type == "Hotspot_Cluster":
        rng = np.random.RandomState(pattern_seed); sigma = width*0.05*size_scale; val = np.zeros(n)
        ccx = cx+rng.uniform(-width*0.1, width*0.1)+dx_offset; ccy = cy+rng.uniform(-height*0.1, height*0.1)+dy_offset
        for _ in range(8):
            bx = ccx+rng.normal(0, width*0.08); by = ccy+rng.normal(0, height*0.08)
            val += np.exp(-((x-bx)**2+(y-by)**2)/(2*sigma**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Edge_Enhancement":
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        target = np.clip(0.8*size_scale+phase_offset*0.05, 0.3, 1.2); bw = 0.15*size_scale
        val = np.clip(1-np.exp(-((sn-target)**2)/(2*bw**2)), 0, 1)
    elif pattern_type == "Core_Shell":
        cs = width*0.1*size_scale; sr = max_dist*(0.5+phase_offset*0.05)*size_scale; ss = width*0.08*size_scale
        val = 0.7*np.exp(-dist_center_s**2/(2*cs**2))+0.5*np.exp(-((dist_center_s-sr)**2)/(2*ss**2))
        val = val/(val.max()+1e-8)
    elif pattern_type == "Branching":
        nb = 5; ba = 2*np.pi/nb; bw = 0.3*size_scale; ra = angle_s+phase_offset*0.5
        mdb = np.inf*np.ones(n)
        for i in range(nb):
            ta = i*ba; ad2 = np.abs(np.mod(ra-ta+np.pi, 2*np.pi)-np.pi)
            mdb = np.minimum(mdb, ad2)
        val = np.exp(-mdb**2/(2*bw**2))*(1-np.exp(-dist_norm*3))
    elif pattern_type == "Laminar_Curved":
        ca = 200*(1+phase_offset*0.3); cf = 400*freq_jitter; bt = 300*size_scale
        cy2 = (y+dy_offset)-ca*np.sin((x+dx_offset)/cf)
        val = ((cy2//bt).astype(int)%2).astype(float)
    elif pattern_type == "Mosaic_Irregular":
        rng = np.random.RandomState(pattern_seed); ns = max(5, int(20/size_scale))
        sx = rng.uniform(x.min(), x.max(), ns)+dx_offset; sy = rng.uniform(y.min(), y.max(), ns)+dy_offset
        sv = rng.rand(ns); ni = np.zeros(n, dtype=int); md = np.inf*np.ones(n)
        for i in range(ns):
            d = np.sqrt((x-sx[i])**2+(y-sy[i])**2); c = d<md; ni[c] = i; md[c] = d[c]
        val = sv[ni]
    elif pattern_type == "Gradient_Sigmoid":
        shift = 0.5+phase_offset*0.1; steep = 10.0/size_scale
        val = 1/(1+np.exp(-steep*(x_norm-shift)))
    elif pattern_type == "Bimodal_Distribution":
        val = (x<cx_s).astype(float)*0.8+(x>=cx_s).astype(float)*0.3
    elif pattern_type == "Punctate_Dense":
        rng = np.random.RandomState(pattern_seed); ps = 40*size_scale; val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())+dx_offset; py = rng.uniform(y.min(), y.max())+dy_offset
            val += np.exp(-((x-px)**2+(y-py)**2)/(2*ps**2))
        val = np.clip(val, 0, 1)
    elif pattern_type == "Periventricular":
        vr = 0.15*size_scale+phase_offset*0.02; rr = 0.25*size_scale+phase_offset*0.02; rw = 0.08*size_scale
        sn = dist_center_s/(dist_center_s.max()+1e-8)
        val = np.exp(-((sn-rr)**2)/(2*rw**2))*(sn>=vr)
    elif pattern_type == "Asymmetric_Lobe":
        l1x = cx-width*0.2+dx_offset; l1y = cy+dy_offset; l2x = cx+width*0.15+dx_offset; l2y = cy+height*0.1+dy_offset
        s1 = width*0.15*size_scale; s2 = width*0.10*size_scale
        val = 0.7*np.exp(-((x-l1x)**2+(y-l1y)**2)/(2*s1**2))+0.5*np.exp(-((x-l2x)**2+(y-l2y)**2)/(2*s2**2))
        val = val/(val.max()+1e-8)
    else:
        rng = np.random.RandomState(pattern_seed); val = rng.rand(n)

    val = val * amp_scale
    if add_noise and bio_noise_std > 0:
        rng_noise = np.random.RandomState(sample_seed ^ (pattern_seed + 1))
        val = val + rng_noise.normal(0, bio_noise_std, size=n)
    return np.clip(val, 0, 1)


def generate_noise_from_parent(coords, parent_pattern, variant_idx,
                               width=3000, height=3000,
                               sample_seed=None, size_scale_range=(0.70, 1.40),
                               shape=8, location=8, scale=8,
                               frequency=8, orientation=8):
    """
    Generate a noise feature as a spatially-distorted variant of a parent GT pattern.

    Five independent parameters (each 1-20) control distinct spatial properties:

      shape       — Sinusoidal coordinate warp that deforms the spatial structure.
                    Level 1 ≈ 100 µm warp;  Level 20 ≈ 2000 µm warp.
      location    — Translates the effective pattern center.
                    Level 1 ≈ ±5 % shift;  Level 20 ≈ ±40 % shift.
      scale       — Stretches or compresses (anisotropically) the pattern.
                    Level 1 ≈ ±5 % change;  Level 20 ≈ ±60 % change.
      frequency   — Overlays unique additive + multiplicative spatial fields.
                    Level 1 ≈ 10 % blend;  Level 20 ≈ 60 % blend.
      orientation — Rotates the coordinate frame.
                    Level 1 ≈ ±5°;  Level 20 ≈ ±90°.

    variant_idx (0..NOISE_PER_PATTERN-1) determines which unique set of
    perturbations is applied.  Different variant_idx values produce different
    random draws for each distortion axis.

    CRITICAL INVARIANT: All distortions use smooth continuous spatial functions
    f(x, y). The RNG draws only scalar parameters (never n-dependent arrays).
    This guarantees identical results on RNA and MSI grids.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]

    # Convert 1-20 integer controls to 0-1 continuous magnitudes.
    # Floor of 0.03 ensures even level=1 produces a small change.
    d_shape  = max(0.03, (shape - 1) / 19.0)
    d_loc    = max(0.03, (location - 1) / 19.0)
    d_scale  = max(0.03, (scale - 1) / 19.0)
    d_freq   = max(0.03, (frequency - 1) / 19.0)
    d_orient = max(0.03, (orientation - 1) / 19.0)

    # Unique RNG for this parent+variant combination — scalar draws only.
    parent_hash = hash(parent_pattern) % 2**31
    variant_seed = (parent_hash * 1000 + variant_idx * 7 + 42) % 2**31
    rng = np.random.RandomState(variant_seed)

    cx_field = width / 2.0
    cy_field = height / 2.0

    # =====================================================================
    # 1. SHAPE — Multi-frequency sinusoidal coordinate warp
    #    Two warp layers at different spatial frequencies produce complex
    #    non-rigid deformations that vary across the field.
    #    Level 1 ≈ 100 µm;  Level 20 ≈ 2000 µm total displacement.
    # =====================================================================
    warp_base = 100 + d_shape * 1900  # µm of maximum displacement

    # Layer 1: low-frequency broad warp
    w1_amp_x  = rng.uniform(0.4, 0.7) * warp_base
    w1_amp_y  = rng.uniform(0.4, 0.7) * warp_base
    w1_freq_x = rng.uniform(2000, 5000)
    w1_freq_y = rng.uniform(2000, 5000)
    w1_ph_x   = rng.uniform(0, 2 * np.pi)
    w1_ph_y   = rng.uniform(0, 2 * np.pi)

    # Layer 2: higher-frequency local warp
    w2_amp_x  = rng.uniform(0.2, 0.5) * warp_base
    w2_amp_y  = rng.uniform(0.2, 0.5) * warp_base
    w2_freq_x = rng.uniform(800, 2000)
    w2_freq_y = rng.uniform(800, 2000)
    w2_ph_x   = rng.uniform(0, 2 * np.pi)
    w2_ph_y   = rng.uniform(0, 2 * np.pi)

    x_w = (x
           + w1_amp_x * np.sin(y / w1_freq_y + w1_ph_y)
           + w2_amp_x * np.sin(y / w2_freq_y + w2_ph_y))
    y_w = (y
           + w1_amp_y * np.sin(x / w1_freq_x + w1_ph_x)
           + w2_amp_y * np.sin(x / w2_freq_x + w2_ph_x))

    # =====================================================================
    # 2. LOCATION — Center shift / translation
    #    Moves the effective origin of the pattern.
    #    Level 1 ≈ ±5 % of field;  Level 20 ≈ ±40 % of field.
    # =====================================================================
    shift_mag = 0.05 + d_loc * 0.35
    shift_x = rng.uniform(-shift_mag, shift_mag) * width
    shift_y = rng.uniform(-shift_mag, shift_mag) * height

    x_w = x_w + shift_x
    y_w = y_w + shift_y

    # =====================================================================
    # 3. SCALE — Anisotropic coordinate rescaling around field center
    #    X and Y are scaled independently for richer variation.
    #    Level 1 ≈ ±5 % change;  Level 20 ≈ ±60 % change.
    # =====================================================================
    scale_range = 0.05 + d_scale * 0.55
    scale_x = 1.0 + rng.uniform(-scale_range, scale_range)
    scale_y = 1.0 + rng.uniform(-scale_range, scale_range)

    x_w = cx_field + (x_w - cx_field) * scale_x
    y_w = cy_field + (y_w - cy_field) * scale_y

    # =====================================================================
    # 4. ORIENTATION — Rotation around field center
    #    Tilts directional patterns and shifts angular features.
    #    Level 1 ≈ ±5°;  Level 20 ≈ ±90°.
    # =====================================================================
    max_rot = 0.09 + d_orient * 1.48  # radians (≈5° to ≈90°)
    rot_angle = rng.uniform(-max_rot, max_rot)
    cos_r, sin_r = np.cos(rot_angle), np.sin(rot_angle)
    dx_c = x_w - cx_field
    dy_c = y_w - cy_field
    x_r = cx_field + dx_c * cos_r - dy_c * sin_r
    y_r = cy_field + dx_c * sin_r + dy_c * cos_r

    # =====================================================================
    # Generate parent pattern at the distorted coordinates.
    # The pattern is evaluated at warped positions, producing actual
    # spatial structure changes (not just intensity/color shifts).
    # =====================================================================
    warped_coords = np.column_stack([x_r, y_r])
    val = generate_pattern_values(warped_coords, parent_pattern, width, height,
                                  sample_seed=sample_seed, add_noise=False,
                                  size_scale_range=size_scale_range)

    # =====================================================================
    # 5. FREQUENCY — Additive + multiplicative spatial fields
    #    Two independent sinusoidal fields that alter the spatial frequency
    #    content: one additive (blends in a new pattern) and one
    #    multiplicative (spatially modulates intensity).
    #    Level 1 ≈ 10 % blend;  Level 20 ≈ 60 % blend.
    # =====================================================================
    blend_strength = 0.10 + d_freq * 0.50

    # Additive field
    add_freq  = rng.uniform(300, 1200)
    add_theta = rng.uniform(0, 2 * np.pi)
    add_phase = rng.uniform(0, 2 * np.pi)
    proj = x * np.cos(add_theta) + y * np.sin(add_theta)
    additive = (np.sin(proj / add_freq + add_phase) + 1) / 2

    # Multiplicative field (different angle/frequency)
    mul_freq_x = rng.uniform(500, 2000)
    mul_freq_y = rng.uniform(500, 2000)
    mul_phase  = rng.uniform(0, 2 * np.pi)
    multiplicative = (np.sin(x / mul_freq_x + mul_phase) *
                      np.cos(y / mul_freq_y + mul_phase * 0.7) + 1) / 2

    # Blend: (1-s)*pattern + s/2*additive + s/2*multiplicative*pattern
    s = blend_strength
    val = val * (1 - s) + additive * (s * 0.5) + val * multiplicative * (s * 0.5)

    val = np.clip(val, 0, 1)

    # =====================================================================
    # Per-sample variation (same mechanism as GT patterns)
    # =====================================================================
    if sample_seed is not None:
        perturb_seed = (variant_seed ^ (sample_seed * 1000003)) % 2**31
        rng_p = np.random.RandomState(perturb_seed)
        amp_scale     = rng_p.uniform(0.85, 1.05)
        bio_noise_std = rng_p.uniform(0.0, 0.06)

        val = val * amp_scale
        if bio_noise_std > 0:
            rng_bio = np.random.RandomState((variant_seed ^ sample_seed) + 2)
            val = val + rng_bio.normal(0, bio_noise_std, size=n)

    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    dy = spacing * np.sqrt(3) / 2
    coords = []; row = 0; y = 0
    while y < height:
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        while x < width:
            coords.append([x, y]); x += spacing
        y += dy; row += 1
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    x = np.arange(0, width, spacing); y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================
if __name__ == "__main__":

    n_gt = len(GT_PATTERNS)
    total_noise = NOISE_PER_PATTERN * n_gt
    n_total_features = n_gt + total_noise

    print("=" * 60)
    print("Generating Ground Truth Dummy Data with 50 PATTERNS")
    print("Each GT pattern produces noise variants with 5 distortion controls")
    print("=" * 60)
    print(f"Groups: {GROUPS}")
    print(f"GT Patterns: {n_gt}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Total noise features: {total_noise}")
    print(f"Total features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise distortion controls (1-20):")
    print(f"  Shape:       {NOISE_SHAPE:2d}  (coordinate warp, 100–2000 µm)")
    print(f"  Location:    {NOISE_LOCATION:2d}  (center shift, ±5–40%)")
    print(f"  Scale:       {NOISE_SCALE:2d}  (anisotropic rescale, ±5–60%)")
    print(f"  Frequency:   {NOISE_FREQUENCY:2d}  (additive+multiplicative field, 10–60%)")
    print(f"  Orientation: {NOISE_ORIENTATION:2d}  (rotation, ±5–90°)")
    print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} um")
    print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}um spacing")
    print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}um pixels")
    print(f"Pattern size scale range: {PATTERN_SIZE_SCALE_RANGE[0]:.2f}-{PATTERN_SIZE_SCALE_RANGE[1]:.2f}")
    print(f"Output: {OUTPUT_DIR}")
    print("=" * 60)

    ground_truth_pairs = []

    for group_idx, group in enumerate(GROUPS):
        for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
            sample_id = f"{group}_{sample_idx}"
            seed = hash(sample_id) % 2**32
            print(f"\n  Processing {sample_id} (seed={seed})...")

            tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)

            # === MSI Data ===
            raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
            mask_msi = [tissue_fn(xi, yi) for xi, yi in raw_msi_coords]
            msi_coords = raw_msi_coords[mask_msi]

            msi_data = np.zeros((len(msi_coords), n_total_features))
            var_names_msi = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    MSI GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                msi_data[:, col] = generate_pattern_values(
                    msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_msi.append(f"MZ_{pat}")
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    msi_data[:, col] = generate_noise_from_parent(
                        msi_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        shape=NOISE_SHAPE, location=NOISE_LOCATION,
                        scale=NOISE_SCALE, frequency=NOISE_FREQUENCY,
                        orientation=NOISE_ORIENTATION)
                    var_names_msi.append(f"MZ_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
            obs_msi['x_um'] = msi_coords[:, 0]
            obs_msi['y_um'] = msi_coords[:, 1]

            adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
            adata_msi.var_names = var_names_msi
            adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))

            # === RNA Data ===
            raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
            mask_rna = [tissue_fn(xi, yi) for xi, yi in raw_rna_coords]
            rna_coords = raw_rna_coords[mask_rna]

            rna_data = np.zeros((len(rna_coords), n_total_features))
            var_names_rna = []
            col = 0

            for i, pat in enumerate(GT_PATTERNS):
                print(f"    RNA GT {i+1:2d}/{n_gt}: {pat}", end="")
                # GT feature
                rna_data[:, col] = generate_pattern_values(
                    rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT,
                    sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE)
                var_names_rna.append(f"Gene_{pat}")
                if group_idx == 0 and sample_idx == 1:
                    ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
                col += 1

                # Noise variants for this pattern
                for v in range(NOISE_PER_PATTERN):
                    rna_data[:, col] = generate_noise_from_parent(
                        rna_coords, pat, v,
                        width=FIELD_WIDTH, height=FIELD_HEIGHT,
                        sample_seed=seed, size_scale_range=PATTERN_SIZE_SCALE_RANGE,
                        shape=NOISE_SHAPE, location=NOISE_LOCATION,
                        scale=NOISE_SCALE, frequency=NOISE_FREQUENCY,
                        orientation=NOISE_ORIENTATION)
                    var_names_rna.append(f"Gene_Noise_{pat}_{v}")
                    col += 1
                print(f" + {NOISE_PER_PATTERN} noise variants")

            obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
            obs_rna['x_um'] = rna_coords[:, 0]
            obs_rna['y_um'] = rna_coords[:, 1]

            adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
            adata_rna.var_names = var_names_rna
            adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

    print("\n" + "=" * 60)
    print("DONE! Data generated.")
    print("=" * 60)
    print(f"\n{n_gt} GROUND TRUTH MATCHES:")
    print("-" * 60)
    for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
        print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
    print("-" * 60)
    print(f"\nTotal features per sample: {n_gt} GT + {total_noise} Noise = {n_total_features}")
    print(f"Noise per pattern: {NOISE_PER_PATTERN}")
    print(f"Noise distortion controls: shape={NOISE_SHAPE}, location={NOISE_LOCATION}, "
          f"scale={NOISE_SCALE}, frequency={NOISE_FREQUENCY}, orientation={NOISE_ORIENTATION}")
    print(f"\nNoise naming: MZ_Noise_<PatternName>_<VariantIdx> / Gene_Noise_<PatternName>_<VariantIdx>")
    print(f"Example: MZ_Noise_Gradient_X_0 is variant 0 of Gradient_X noise")

Generating Ground Truth Dummy Data with 50 PATTERNS
Each GT pattern produces noise variants with 5 distortion controls
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise per pattern: 10
Total noise features: 500
Total features per sample: 50 GT + 500 Noise = 550
Noise distortion controls (1-20):
  Shape:        8  (coordinate warp, 100–2000 µm)
  Location:     8  (center shift, ±5–40%)
  Scale:        8  (anisotropic rescale, ±5–60%)
  Frequency:    8  (additive+multiplicative field, 10–60%)
  Orientation:  8  (rotation, ±5–90°)
Field size: 6000x6000 um
RNA grid: Visium-like hexagonal, 100um spacing
MSI grid: Cartesian, 60um pixels
Pattern size scale range: 0.90-1.10
Output: dummy_data_50

  Processing YC_1 (seed=4166224011)...
    MSI GT  1/50: Gradient_X + 10 noise variants
    MSI GT  2/50: Gradient_Y + 10 noise variants
    MSI GT  3/50: Gradient_Diagonal_NE + 10 noise variants
    MSI GT  4/50: Gradient_Diagonal_NW + 10 noise variants
    MSI GT  5/50: Gradient_Radial_In + 1